# Explicabilidad y simulador de riesgo por vano

Hermano de `04_uiti_vano_trayectorias_vano.ipynb`: reutiliza su clasificacion KMeans por
vano x ventana -- que nunca se reajusta aqui -- y agrega encima un simulador de "que
pasaria si" a nivel de vano.

**Un solo modelo y una sola unidad.** Todo lo que produce el boton "Simular" -- el mapa
**Criticidad Simulada**, el **top 10 de variables por vano**, el **grafo de relevancia** y
la comparacion **base contra simulado** -- viene del MIL entrenado en
`05_mil_vano_ventana`, que puntua **bolsas**: una bolsa es una celda (vano, ventana) y sus
instancias son los eventos de ese vano en esa ventana. La clase sale de
`asignar_clase(n_obs observado, u-hat predicho)` sobre la geometria KMeans de 04, la misma
del mapa base, de modo que los dos mapas comparten paleta por construccion y no por
convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.

**Que mide cada mapa.** **Criticidad Original** (izquierda) es el grupo historico que 04
ya calculo sobre eventos observados. **Criticidad Simulada** (derecha) es lo que el modelo
predice al aplicar las variables del simulador. Son dos mediciones distintas y por eso
nunca comparten titulo, aunque si comparten la escala de color, que es la misma por
construccion.

**Se estudian hasta cinco vanos a la vez.** Cada uno recibe su propia columna de controles,
su propia serie de tiempo y su propio top de variables. Ese tope no es decorativo: es lo
que hace que la pregunta del tablero sea "que le pasa a ESTE vano" y no "que le pasa al
promedio de un circuito".

**Y cuanto cuesta.** La fila 5 costea la intervencion con las tarifas del propio contrato
de CHEC (`data/COSTOS ITEMS CONTRATOS.xlsx`): se marcan las actividades, se dice cuantas
veces van sobre cada vano, y el tablero produce el costo por vano y el total. Es la otra
mitad de la decision de mantenimiento -- todo lo demas dice cuanto BAJA el riesgo, y esto
dice cuanto cuesta bajarlo.

**Las dos mitades no estan atadas, y es a proposito.** Marcar "PODA EN REDES RURALES TIPO
A" no mueve `NR_T`, y bajar `NR_T` no programa una poda. El modelo no tiene ningun mapa de
actividades del contrato a variables, e inventarlo produciria un numero con toda la pinta
de ser el beneficio estimado de esa actividad sin serlo. El tablero pone lado a lado el
efecto simulado y el costo cotizado del plan que el usuario dice que ejecutaria; el puente
entre los dos se queda en su cabeza, que es el sitio honesto hasta que exista esa
correspondencia.

**Requiere dos artefactos de `05_mil_vano_ventana.ipynb`**: `data/models/mil_vano_ventana_v1.pt`
y `data/derived/bolsas_mil_full.joblib`. Los dos viven bajo `data/`, que git ignora; si
faltan, la celda del modelo falla de inmediato nombrando el cuaderno que los produce.

**La interfaz es este cuaderno, y solo este cuaderno.** Todo se controla y se ve en la
figura de siete paneles con los controles de `ipywidgets` encima. Al ejecutar no se escribe
ningun archivo ni se abre ningun navegador. El panel HTML autocontenido que existia antes
se elimino: era una transcripcion completa del modelo a JavaScript -- casi la mitad del
codigo del cuaderno -- que habia que mantener en paralelo con la version de Python y que
se quedaba atras en cada cambio.

In [ ]:
import asyncio
import gc
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este cuaderno requiere ipywidgets para la interfaz interactiva.') from exc
from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

# Un kernel que ya importo estos paquetes se queda con la version VIEJA en `sys.modules`:
# "Run All" sin reiniciar NO vuelve a leer el disco. Un rename en src/ -- por ejemplo
# `SelectorVanos._caja` -> `.caja` -- estalla entonces como AttributeError diez celdas mas
# abajo, con el codigo del disco ya correcto. Se purgan ANTES de importarlos, asi el
# cuaderno corre SIEMPRE contra la fuente actual, con o sin reinicio de kernel. Va aqui y no
# como `importlib.reload`: reload no rehace los objetos ya construidos con la clase vieja,
# y este cuaderno los reconstruye todos de esta celda para abajo.
for _modulo in [m for m in list(sys.modules)
                if m.split('.')[0] in ('chec_impacto', 'chec_local_interpreter', 'scripts')]:
    del sys.modules[_modulo]

from chec_impacto.data import procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    cargar_geometria_014,
    verificar_sha1_geometrias,
)
from chec_impacto.data.bags import cargar_bolsas
from chec_impacto.models.mil_persistencia import cargar_modelo_mil
from chec_impacto.training import resolve_training_device
from chec_local_interpreter.costos_items import (
    MAX_REPETICIONES,
    costos_de_intervencion,
    leer_catalogo_costos,
)
from chec_local_interpreter.mil_simulador_015 import (
    gates_de_bolsas,
    grafo_de_gates,
    grafo_diferencia,
    plan_hacia_clase_minima,
    relevancia_hacia_uiti_minimo,
    seleccionar_bolsas,
    simular_bolsas,
    trazas_grafo,
    valores_actuales_por_vano,
)
from chec_local_interpreter.simulador_variables import (
    barras_uiti_por_vano,
    GRUPO_POR_KNOB,
    columnas_panel,
    rotacion_radial,
    rotulo_en_barra,
    knobs_bloqueados,
    knobs_simulables,
    tabla_variables_simulables,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.vano_widgets import (
    MAX_VANOS_ANALISIS,
    VANOS_POR_PAGINA,
    construir_selector_casillas,
    construir_selector_vanos,
    figura_de_mapas,
)
from chec_local_interpreter.ventanas_015 import (
    CAMBIOS,
    CAMBIO_EMPEORA,
    CAMBIO_IGUAL,
    CAMBIO_MEJORA,
    bounds_de_fids,
    cajas_por_cambio_de_grupo,
    cajas_seleccion,
    capas_mapa_historico,
    cargar_clases_desde_014,
    centro_y_zoom,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
    fid_de_punto,
    clases_de_series,
    series_temporal_vanos,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)

# Sonda del contrato que rompio el cuaderno dos veces. Si el kernel siguiera sirviendo una
# version vieja de vano_widgets, falla AQUI -- primera celda, mensaje que dice que hacer --
# en vez de a los 10 minutos de procesamiento, en la celda del panel.
_sonda = construir_selector_vanos(['0'])
assert hasattr(_sonda, 'caja'), (
    'vano_widgets viejo en memoria: el selector de casillas sin `.caja`. '
    'Reinicia el kernel. '
    f'(modulo cargado desde {sys.modules["chec_local_interpreter.vano_widgets"].__file__})'
)
del _sonda

In [ ]:
# Ventana climatica igual que 03_mgcecdl_training / 09_simulador: cambiarla generaria un
# set de features distinto al que el modelo cargado en la celda SEAM espera.
VENTANA_CLIMATICA_HORAS = 12
# El espacio de agrupamiento no se elige: viene fijo de `criticality_assignment.py`.
# La clave paso de '2' a '0' el 2026-08-09, cuando 04 dejo de enumerar ocho espacios;
# la geometria es la MISMA, solo cambio el indice bajo el que esta archivada, asi que
# esto se lee de la constante y no se escribe a mano.
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO
DEVICE = resolve_training_device('auto')

# Misma paleta que 01.4: los grupos historicos de este cuaderno SON los de 01.4, nunca se
# reajustan, asi que el color tiene que significar lo mismo en los dos cuadernos.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
# Semaforo: verde Bajo, amarillo Medio, naranja Medio-Alto, rojo Alto. Antes era una
# rampa de rojos, que ordenaba por SATURACION: `Bajo` y `Medio` eran dos rosas que solo
# se distinguian mirandolos uno al lado del otro, y en el mapa el mas claro quedaba a un
# paso del fondo. El semaforo ordena por TONO, que es lo que se lee de un vistazo y sin
# tener la leyenda al lado.
# Contraste medido contra el fondo de carto-positron (#f2f0eb): 3,36 / 1,48 / 2,71 / 4,94.
# El amarillo es el mas flojo de los cuatro, pero el peor caso NO empeora -- el `Bajo`
# salmon de la rampa vieja medía 1,44 --, y oscurecerlo mas lo acerca al naranja hasta
# volver indistinguibles los dos niveles del medio.
# En formato `rgb(...)` y no hexadecimal a proposito: el relleno de los violines y de los
# contornos sale de `.replace('rgb', 'rgba')`, que sobre un hex no encuentra nada y deja
# el relleno sin aplicar, en silencio.
COLORES_GRUPOS = ['rgb(26,150,65)', 'rgb(242,194,0)', 'rgb(239,108,0)', 'rgb(198,40,40)']
# UN solo codigo de ausencia en los DOS mapas: negro, el `COLOR_SIN_EVENTO` de 01.4.
# El vano sin eventos en la ventana no tiene clase, y la ausencia no es la clase mas baja;
# que el base lo pintara gris y el simulado negro obligaba a recordar dos codigos para la
# misma cosa. Los cuatro colores KMeans significan lo mismo en los dos mapas.
COLOR_SIN_EVENTO = 'rgb(0,0,0)'
COLOR_MARCADO = '#0072b2'
# Equipos: mismos colores que 01.4, por el mismo motivo que la paleta de grupos --
# un naranja tiene que seguir siendo un transformador al pasar de un cuaderno a otro.
COLOR_TRAFO = '#f59e0b'
COLOR_SWITCH = '#7c3aed'
ANCHO_MAPA = 3.0
ANCHO_MAPA_MARCADO = round(ANCHO_MAPA * 1.4, 2)

# Fila 1, paridad 01.4: un vano MARCADO se dibuja con el color de SU clase, sobre un halo
# blanco que lo despega del fondo (01.4: `width=ANCHO_MAPA_RESALTE * 2.6, color='white'`).
# Un color plano de "seleccionado" encima de la clase congela lo que se ve: la ventana
# cambia la clase por debajo y el vano marcado sigue igual en pantalla. COLOR_MARCADO
# queda solo para la fila 2, donde la clase la pone el modelo y no el KMeans.
COLOR_HALO = 'white'
ANCHO_HALO = round(ANCHO_MAPA_MARCADO * 2.6, 2)
# El vano SELECCIONADO se encierra ademas en una caja amarilla translucida, GIRADA a la
# inclinacion del propio vano. El halo blanco y el ancho extra lo separan de su vecino
# inmediato, pero no lo hacen ENCONTRABLE en un circuito de cientos de tramos; una caja
# si, y sigue siendo una caja a cualquier zoom, donde una linea deja de distinguirse de
# las de al lado.
# El giro no es cosmetica: con el rectangulo min/max, el grosor del resalte dependia del
# rumbo y del largo del vano. Medido sobre los 59.776 tramos, la caja de ejes es 1,3 veces
# mas gorda que la banda a traves del trazo en la mediana, 4 veces en el p90 y 169 veces
# en el peor caso, y el 52,8% de los tramos corre en diagonal. La misma marca se veia como
# una funda ajustada sobre un vano norte-sur y como un parche suelto sobre uno diagonal.
# Va por `layout.map.layers` con `below='traces'` y NO como traza. Las dos cosas importan:
# una traza rellena por encima se comeria el clic -- que es justo lo que alterna la
# seleccion -- y ademas tiniria de amarillo la linea del vano, borrando el color de su
# clase. Debajo de las trazas, la caja rodea al vano y la clase se sigue leyendo.
COLOR_CAJA_SELECCION = '#ffd400'
OPACIDAD_CAJA_SELECCION = 0.5
# El recuadro del mapa SIMULADO sale de la MISMA geometria, asi que hereda la inclinacion
# sin codigo aparte: los dos mapas tienen que encerrar el mismo vano con el mismo
# rectangulo, y dos formas distintas se leerian como dos vanos. No contesta "cual estoy
# estudiando" -- eso ya lo dice el del mapa base, con el mismo vano encerrado a la
# izquierda -- sino QUE LE PASO al vano:
# verde claro si bajo de grupo de criticidad, amarillo si se quedo en el mismo, rojo si
# subio. Son TRES capas y no una porque una capa de `layout.map.layers` pinta con UN color.
# El amarillo es exactamente el del mapa base a proposito: "no cambio" es justo el estado
# en que los dos mapas dicen lo mismo, y un cuarto color inventaria una diferencia que no
# hay.
COLOR_CAJA_MEJORA = '#4ade80'
COLOR_CAJA_IGUAL = COLOR_CAJA_SELECCION
COLOR_CAJA_EMPEORA = '#dc2626'
# Lado minimo de la caja, en grados (~50 m a esta latitud). A TRAVES del trazo la caja
# nace de ancho CERO -- una linea no tiene grosor --, y cero pixeles de ancho no se ve.
# Con la caja girada este es el ancho de la banda en TODOS los vanos, no solo en los que
# corrian sobre un eje.
LADO_MINIMO_CAJA = 0.00045
# Margen a cada lado (~10 m): sin el, el borde de la caja cae encima del trazo del vano y
# no se distingue cual es cual.
MARGEN_CAJA = 0.00009
OPACIDAD_NUBE = 0.45               # 01.4, para que la nube de fondo no tape el resaltado
OPACIDAD_FRONTERA = 0.28           # 01.4: el contorno es fondo, no dato
# Paleta de 01.4 para las series por vano: apta para daltonismo y distinta de la escala de
# grupos, porque aqui el color identifica AL VANO, no a su clase.
# DIEZ colores porque el tope de vanos subio a diez con el diagnostico del circuito. Los
# seis primeros son los de 01.4 y no se tocan: un vano que era azul en un cuaderno tiene
# que seguir siendo azul en el otro. Los cuatro nuevos se eligen lejos de esos seis y
# entre si, porque el color aqui identifica AL VANO y dos parecidos serian dos series
# indistinguibles.
COLORES_VANOS = ['#0072b2', '#009e73', '#cc79a7', '#56b4e9', '#e69f00', '#8c564b',
                 '#d55e00', '#6a3d9a', '#17becf', '#666666']
# Cuantas barras lleva cada grupo del top por vano. DIEZ y no cinco: el barrido puntua
# trece variables numericas y cortar en cinco dejaba fuera de la vista mas de la mitad del
# ranking que ya se calculo -- las pasadas del modelo son las mismas, solo cambia cuantas
# se muestran. Lo que el cinco protegia era el rotulo escrito dentro de la barra, y de eso
# se encarga ahora la cascada resumen -> inicial -> nada de
# `simulador_variables.rotulo_en_barra`, que decide barra por barra segun lo que mida.
TOP_VARIABLES_POR_VANO = 10
# Cuantos valores se prueban de cada control al buscar el que MINIMIZA el UITI del vano.
# Nueve y no dos: medido sobre el modelo real, 10 de los 15 controles numericos tienen su
# mejor valor en el INTERIOR del rango para alguna bolsa -- `DDT` para todas --, asi que
# mirar solo los extremos muestrea los dos puntos equivocados. Nueve puntos son 136
# pasadas de bolsas para toda la seleccion, medidas en 0,2 s.
PUNTOS_REJILLA_RELEVANCIA = 9
# Cuantos cambios encadena como mucho el plan combinado. Cuatro: medido sobre 59 bolsas
# de 40 circuitos, en Medio-Alto y Alto NINGUNA variable sola alcanza el grupo Bajo -- 0
# de 18 y 0 de 8 --, y la mejor cubre el 60% y el 49% del camino. La combinacion no es un
# extra del ranking, es la respuesta para esos dos grupos.
MAX_PASOS_PLAN = 4
# El boton de diagnostico del circuito: cuantos vanos mira y cuantas variables reporta de
# cada mitad de la decision. Diez vanos porque es una lista que cabe en una orden de
# trabajo; cinco de intervencion y tres de escenario porque la pregunta no es simetrica --
# lo que se HACE es lo que se cotiza, y lo que se ANTICIPA sirve para saber bajo que
# condiciones esa obra rinde.
TOP_VANOS_CIRCUITO = 10
# El diagnostico mira PRIMERO el grupo Alto y completa con Medio-Alto. Los grupos de abajo
# no entran: la pregunta es por donde empezar, y un vano en Medio no es por donde se
# empieza mientras queden vanos en Alto sin atender.
GRUPOS_DIAGNOSTICO = [3, 2]
TOP_INTERVENCION_CIRCUITO = 5
TOP_ESCENARIO_CIRCUITO = 3
# Fuente del rotulo dentro de la barra. Los anchos de caracter con los que
# `rotulo_en_barra` decide si el nombre cabe estan MEDIDOS a este tamanio
# (`simulador_variables.TAM_FUENTE_MEDIDO`): cambiarlo aqui sin volver a medir en el
# modulo deja al panel decidiendo con numeros de otra fuente.
TAM_FUENTE_BARRA = 8
# El punto de la VENTANA VIGENTE en las dos series se dibuja al triple, como el dia
# vigente en la serie del cuaderno 01. `marker.size` es un ARRAY por eso: mover el
# deslizador solo reescribe ese arreglo y el punto grande viaja con el.
SERIE_TAM_UITI = 9
SERIE_TAM_EVENTOS = 8
FACTOR_PUNTO_ACTIVO = 3
# Las dos barras de la fila 4. NO son la misma cantidad medida dos veces: la gris es lo
# que dice la base de datos y la azul es lo que predice el modelo. Ver `_pintar_barras_uiti`
# -- el color separa medicion de prediccion, y por eso no comparten paleta con los grupos.
# Colores fuera de la escala de criticidad a proposito -- aqui el color distingue dos
# CORRIDAS, no dos niveles, y reusar la paleta de grupos invitaria a leerlos como clases.
# El punto de la serie sin celda en la ventana: no tiene grupo, y eso NO es el grupo mas
# bajo -- es la ausencia del dato. Gris, fuera de la escala de criticidad.
COLOR_SIN_GRUPO = '#94a3b8'
COLOR_BARRA_MEDIDA = '#94a3b8'
COLOR_BARRA_SIMULADA = '#0072b2'
# A que distancia del centro se planta el rotulo de cada nodo del grafo. Los nodos viven
# sobre el circulo de radio 1; 1,05 despega el texto de su propio marcador sin alejarlo
# tanto como para que deje de leerse como suyo.
RADIO_ROTULO_GRAFO = 1.05
# La fila del costo de intervencion. El azul identifica a un VANO, igual que en las
# series; el gris oscuro es el TOTAL, que no es un vano mas sino la suma de todos y por
# eso no comparte su color. Ninguno de los dos toca la paleta de criticidad: aqui se
# miden pesos, no riesgo, y un rojo aqui se leeria como un grupo.
COLOR_BARRA_COSTO = '#0072b2'
COLOR_BARRA_TOTAL = '#5b4a48'
ETIQUETA_TOTAL = 'TOTAL'
# Ancho de la casilla de cada actividad del contrato. Los nombres del contrato son
# largos -- mediana de 63 caracteres, maximo 143 -- asi que el rotulo arranca por el
# PRECIO: es el dato con el que se elige, y puesto al final lo recortaria el ancho fijo.
ANCHO_CASILLA_ITEM = '580px'

# Paleta de los MODOS de variable en el grafo. Deliberadamente fuera de la familia de los
# grupos KMeans (rojos/naranjas) y de los equipos: un rojo en el mapa y un rojo en el grafo
# significarian cosas sin ninguna relacion. Se recorre en el orden de las modalidades del
# artefacto.
PALETA_MODALIDADES = ['#0d9488', '#be185d']   # verde azulado y rosa oscuro

# Guion horizontal negro en cada extremo de CADA vano, tenga o no eventos, igual que el
# mapa de 01: grados de longitud a cada lado del extremo (~14 m a esta latitud). Marca
# donde empieza y donde termina un vano, que es lo unico que distingue dos vanos vecinos
# dibujados con el mismo color.
MARCA_VANO = 0.00013
# Densificacion del hover del mapa: el hover de una traza de lineas en Scattermap se
# resuelve contra los VERTICES y no contra la linea, y los tramos de MVLINSEC traen
# exactamente dos. Sin esto el centro de un vano largo no muestra etiqueta y, como Plotly
# solo convierte un clic en evento donde hay hover, tampoco se puede marcar tocandolo ahi.
PASO_VERTICE = 0.00022      # grados ~= 25 m a esta latitud



In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aqui, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
# La geometria en si (no solo su sha1): la nube KMeans de la fila 3 tiene que dibujarse en
# el MISMO espacio en que se asignan las clases -- el canonico '2' es (log_x=False,
# log_y=True). Leerlo de la geometria y no fijarlo a mano evita que un cambio de espacio
# deje la nube en ejes que ya no corresponden a las fronteras.
GEOMETRIA_014 = cargar_geometria_014(GEOMETRIAS_PATH, CLAVE_ESPACIO)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...) | '
      f'espacio {CLAVE_ESPACIO}: log_x={GEOMETRIA_014.logs[0]}, log_y={GEOMETRIA_014.logs[1]}')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
COSTOS_ITEMS_PATH = ROOT / 'data' / 'COSTOS ITEMS CONTRATOS.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
# Sin `.copy()`: `reset_index` ya devuelve un objeto propio, y `df_original_copy` es --
# como dice su nombre -- una copia que el pipeline hizo para nosotros. Copiar otra vez
# dejaba DOS juegos vivos: medido, 506 MB duplicados entre `datos` y estas variables.
n_filas_x = len(datos['X'])
Xdf = datos['Xdata'].reset_index(drop=True)
context_df = datos['df_original_copy'].reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

# Ya no se construye el escalador min-max de MGCECDL. El simulador y la importancia de
# variables corren sobre el modelo MIL del cuaderno 05, cuya matriz de instancias es RAW,
# asi que `preparar_splits_estratificados` + `escalar_features_minmax_mgcecdl` -- de lo
# mas caro de esta celda -- no alimentaban ya a nadie.
assert len(context_df) == n_filas_x, (
    'context_df y la matriz de features deben quedar alineados fila a fila'
)
# `datos` se suelta AQUI, en la misma celda que lo creo: mantiene la celda re-ejecutable
# -- volver a correrla lo reconstruye -- y evita que el dict siga sosteniendo la matriz
# cruda y los dos DataFrame despues de que ya los tenemos.
# Aqui vivia `X_raw_model`, 44,7 MB de float32 que solo se usaban para este `len`.
del datos
gc.collect()
print(f'{len(context_df):,} filas | {len(feature_names)} features')

In [ ]:
# --- UN solo modelo: el MIL por bolsas del cuaderno 05 (cierra el SEAM D1) ------------
# El tablero entero -- mapa "Criticidad Simulada", grafo reconstruido e "Importancia
# Variables" -- responde a este modelo y a esta unidad: la BOLSA (vano x ventana), que es
# la unidad en la que 04 define la criticidad. MGCECDL por fila salio del cuaderno: tener
# dos modelos contestando paneles vecinos del mismo tablero significaba que el panel y el
# mapa hablaban de cosas distintas sin que nada en pantalla lo dijera.
# Requiere DOS artefactos que produce el cuaderno 05 y que viven bajo `data/` (ignorado
# por git): el modelo y el cache de bolsas. Falla AQUI, con el nombre del cuaderno que los
# genera, en vez de a los diez minutos en la celda del boton.
RUTA_MODELO_MIL = MODEL_DIR / 'mil_vano_ventana_v1.pt'
RUTA_BOLSAS_MIL = ROOT / 'data' / 'derived' / 'bolsas_mil_full.joblib'
for _ruta in (RUTA_MODELO_MIL, RUTA_BOLSAS_MIL):
    assert _ruta.exists(), (
        f'Falta {_ruta.name}: lo produce 05_mil_vano_ventana.ipynb. Corre ese cuaderno '
        'antes que este.'
    )

BOLSAS = cargar_bolsas(RUTA_BOLSAS_MIL)
# `float32` y no el `float64` del artefacto: los pesos del modelo son float32, asi que la
# conversion ya ocurria en cada llamada y la mitad de cada numero se descartaba. Medido
# sobre 523 bolsas de 3 circuitos con un override aplicado: la clase simulada y el UITI
# salen IDENTICOS BIT A BIT, y la matriz baja de 184,7 a 92,4 MB.
X_INST = np.asarray(BOLSAS['X'], dtype=np.float32)
FEATURES_MIL, BAG_INDEX = BOLSAS['features'], BOLSAS['bag_index']
# El dict del artefacto sigue sosteniendo la version float64: sin soltarlo, el ahorro
# seria un tercer juego de la misma matriz en vez de un reemplazo.
del BOLSAS
gc.collect()
# `device='cpu'` a proposito y no DEVICE: una seleccion son decenas de instancias, asi que
# el traslado a GPU/MPS cuesta mas de lo que ahorra, y saca una variable de dtype de en
# medio de un camino interactivo.
MIL = cargar_modelo_mil(RUTA_MODELO_MIL, device='cpu', features_esperadas=FEATURES_MIL)

# La guarda que hace comparables los dos mapas: si el MIL se hubiera entrenado con OTRA
# geometria KMeans, sus clases usarian los mismos 4 colores para significar otra cosa.
for _campo in ('offset', 'scale', 'centroides'):
    assert np.allclose(getattr(MIL.geometria, _campo), getattr(GEOMETRIA_014, _campo)), (
        f'La geometria del modelo MIL difiere de la de 01.4 en {_campo}: sus clases NO son '
        'las del mapa base y no pueden compartir la paleta.'
    )
assert tuple(MIL.geometria.logs) == tuple(GEOMETRIA_014.logs)

# Las 70 primeras features del MIL son exactamente las de MGCECDL (22 estaticas + 48 de
# clima); las 10 restantes son COD_CAUSA y sus indicadores, que no son controles del
# simulador. Por eso el catalogo de knobs de la celda siguiente sirve para los dos.
assert list(FEATURES_MIL[:len(feature_names)]) == list(feature_names), (
    'Las features del MIL ya no empiezan por las de MGCECDL: el catalogo de knobs '
    'apuntaria a columnas equivocadas.'
)
# Los MODOS de variable con los que el modelo agrupa las columnas -- son los mismos que
# usa la fusion FiLM (el clima reescala lo estructural), asi que colorear los nodos del
# grafo por modalidad muestra exactamente la particion que el modelo usa por dentro.
COLUMNAS_MODALIDAD = {m: set(int(i) for i in idx)
                      for m, idx in MIL.model.base.modality_feature_indices.items()}
MODALIDADES_MIL = list(COLUMNAS_MODALIDAD)
assert len(MODALIDADES_MIL) == len(PALETA_MODALIDADES), (
    f'El artefacto trae {len(MODALIDADES_MIL)} modalidades y la figura tiene trazas de '
    f'nodo para {len(PALETA_MODALIDADES)}: agrega la traza que falta antes de seguir.'
)
COLORES_MODALIDAD = dict(zip(MODALIDADES_MIL, PALETA_MODALIDADES))

print(f'MIL cargado -- {len(BAG_INDEX.keys):,} bolsas | {X_INST.shape[0]:,} instancias x '
      f'{len(FEATURES_MIL)} features | geometria identica a 01.4')
print('modos de variable: ' + ' | '.join(
    f'{m} ({len(COLUMNAS_MODALIDAD[m])})' for m in MODALIDADES_MIL))

In [ ]:
# --- construir_ventanas + per-(vano, ventana) events + caches (design section A) -------
VENTANAS = construir_ventanas(context_df['FECHA'])
TABLA = construir_tabla_vano_ventana(context_df, VENTANAS)
mask_para = construir_mask_cache(TABLA)
clases_para = construir_hist_class_cache(TABLA, mask_para)

# Aqui se clasificaban las 111 mil celdas de una sola pasada y se submuestreaba una nube
# de 20 mil puntos para el panel KMeans. Ese panel ya no existe, asi que el calculo
# tampoco: eran una pasada de `cargar_clases_desde_014` sobre la tabla entera y ~1,2 MB de
# coordenadas que nadie iba a dibujar. La clase que el mapa necesita la sigue dando
# `clases_para`, ventana por ventana y cacheada.

CIRCUITOS = sorted(TABLA['CIRCUITO'].astype(str).unique())
VANOS_POR_CIRCUITO = {
    c: sorted(g['FID_VANO'].unique().tolist())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}
# Las ventanas en que ESE circuito registro al menos un evento. No son las once para
# todos: un circuito tranquilo puede no tener una sola celda en media ventana del ano, y
# el deslizador lo llevaba igual hasta ahi -- a un mapa sin un solo tramo de color, que se
# lee como que el tablero se rompio y no como que no hubo eventos. Se recorta a lo que
# existe, asi que cada posicion del deslizador tiene algo que mostrar.
VENTANAS_POR_CIRCUITO = {
    c: sorted(int(i) for i in g['ventana_i'].unique())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}
_RECORRIDO = [len(v) for v in VENTANAS_POR_CIRCUITO.values()]
print(f'ventanas con eventos por circuito: de {min(_RECORRIDO)} a {max(_RECORRIDO)} '
      f'de {len(VENTANAS)}')

print(f'{len(TABLA):,} celdas vano x ventana con eventos | {len(VENTANAS)} ventanas | '
      f'{TABLA["FID_VANO"].nunique():,} vanos distintos | {len(CIRCUITOS)} circuitos')


# Geometria FISICA de cada vano (no confundir con la geometria KMeans de la celda 4): mismo
# shapefile y mismo join que el mapa de 01.3/01.4. No se extrae a src/ porque es solo
# lectura + reindexado geoespacial, sin logica propia que valga la pena testear por fuera
# de lo que TABLA/capas_mapa_historico ya cubren.
def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])
_utiles = _lineas[_lineas['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]

GEO_POR_CIRCUITO = {}
for _c, _g in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_g['FID_VANO_GEO'], _g.geometry):
        if _geom is None or _geom.is_empty:
            continue
        for _p in ([_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))):
            xs, ys = _p.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        # `bounds` es lo que permite encuadrar el mapa sobre el circuito elegido, igual
        # que 01.4: [lat_min, lat_max, lon_min, lon_max].
        _la = [v for l in lats for v in l]
        _lo = [v for l in lons for v in l]
        GEO_POR_CIRCUITO[_c] = {
            'fids': fids, 'lat': lats, 'lon': lons,
            'bounds': [round(min(_la), 5), round(max(_la), 5),
                       round(min(_lo), 5), round(max(_lo), 5)],
        }


def _equipo(nombre):
    """Transformadores e interruptores del circuito, igual que 01.4 celda 5. Si el
    shapefile no esta, el mapa se dibuja sin equipos en vez de fallar: son contexto
    de lectura, no el dato del tablero."""
    ruta = ROOT / 'data' / 'GEO' / nombre
    if not ruta.exists():
        return {}
    g = gpd.read_file(ruta)
    if str(g.crs) != 'EPSG:4326':
        g = g.to_crs('EPSG:4326')
    g = g[g['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    return {c: {'lat': [round(float(p.y), 5) for p in gg.geometry],
                'lon': [round(float(p.x), 5) for p in gg.geometry]}
            for c, gg in g.groupby(g['CIRCUITO'].astype(str))}


TRAFOS = _equipo('GDBCHEC_TRANSFOR.shp')
SWITCHES = _equipo('SWITCHES.shp')

# UITI y eventos por vano y ventana: solo alimentan el hover, igual que 01.4. El grupo
# NO se guarda aqui -- sale de `clases_para`, que es la unica fuente de clases.
DATOS_VENTANA = [{} for _ in VENTANAS]
for _fid, _vi, _u, _n in zip(TABLA['FID_VANO'], TABLA['ventana_i'],
                             TABLA['uiti_acumulado'], TABLA['num_eventos']):
    DATOS_VENTANA[int(_vi)][str(_fid)] = (float(_u), int(_n))

# El shapefile crudo se suelta en la MISMA celda que lo leyo -- 76 MB entre las dos
# tablas --: `GEO_POR_CIRCUITO` ya tiene lo unico que el tablero mira, y volver a correr
# esta celda lo vuelve a leer, asi que la celda sigue siendo re-ejecutable.
del _lineas, _utiles
gc.collect()

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria fisica | '
      f'{sum(len(v["lat"]) for v in TRAFOS.values()):,} transformadores | '
      f'{sum(len(v["lat"]) for v in SWITCHES.values()):,} switches')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

In [ ]:
# --- Que variables tiene sentido simular, y cuales no ---------------------------------
# El catalogo de knobs se construye desde la lista de features y no sabe que significa
# ninguna: ofrece el nivel de riesgo por vegetacion -- que se baja con la cuadrilla de la
# semana entrante -- al lado de las coordenadas del vano, como si mover un vano fuera una
# opcion de mantenimiento. Esa distincion vivia solo en la cabeza de quien lee.
# Aqui queda como dato: rango real de cada control y veredicto sobre si simularlo
# significa algo, con el motivo pegado para poder discutirlo. Los veredictos salen del
# diccionario del propio proyecto (`data/Variables_seleccion.xlsx`), citado en cada
# motivo, y viven en `simulador_variables.JUICIO_SIMULACION`, con sus pruebas.
# Una variable nueva del modelo aparece como "Sin evaluar" y no como una palanca mas:
# suponerla valida meteria un control sin revisar en el panel.
TABLA_VARIABLES = tabla_variables_simulables(KNOBS)

_COLOR_VEREDICTO = {
    'Si -- intervencion': '#dcfce7',   # verde: hay una obra detras
    'Si -- escenario': '#dbeafe',      # azul: nadie lo controla, pero es el what-if
    'Limitado': '#fef3c7',             # ambar: una sola lectura lo hace interpretable
    'No': '#fee2e2',                   # rojo: circular o identidad del vano
    'Sin evaluar': '#e5e7eb',
}
_conteo = TABLA_VARIABLES['Sentido de simular'].value_counts()
print(f'{len(TABLA_VARIABLES)} controles simulables -- '
      + ', '.join(f'{_conteo.get(v, 0)} {v}' for v in _COLOR_VEREDICTO if _conteo.get(v, 0)))
print('Las constantes quedan fuera: un control con un solo valor observado no mueve nada.')
_sin_unidad = int((TABLA_VARIABLES['Unidad'] == '').sum())
print(f'{_sin_unidad} sin unidad: categoricas, binarias, indices y las que el diccionario '
      'del proyecto no documenta (ver `simulador_variables.UNIDADES`).')

# `Por que` es una frase entera: sin `white-space: normal` pandas la muestra en una sola
# linea y la tabla se sale de la celda. El ancho se fija por columna para que el motivo
# se lleve el espacio y el rango no.
display(
    TABLA_VARIABLES.style
    .hide(axis='index')
    .format({'vmin': '{:,.4g}', 'vmax': '{:,.4g}'}, na_rep='--')
    .map(lambda v: f'background-color: {_COLOR_VEREDICTO.get(v, "")}',
         subset=['Sentido de simular'])
    .set_properties(**{'white-space': 'normal', 'vertical-align': 'top',
                       'font-size': '12px'})
    .set_properties(subset=['Por que'], **{'width': '460px', 'color': '#4b5563'})
    .set_properties(subset=['Variable'], **{'font-weight': '600'})
    .set_properties(subset=['Unidad'], **{'white-space': 'nowrap'})
    .set_table_styles([{'selector': 'th',
                        'props': [('font-size', '12px'), ('text-align', 'left')]}])
)


In [ ]:
# --- El catalogo de actividades del contrato -------------------------------------------
# El simulador contesta que le pasa al riesgo del vano; esto contesta cuanto cuesta el
# plan. Es la otra mitad de la decision de mantenimiento, y juntas producen la frase con
# la que se aprueba una orden de trabajo: "baja un grupo de criticidad por 283.472 pesos".
#
# Las dos mitades NO estan atadas entre si, y es a proposito. Marcar "PODA EN REDES
# RURALES TIPO A" no mueve `NR_T`, y bajar `NR_T` no programa una poda: el modelo no
# tiene ningun mapa de actividades del contrato a features, e inventarlo produciria un
# numero con toda la pinta de ser el beneficio estimado de esa actividad sin serlo. Lo
# que el panel pone lado a lado es el efecto simulado y el costo cotizado del plan que
# el usuario dice que ejecutaria; el puente entre los dos se queda en su cabeza, que es
# el sitio honesto hasta que CHEC entregue esa correspondencia.
#
# El libro es una exportacion de tabla dinamica, y `leer_catalogo_costos` se ocupa de
# las dos trampas que trae: su ultima fila es el pie `Total general` -- que ofrecido
# como actividad agrega 254 mil pesos de puro artefacto -- y doce filas no traen costo
# unitario. Esas doce no se pueden costear, pero tampoco se hacen desaparecer.
CATALOGO_COSTOS = leer_catalogo_costos(COSTOS_ITEMS_PATH)
COSTO_POR_ITEM = CATALOGO_COSTOS.por_nombre

print(f'{len(CATALOGO_COSTOS.items)} actividades costeables '
      f'({min(COSTO_POR_ITEM.values()):,.0f} a {max(COSTO_POR_ITEM.values()):,.0f} COP)')
if CATALOGO_COSTOS.sin_costo:
    print(f'{len(CATALOGO_COSTOS.sin_costo)} sin costo unitario en el libro, fuera de la '
          f'lista: no se puede costear lo que no tiene precio.')
    for _nombre in CATALOGO_COSTOS.sin_costo:
        print(f'  - {_nombre}')


## La matematica del ranking: como se busca el grupo Bajo

Esta celda describe lo que calcula la fila 3, columnas 3-4 del tablero de abajo, y el
plan que aparece bajo el panel. Va **antes** del tablero a proposito: sin ella, las
barras se leen como "importancia" generica, que es justo lo que NO son.

### La pregunta

No es *que variable explica el UITI de este vano* -- esa es la pregunta de SHAP -- sino
**que muevo, y a que valor, para que este vano baje al grupo Bajo**. Son preguntas
distintas y sus respuestas no coinciden: sobre un vano real, el ranking por sensibilidad
y el ranking por caida alcanzable **no comparten ni una** de sus cinco primeras
variables.

### La meta, que si existe

La clase de una bolsa sale de la geometria KMeans de 01.4 sobre
$\zeta_b=(n_b,\ \log_{10}\hat u_b)$, y $n_b$ -- los eventos observados -- **no se simula
nunca**. Con $n_b$ fijo, la clase solo depende de $\hat u$, asi que existe un umbral:

$$u^{\star}(n_b)=\max\{\,u>0 \;:\; \arg\min_k\lVert\zeta(n_b,u)-c_k\rVert = 0\,\}$$

Se resuelve por **rejilla** y no por biseccion: nada garantiza que al subir $u$ con $n_b$
fijo se recorran los grupos en orden, y una biseccion asume esa monotonia.

Medido sobre la geometria real, $u^{\star}$ existe en todo el rango de eventos observado,
pero **se desploma** al acumularse eventos:

| $n_b$ | 1 | 5 | 10 | 20 | 30 | 46 |
|---|---|---|---|---|---|---|
| $u^{\star}$ | 4,41 | 3,93 | 3,37 | 1,15 | 0,114 | 0,0029 |

Un vano con muchos eventos necesita un UITI casi nulo para bajar de grupo. Es una
propiedad del espacio de criticidad, no del simulador.

### El ranking: una variable a la vez

Para cada control $\kappa$ que el panel ofrece se recorre su conjunto de candidatos
$\mathcal{G}_\kappa$ -- una rejilla de $G=9$ valores si es numerico, **sus categorias** si
es categorico -- moviendo sus columnas $F(\kappa)$ a la vez, y se guarda el valor que
**minimiza** el UITI de cada bolsa:

$$v^{\star}_{b,\kappa}=\arg\min_{v\in\mathcal{G}_\kappa}\hat u_b\!\left(X^{\kappa\to v}\right),
\qquad
\hat u^{\star}_{b,\kappa}=\min_{v\in\mathcal{G}_\kappa}\hat u_b\!\left(X^{\kappa\to v}\right)$$

La barra mide la **caida en ordenes de magnitud**, que es el eje que usa la geometria, y
el hover trae la fraccion del camino que cubre:

$$c_{b,\kappa}=\log_{10}\hat u_b(X)-\log_{10}\hat u^{\star}_{b,\kappa},
\qquad
\text{avance}_{b,\kappa}=\frac{c_{b,\kappa}}{\log_{10}\hat u_b(X)-\log_{10}u^{\star}(n_b)}$$

En unidades de UITI el ranking de un vano caro seria incomparable con el de uno barato;
en ordenes de magnitud, dos barras de la misma altura significan lo mismo en cualquier
grupo. La barra se pinta **verde** cuando $\hat u^{\star}_{b,\kappa}\le u^{\star}(n_b)$:
esa sola variable cambia de grupo.

**Se recorren todos los controles del panel, tambien los categoricos.** El barrido
anterior los saltaba, y con ellos se caian del ranking el conductor, el calibre del
neutro y el tipo de proteccion -- tres obras que CHEC ejecuta. Ademas el top **reserva
sitio para los dos grupos**, intervencion y escenario: sin la reserva, las cuatro
familias climaticas copan la lista y no queda ni una palanca que una cuadrilla pueda
ejecutar.

### Por que dejo de ser un barrido min-max

Los dos defectos, los dos medidos sobre este modelo:

- $s=\max(|\Delta^-|,|\Delta^+|)$ **no lleva signo**: una variable que dispara el riesgo
  en los dos extremos encabezaba el ranking, y la cabeza de la lista se llenaba de
  palancas que no hay que tocar.
- Solo miraba los **dos extremos**. **10 de los 15** controles numericos tienen su optimo
  en el INTERIOR del rango para alguna bolsa (`DDT` para todas): la funcion no es
  monotona y los extremos son los dos puntos equivocados.

### El plan: la combinacion, porque una variable casi nunca basta

Medido sobre **59 bolsas de 40 circuitos**:

| Grupo | Bolsas | Alcanzan Bajo con UNA variable | Avance de la mejor (mediana) |
|---|---|---|---|
| Medio | 33 | 20 / 33 (**61%**) | 100% |
| Medio-Alto | 18 | **0 / 18 (0%)** | 60% |
| Alto | 8 | **0 / 8 (0%)** | 49% |

En Medio-Alto y Alto **ninguna variable sola alcanza jamas**. No es la rareza de un vano:
es el caso normal justo en los grupos donde la pregunta de mantenimiento pesa. Por eso el
plan combinado no es un adorno del ranking, es su continuacion.

Es un **descenso por coordenadas, goloso**. Partiendo del estado observado $X^{(0)}=X$, en
cada ronda se prueban todos los candidatos de todos los controles que ese vano no haya
usado todavia, y se aplica el que mas baja su UITI:

$$(\kappa_t,v_t)=\arg\min_{\kappa\notin\mathcal{U}_{t-1},\,v\in\mathcal{G}_\kappa}
\hat u_b\!\left(X^{(t-1),\,\kappa\to v}\right),
\qquad
\mathcal{U}_t=\mathcal{U}_{t-1}\cup\{\kappa_t\}$$

Se detiene al cumplir $\hat u_b\le u^{\star}(n_b)$, al agotar los $T=4$ pasos, o cuando
ningun candidato mejora. Cada control entra **como mucho una vez**: un plan que reajusta
dos veces la misma variable no es una orden de trabajo mas barata, es la misma obra
contada dos veces.

**Goloso y no exhaustivo, dicho de frente.** Con 18 controles y 9 valores, dos cambios
simultaneos ya son 13 mil combinaciones y cuatro son 26 millones -- fuera del presupuesto
de un boton que debe sentirse inmediato. El plan es **bueno, no demostrablemente el
minimo**. Y cuando ni moviendolo todo se llega, se reporta lo conseguido y se dice que no
alcanza, que vale mas que un plan que insinua lo contrario.

### Lo que cuesta

| Paso | Pasadas de bolsas |
|---|---|
| Mapa simulado (base + simulado) | 2 |
| Compuertas para el grafo | 1 |
| Ranking (base compartida + rejilla) | $1+G\,K$ |
| Plan (por ronda, hasta agotar los pendientes) | $\le T\,G\,K$ |

$K$ son los controles que el panel ofrece y $G$ sus candidatos. Las rondas del plan se
**comparten entre vanos**: un candidato se aplica a la vez sobre el estado propio de cada
bolsa y la pasada devuelve un $\hat u$ por bolsa, asi que cada vano elige su mejor paso
sin que la ronda cueste una tanda por vano.


In [ ]:
# --- Inventario de trazas -------------------------------------------------------------
# La figura tiene SIETE paneles. Los dos mapas ocupan el cuadrante superior, uno al lado
# del otro; las filas 3 y 4 responden cuatro preguntas distintas sobre los vanos elegidos
# -- como vienen en el tiempo, que variable mueve a CADA uno, como se relacionan esas
# variables entre si, y cuanto cambia el UITI al simular --; y la fila 5, a todo lo ancho,
# contesta la que convierte el analisis en una orden de trabajo: cuanto cuesta.
# Salieron la nube KMeans, la barra de importancia agregada de la seleccion -- la
# reemplaza el top 5 POR VANO, que es la pregunta que sostiene una orden de trabajo -- y
# el violin de eventos por grupo, cuyo hueco pasa a comparar medido contra simulado.
#
# El inventario ya no esta congelado por indices sueltos: se ARMA sobre la marcha y `IDX`
# se llena con lo que devuelve `add_trace`. Congelarlo a mano tenia sentido cuando las
# trazas se agregaban de a una en PRs sucesivos; ahora el orden lo fija este bloque y
# cualquier reordenamiento se ve aqui mismo.
IDX = {}

# La caja amarilla del vano seleccionado vive en el LAYOUT del mapa base y no en el
# inventario de trazas: `below='traces'` la deja debajo de todos los tramos, con lo que no
# intercepta ni el hover ni el clic -- que es justo lo que alterna la seleccion -- y no
# tapa el color de clase del vano que esta senialando. Nace vacia; el repintado del mapa
# le escribe el `source`.
CAPA_CAJA_SELECCION = dict(
    sourcetype='geojson', type='fill', below='traces',
    source={'type': 'FeatureCollection', 'features': []},
    color=COLOR_CAJA_SELECCION, opacity=OPACIDAD_CAJA_SELECCION,
)
# El mapa simulado lleva TRES capas de caja, una por desenlace, por la misma razon por la
# que la de la izquierda es una sola: una capa pinta con UN color. Nacen vacias y siempre
# en el mismo orden (`CAMBIOS`), asi que el repintado es una escritura de `source` por capa
# y nunca un quitar y poner capas del mapa -- que en MapLibre reordena lo que hay debajo.
COLOR_POR_CAMBIO = {CAMBIO_MEJORA: COLOR_CAJA_MEJORA,
                    CAMBIO_IGUAL: COLOR_CAJA_IGUAL,
                    CAMBIO_EMPEORA: COLOR_CAJA_EMPEORA}
CAPAS_CAJA_SIMULADA = [
    dict(sourcetype='geojson', type='fill', below='traces',
         source={'type': 'FeatureCollection', 'features': []},
         color=COLOR_POR_CAMBIO[_cambio], opacity=OPACIDAD_CAJA_SELECCION)
    for _cambio in CAMBIOS
]
IDX_CAPA_CAMBIO = {_cambio: _i for _i, _cambio in enumerate(CAMBIOS)}


def _agregar(traza, fila, columna, **kwargs):
    """Agrega la traza y devuelve su indice, que es lo unico que el resto del cuaderno
    necesita saber de ella."""
    _fig.add_trace(traza, row=fila, col=columna, **kwargs)
    return len(_fig.data) - 1


_fig = make_subplots(
    rows=6, cols=4,
    specs=[[{'type': 'map', 'rowspan': 2, 'colspan': 2}, None,
            {'type': 'map', 'rowspan': 2, 'colspan': 2}, None],
           [None, None, None, None],
           [{'type': 'xy', 'colspan': 2, 'secondary_y': True}, None,
            {'type': 'xy', 'colspan': 2}, None],
           # A todo lo ancho: son hasta once grupos de dos barras -- diez vanos mas el
           # circuito entero -- y a media fila los nombres de vano se encimaban.
           [{'type': 'xy', 'colspan': 4}, None, None, None],
           # A todo lo ancho: el costo es la conclusion del tablero, no un panel mas
           # que se compare con el de al lado.
           [{'type': 'xy', 'colspan': 4}, None, None, None],
           # El grafo va CENTRADO y a media fila: es circular, asi que a lo ancho
           # completo quedaria un disco pequenio con dos franjas vacias a los lados.
           [None, {'type': 'xy', 'colspan': 2}, None, None]],
    subplot_titles=(
        'Criticidad Original',
        'Criticidad Simulada',
        'UITI acumulado y eventos por ventana',
        f'Top {TOP_VARIABLES_POR_VANO}: que baja el UITI de cada vano',
        'UITI acumulado: medido contra simulado',
        'Costo de la intervencion',
        'Cuanto movio la simulacion el grafo',
    ),
    # La ultima fila se lleva casi un cuarto: el grafo es circular y su diametro lo fija
    # la dimension MENOR del panel. Con el reparto anterior el panel media 160 px de alto
    # y el circulo salia de 106 px, con 66 nombres alrededor.
    row_heights=[0.14, 0.14, 0.17, 0.17, 0.14, 0.24],
    # 0,05 y no 0,035: en el hueco entre la serie de tiempo y el top 5 tienen que caber
    # CUATRO cosas -- las marcas del eje secundario de la serie, su rotulo "Eventos", el
    # rotulo "Relevancia variables" del top y las marcas de ese eje. Medido a 1.900 px,
    # con 0,035 el hueco era de 64 px y las marcas del eje secundario ya llegaban a 21 de
    # esos: los dos rotulos se encimaban.
    # 0,06 y no 0,095: el espaciado vertical es una fraccion POR HUECO, y al pasar de
    # cinco filas a seis los huecos pasaron de cuatro a cinco. Con 0,095 se llevaban el
    # 47,5% del alto util y no quedaba panel que repartir.
    horizontal_spacing=0.055, vertical_spacing=0.06,
)

# --- Mapa base (filas 1-2, columnas 1-2) ---------------------------------------------
IDX['clases'] = [
    _agregar(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase], legendgroup='hist',
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), 1, 1) for _clase in range(4)
]
IDX['sin_dato'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Sin evento en la ventana', legendgroup='hist',
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO), hovertext=[], hoverinfo='text',
), 1, 1)
# El marcado va en DOS capas, como en 01.4: primero el halo blanco ancho y despues la
# linea con el color de SU clase. Un color plano de "seleccionado" encima congelaria lo
# que se ve: la ventana cambia la clase por debajo y el vano seguiria igual en pantalla.
IDX['marcados'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='hist', showlegend=False,
    line=dict(width=ANCHO_HALO, color=COLOR_HALO), hovertext=[], hoverinfo='text',
), 1, 1)
IDX['marcados_clases'] = [
    _agregar(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase], legendgroup='hist',
        showlegend=False, line=dict(width=ANCHO_MAPA_MARCADO, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), 1, 1) for _clase in range(4)
]
IDX['marcados_sin_dato'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Marcado sin eventos', legendgroup='hist',
    showlegend=False, line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), 1, 1)

# --- Mapa simulado (filas 1-2, columnas 3-4) -----------------------------------------
IDX['pred_clases'] = [
    _agregar(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase], legendgroup='pred',
        showlegend=False, line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), 1, 3) for _clase in range(4)
]
IDX['pred_sin_dato'] = _agregar(go.Scattermap(
    lat=[], lon=[], mode='lines', name='Sin evento / no simulado', legendgroup='pred',
    showlegend=False, line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), 1, 3)

# Los equipos van DESPUES de los tramos para dibujarse encima. Se repiten por mapa porque
# una traza pertenece a un solo subplot: sin ellos la derecha se leeria como otra geografia.
for _columna_mapa, _leyenda in ((1, True), (3, False)):
    _claves = ('trafos', 'switches') if _columna_mapa == 1 else ('pred_trafos', 'pred_switches')
    for _clave, _nombre, _color, _tam in zip(
            _claves, ('Transformadores', 'Switches'), (COLOR_TRAFO, COLOR_SWITCH), (6, 5)):
        IDX[_clave] = _agregar(go.Scattermap(
            lat=[], lon=[], mode='markers', name=_nombre, legendgroup='equipos',
            showlegend=_leyenda, marker=dict(size=_tam, color=_color),
            hovertext=[], hoverinfo='text',
        ), 1, _columna_mapa)

# --- Fila 3, columnas 1-2: la serie de tiempo de los vanos elegidos --------------------
# Paridad con 03 y 04: la LINEA y el ANILLO del punto llevan el color de identidad del
# vano -- dicen de QUE vano es la serie -- y el RELLENO del punto lleva el color del grupo
# de riesgo en que cayo ese vano en esa ventana. Son dos codigos sobre el mismo dato,
# separados por canal para que los dos se lean a la vez. Un relleno gris es una ventana sin
# celda, que no tiene grupo -- distinto de caer en el mas bajo. Doble eje porque UITI y eventos viven en escalas muy distintas y
# compartir eje aplastaria una de las dos.
# `marker.size` es un ARRAY: el punto de la ventana activa va al triple y viaja con el
# deslizador sin partir la serie en una segunda traza.
_VACIO = [None] * len(VENTANAS)
# El eje x son los INDICES de ventana, pero las marcas llevan la FECHA en que empieza
# cada una y no su etiqueta "V1", "V2": un rotulo "V7" obliga a ir a buscar a que periodo
# corresponde cada vez que se mira el panel. El ano se recorta como en 03 y 04 -- todas
# las ventanas caen en el mismo -- para que las once quepan sin apilarse.
_X_VENTANAS = [v['i'] for v in VENTANAS]
_FECHAS_VENTANA = [str(v['desde'].date()) for v in VENTANAS]
_ANIOS_VENTANA = sorted({f[:4] for f in _FECHAS_VENTANA})
_TICKS_VENTANA = ([f[5:] for f in _FECHAS_VENTANA] if len(_ANIOS_VENTANA) == 1
                  else _FECHAS_VENTANA)
IDX['serie_uiti'] = [
    _agregar(go.Scatter(
        x=_X_VENTANAS, y=list(_VACIO), mode='lines+markers', name='', showlegend=False,
        line=dict(color=COLORES_VANOS[_s], width=2),
        marker=dict(size=[SERIE_TAM_UITI] * len(VENTANAS),
                    color=[COLOR_SIN_GRUPO] * len(VENTANAS),
                    line=dict(width=1.2, color=COLORES_VANOS[_s])),
        hovertext=[], hoverinfo='text', connectgaps=False,
    ), 3, 1, secondary_y=False) for _s in range(MAX_VANOS_ANALISIS)
]
IDX['serie_eventos'] = [
    _agregar(go.Scatter(
        x=_X_VENTANAS, y=list(_VACIO), mode='lines+markers', name='', showlegend=False,
        line=dict(color=COLORES_VANOS[_s], width=1.1, dash='dot'),
        marker=dict(size=[SERIE_TAM_EVENTOS] * len(VENTANAS), symbol='square',
                    color=[COLOR_SIN_GRUPO] * len(VENTANAS),
                    line=dict(width=1.1, color=COLORES_VANOS[_s])),
        hovertext=[], hoverinfo='text', connectgaps=False,
    ), 3, 1, secondary_y=True) for _s in range(MAX_VANOS_ANALISIS)
]
_fig.update_xaxes(title_text=('Inicio de la ventana'
                              + (f' ({_ANIOS_VENTANA[0]})' if len(_ANIOS_VENTANA) == 1 else '')),
                  tickmode='array', tickvals=_X_VENTANAS, ticktext=_TICKS_VENTANA,
                  tickangle=-45, tickfont=dict(size=8), row=3, col=1)
# Eje LINEAL. Con `type='log'` las ventanas sin eventos -- que ahora valen cero, no un
# hueco -- desaparecian del panel: `log(0)` no existe y Plotly descarta el punto en
# silencio, asi que la secuencia completa de ventanas que el eje promete no se dibujaba.
_fig.update_yaxes(title_text='UITI acumulado', rangemode='tozero',
                  row=3, col=1, secondary_y=False)
# Marcas mas pequenias en los dos ejes que comparten el hueco: cada digito que se
# ahorran es espacio para separar sus rotulos, que estaban a 5 px uno del otro.
_fig.update_yaxes(title_text='Eventos', rangemode='tozero', showgrid=False,
                  title_standoff=2, tickfont=dict(size=9),
                  row=3, col=1, secondary_y=True)

# --- Fila 3, columnas 3-4: el top de variables POR VANO -------------------------------
# Un grupo de barras por vano. Cada traza es una POSICION del ranking (la 1a, la 2a...),
# no una variable: las variables cambian de vano a vano, asi que una traza por variable
# necesitaria tantas como el catalogo entero y casi todas vacias.
# El nombre de la variable va DENTRO de la barra: con cinco grupos de diez barras no hay
# sitio para una leyenda de cincuenta entradas, y el rotulo pegado al dato no obliga a
# cruzarlo. Cual de los tres rotulos posibles -- resumen, inicial o ninguno -- se escribe
# lo decide el repintado segun lo que mida cada barra, y el nombre completo esta siempre en
# la etiqueta del mouse.
# El color codifica la POSICION en el ranking y nada mas. Antes salia de `COLORES_GRUPOS`,
# lo que con diez posiciones dejaba siete del mismo rojo oscuro y, peor, invitaba a leer
# una barra como un grupo de criticidad, que es otra cosa. Una rampa de opacidad sobre UN
# color dice "primera, segunda, tercera" sin pedir prestada la paleta del mapa.
# El color por defecto codifica la POSICION en el ranking y nada mas: una rampa de
# opacidad sobre UN color, deliberadamente ajena a la paleta de los grupos para que una
# barra no se lea como un grupo de criticidad. El repintado la sobrescribe con VERDE en
# las barras cuya variable, sola, ya baja al vano al grupo Bajo.
COLOR_POSICION_BARRA = [f'rgba(203,24,29,{0.95 - 0.055 * _p:.2f})'
                        for _p in range(TOP_VARIABLES_POR_VANO)]
IDX['top_vano'] = [
    _agregar(go.Bar(
        x=[], y=[], name=f'{_p + 1}o', showlegend=False,
        text=[], textposition='inside', insidetextanchor='middle',
        textangle=-90, constraintext='none',
        insidetextfont=dict(size=TAM_FUENTE_BARRA, color='white'),
        marker=dict(color=[], line=dict(width=0.4, color='rgba(60,10,10,0.6)')),
        hovertext=[], hoverinfo='text',
    ), 3, 3) for _p in range(TOP_VARIABLES_POR_VANO)
]
_fig.update_yaxes(title_text='Caida de UITI alcanzable (ordenes de magnitud)',
                  title_standoff=2, tickfont=dict(size=9), row=3, col=3)
_fig.update_xaxes(title_text='Vano', tickfont=dict(size=9), row=3, col=3)

# --- Fila 6, columnas 2-3: cuanto MOVIO la simulacion el grafo -------------------------
# Ya no es el grafo de la seleccion sino |grafo_base - grafo_simulado|. Los dos comparten
# los pesos fijos del experto y solo difieren por las compuertas, asi que puestos uno al
# lado del otro se ven iguales y el efecto de la intervencion -- que es lo que el panel
# viene a mostrar -- se pierde. La diferencia aisla exactamente lo que cambio.
# El peso viaja en un marcador en el PUNTO MEDIO de cada arista y no en el ancho de la
# linea: una sola traza de lineas no puede variar su ancho por segmento, y partirla en una
# traza por arista serian decenas que hay que restilar una por una.
IDX['grafo_aristas'] = _agregar(go.Scattergl(
    x=[], y=[], mode='lines', showlegend=False,
    line=dict(width=1.0, color='rgba(120,110,110,0.45)'), hoverinfo='skip',
), 6, 2)
IDX['grafo_pesos'] = _agregar(go.Scattergl(
    x=[], y=[], mode='markers', showlegend=False,
    marker=dict(size=[], color=[], colorscale='Reds', cmin=0.0, showscale=False,
                line=dict(width=0.4, color='#5b4a48')),
    hovertext=[], hoverinfo='text',
), 6, 2)
# `mode='markers'` a secas: el NOMBRE del nodo ya no viaja en la traza. Un `Scatter` no
# puede girar su texto -- comprobado contra plotly 6.8.0, solo `Bar` y las anotaciones
# llevan `textangle` --, y con los rotulos horizontales los nombres de nodos vecinos se
# montaban unos sobre otros alrededor del anillo. Van como anotaciones, mas abajo.
IDX['grafo_nodos'] = [
    _agregar(go.Scatter(
        x=[], y=[], mode='markers', name=_modalidad, legendgroup='grafo',
        marker=dict(size=7, color=COLORES_MODALIDAD[_modalidad],
                    line=dict(width=0.5, color='#1f2937')),
        hovertext=[], hoverinfo='text',
    ), 6, 2) for _modalidad in MODALIDADES_MIL
]
# Los ejes del grafo se PREGUNTAN a su traza en vez de escribirse a mano: el numero
# depende de la posicion del subplot en la grilla y del eje secundario de la fila 3, y
# adivinarlo deja el aviso flotando sobre otro panel.
_EJE_X_GRAFO = _fig.data[IDX['grafo_aristas']].xaxis or 'x'
_EJE_Y_GRAFO = _fig.data[IDX['grafo_aristas']].yaxis or 'y'
# Sin ejes: una disposicion circular no mide nada en x ni en y. El rango se fija a mano y
# con holgura -- sin ella los rotulos de los nodos del borde salen cortados.
# El rango en x va JUSTO -- el rotulo mas largo llega a 1,80 unidades del centro -- y no
# holgado. Con `scaleanchor`, el eje que manda es el que tiene menos pixeles por unidad:
# con un rango ancho mandaba la x, y entonces el circulo se encogia con la ventana hasta
# que los rotulos se tocaban. Medido: con 6,4 unidades de rango, por debajo de 1.400 px de
# pantalla el radio caia de los 94,5 px que hacen falta para 66 nombres; con 3,9 unidades
# manda siempre la y y el radio se queda en 109,3 px a cualquier ancho.
_fig.update_xaxes(visible=False, showticklabels=False, range=[-1.95, 1.95], row=6, col=2)
# `scaleanchor` iguala los pixeles por unidad de los dos ejes. Sin el, el panel es mucho
# mas ancho que alto y la disposicion circular se dibujaba como una ELIPSE aplastada 2,93
# veces -- medido --, que ademas hace que el giro radial de cada rotulo deje de coincidir
# con la direccion que se ve. El rango va holgado a proposito: los rotulos de arriba y de
# abajo salen casi verticales y necesitan su largo completo fuera del circulo.
_fig.update_yaxes(visible=False, showticklabels=False, range=[-1.75, 1.75],
                  scaleanchor=_EJE_X_GRAFO, scaleratio=1.0, row=6, col=2)
_fig.add_annotation(text='', xref=f'{_EJE_X_GRAFO} domain', yref=f'{_EJE_Y_GRAFO} domain',
                    x=0.5, y=0.5, showarrow=False, align='center',
                    font=dict(size=11, color='#7a5c58'))
IDX_ANOTACION_GRAFO = len(_fig.layout.annotations) - 1

# Un rotulo por nodo, como ANOTACION y no como texto de la traza, para poder girarlo.
# La reserva se crea entera al armar la figura y despues solo se le cambia el contenido:
# agregar y quitar anotaciones en cada repintado correria los indices de todas las demas
# -- los avisos del grafo, de los costos y del mapa simulado --, que se guardan por
# posicion. Sobran las que no se usen; se dejan con texto vacio.
MAX_NODOS_GRAFO = len(FEATURES_MIL)
IDX_ANOTACIONES_NODOS = []
for _ in range(MAX_NODOS_GRAFO):
    _fig.add_annotation(text='', xref=_EJE_X_GRAFO, yref=_EJE_Y_GRAFO, x=0, y=0,
                        showarrow=False, font=dict(size=7, color='#334155'),
                        xanchor='left', yanchor='middle', textangle=0, visible=False)
    IDX_ANOTACIONES_NODOS.append(len(_fig.layout.annotations) - 1)

# --- Fila 4: UITI acumulado MEDIDO contra el simulado, vano por vano -------------------
# Un grupo por vano y un ultimo grupo con el circuito entero. Reemplaza a los violines:
# con diez vanos, dos violines resumian en una densidad lo que aqui se lee vano por vano,
# que es el grano en que se decide una obra.
# Las dos barras son cantidades de NATURALEZA distinta -- una medicion contra una
# prediccion -- y eso no se puede esconder: medido sobre 599 bolsas, el modelo correlaciona
# 0,950 con el UITI observado pero su nivel corre +34%. Por eso la barra simulada lleva
# barra de error con el desfase del modelo en la base de ESE vano, y el titulo publica la
# reduccion con su +-: sin eso, el sesgo del modelo se leeria como ahorro.
IDX['barra_observada'] = _agregar(go.Bar(
    x=[], y=[], name='UITI medido', marker=dict(color=COLOR_BARRA_MEDIDA,
                                                line=dict(width=0.4, color='#5b4a48')),
    hovertext=[], hoverinfo='text', legendgroup='uiti',
), 4, 1)
IDX['barra_simulada'] = _agregar(go.Bar(
    x=[], y=[], name='UITI simulado', marker=dict(color=COLOR_BARRA_SIMULADA,
                                                  line=dict(width=0.4, color='#5b4a48')),
    # `visible=True` con `array` vacio no dibuja nada; se llena al simular.
    error_y=dict(type='data', array=[], visible=True, color='#5b4a48', thickness=1.2,
                 width=4),
    hovertext=[], hoverinfo='text', legendgroup='uiti',
), 4, 1)
_fig.update_yaxes(title_text='UITI acumulado', rangemode='tozero', row=4, col=1)
_fig.update_xaxes(title_text='Vano', tickfont=dict(size=9), row=4, col=1)
_EJE_X_BARRAS = _fig.data[IDX['barra_observada']].xaxis or 'x'
_EJE_Y_BARRAS = _fig.data[IDX['barra_observada']].yaxis or 'y'
_fig.add_annotation(text='', xref=f'{_EJE_X_BARRAS} domain', yref=f'{_EJE_Y_BARRAS} domain',
                    x=0.5, y=0.5, showarrow=False, align='center',
                    font=dict(size=11, color='#7a5c58'))
IDX_ANOTACION_BARRAS = len(_fig.layout.annotations) - 1
# El titulo del panel se reescribe en cada simulacion para publicar la reduccion, asi que
# hace falta su indice. Se BUSCA por el texto con que nacio en vez de contar posiciones:
# los titulos de subplot son las primeras anotaciones y su orden depende de la rejilla,
# que ya cambio una vez.
IDX_TITULO_BARRAS = next(
    i for i, _a in enumerate(_fig.layout.annotations)
    if (_a.text or '').startswith('UITI acumulado: medido'))

# --- Fila 5: el costo de la intervencion ----------------------------------------------
# UNA sola traza con una barra por vano mas la del TOTAL, y no dos trazas: las dos
# mediciones estan en la misma escala de pesos y separarlas obligaria a una leyenda para
# decir algo que la posicion ya dice. El color va como ARRAY -- azul de vano, gris para
# el total -- que es lo que permite distinguirlas dentro de una misma traza.
# El TOTAL comparte eje con los vanos a proposito, aunque sea siempre el mas alto: es su
# suma, y ponerlo en un eje propio dejaria de mostrar cuanto pesa cada vano dentro de el.
# El desglose por actividad viaja en el hover: el total contesta cuanto y el detalle
# contesta por que, sin obligar a reabrir el panel para averiguarlo.
IDX['costos'] = _agregar(go.Bar(
    x=[], y=[], name='Costo', showlegend=False, width=0.5,
    marker=dict(color=[], line=dict(width=0.4, color='rgba(60,10,10,0.6)')),
    text=[], textposition='outside', textfont=dict(size=10),
    hovertext=[], hoverinfo='text',
), 5, 1)
_fig.update_yaxes(title_text='COP', rangemode='tozero', tickformat=',.0f', row=5, col=1)
_fig.update_xaxes(title_text='Vano', row=5, col=1)
# El aviso de la fila 5 va anclado a SU eje, igual que el del grafo: antes de simular no
# hay costo, y un panel vacio sin explicacion se lee como que la seleccion no cuesta nada.
_EJE_X_COSTOS = _fig.data[IDX['costos']].xaxis or 'x'
_EJE_Y_COSTOS = _fig.data[IDX['costos']].yaxis or 'y'
_fig.add_annotation(text='', xref=f'{_EJE_X_COSTOS} domain', yref=f'{_EJE_Y_COSTOS} domain',
                    x=0.5, y=0.5, showarrow=False, align='center',
                    font=dict(size=11, color='#7a5c58'))
IDX_ANOTACION_COSTOS = len(_fig.layout.annotations) - 1

# El aviso del mapa simulado va en coordenadas de PAPEL y no de eje: un subplot de tipo
# `map` no tiene ejes cartesianos a los que anclar una anotacion. El centro sale del
# dominio que `make_subplots` ya calculo, asi que cambiar `row_heights` no lo desalinea.
_dominio_simulado = _fig.layout.map2.domain
_fig.add_annotation(
    text='', xref='paper', yref='paper',
    x=(_dominio_simulado.x[0] + _dominio_simulado.x[1]) / 2.0,
    y=(_dominio_simulado.y[0] + _dominio_simulado.y[1]) / 2.0,
    showarrow=False, align='center', font=dict(size=13, color='#5b4a48'),
    bgcolor='rgba(255,255,255,0.88)', bordercolor='#e4c4c0', borderwidth=1, borderpad=8,
)
IDX_ANOTACION_SIMULADO = len(_fig.layout.annotations) - 1

_fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10,
             layers=[CAPA_CAJA_SELECCION]),
    map2=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10,
              layers=CAPAS_CAJA_SIMULADA),
    title=dict(text='Simulador Criticidad'),
    # Margenes explicitos. Los de Plotly por defecto (l=80, r=80, t=100, b=80) se llevaban
    # 160 px de ancho -- medido, el 8,5% de una pantalla de 1.920 -- y a la derecha no hay
    # nada que rotular. El izquierdo NO puede bajar a cero: es donde viven el titulo y las
    # marcas del eje y de los paneles de la primera columna.
    margin=dict(l=52, r=14, t=78, b=44),
    barmode='group', bargap=0.25, bargroupgap=0.05,
    # SIN `width`: con un ancho fijo Plotly ignora el contenedor. Pero dejarlo en None NO
    # basta por si solo -- `autosize` mide el contenedor UNA vez, al montar, y el widget
    # monta antes de que el CSS de la celda lo estire. Lo que cierra el circulo es
    # `responsive` en el config del widget, mas abajo.
    # 2.400 y no 1.850: la fila del grafo es nueva y las barras de la 4 pasaron a ocupar
    # el ancho completo. Medido a 1.900 px de ancho, con este alto el panel del grafo
    # queda en 383 px y el circulo en 255 de diametro, que es lo que deja sitio a los 66
    # nombres alrededor.
    height=2400, autosize=True, template='plotly_white',
    # La leyenda va HORIZONTAL y justo debajo de los mapas. Vertical y a la derecha se
    # llevaba 196 px medidos de ancho para decir siete nombres. `y` sale del dominio del
    # mapa y no de un numero escrito a mano: cambiar `row_heights` mueve los mapas.
    legend=dict(orientation='h', x=0.5, xanchor='center',
                y=_fig.layout.map.domain.y[0] - 0.004, yanchor='top',
                font=dict(size=10), tracegroupgap=22),
)

# El alto en pixeles del panel del top. Va DESPUES de `update_layout` porque el alto de
# la figura se fija alli: leido antes, `_fig.layout.height` todavia es None.
# Es su dominio por el alto de la figura. Es lo que
# permite decidir, barra por barra, si el nombre de la variable cabe escrito adentro --
# el rotulo va girado -90, asi que lo que lo limita es el LARGO de la barra y no su ancho.
# El eje se PREGUNTA a la traza en vez de escribirse a mano: su numero depende del eje
# secundario de la fila 3, y adivinarlo mediria el panel equivocado.
_EJE_Y_TOP = _fig.data[IDX['top_vano'][0]].yaxis or 'y'
_DOMINIO_TOP = _fig.layout[_EJE_Y_TOP.replace('y', 'yaxis', 1)].domain
ALTO_PANEL_TOP_PX = float(_fig.layout.height) * float(_DOMINIO_TOP[1] - _DOMINIO_TOP[0])

# Los indices se verifican al generar: si alguien reordena las trazas, esto falla AQUI y
# no se descubre en silencio al dibujar.
assert len(_fig.data) == 4 + 1 + 1 + 4 + 1 + 4 + 1 + 4 + 2 * MAX_VANOS_ANALISIS \
    + TOP_VARIABLES_POR_VANO + 2 + len(MODALIDADES_MIL) + 2 + 1, len(_fig.data)
assert _fig.layout.width is None and _fig.layout.height, (
    'la figura no puede llevar ancho fijo: con uno, Plotly ignora el contenedor. El alto '
    'si es propio. Cuidado: sin ancho fijo NO alcanza -- ver `responsive` abajo.')
# El marcado con color de clase va DESPUES del halo blanco, o el halo lo taparia.
assert min(IDX['marcados_clases']) > IDX['marcados']
assert [_fig.data[i].line.color for i in IDX['marcados_clases']] == COLORES_GRUPOS
# Los equipos son PUNTOS y van despues de todas las lineas: si alguien los adelanta,
# quedan tapados por los tramos.
assert all(_fig.data[i].mode == 'markers'
           for i in (IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']))
assert min(IDX['trafos'], IDX['pred_trafos']) > max(IDX['pred_clases'])
# UN solo negro para la ausencia en los DOS mapas.
assert (_fig.data[IDX['sin_dato']].line.color
        == _fig.data[IDX['pred_sin_dato']].line.color == COLOR_SIN_EVENTO)
assert all(isinstance(_fig.data[i].marker.size, (list, tuple))
           for i in IDX['serie_uiti'] + IDX['serie_eventos']), (
    'marker.size debe ser un array: el punto de la ventana vigente va al triple')
assert all(_fig.data[i].type == 'bar' for i in IDX['top_vano'])
assert _fig.layout.barmode == 'group', 'las barras del top 5 se agrupan POR VANO'
assert all(_fig.data[i].type == 'bar'
           for i in (IDX['barra_observada'], IDX['barra_simulada']))
# La barra de error existe desde el armado: llenarla en el repintado es escribir un
# array, no crear la propiedad, que es lo que mantiene el repintado en una sola pasada.
assert _fig.data[IDX['barra_simulada']].error_y.visible is True
# El color de la barra de costo es un ARRAY: una sola traza lleva los vanos y el total,
# y sin array habria que partirla en dos y explicar la particion con una leyenda.
assert _fig.data[IDX['costos']].type == 'bar'
assert isinstance(_fig.data[IDX['costos']].marker.color, (list, tuple))
assert [_fig.data[i].name for i in IDX['grafo_nodos']] == MODALIDADES_MIL
# La caja de seleccion es una CAPA del mapa y no una traza, y va DEBAJO de las trazas: si
# alguien la sube por encima vuelve a comerse el clic que alterna la seleccion. El mapa
# base lleva UNA -- una sola pregunta, "cual elegi" -- y el simulado TRES, una por
# desenlace, porque una capa pinta con un solo color.
assert len(_fig.layout.map.layers) == 1
assert len(_fig.layout.map2.layers) == len(CAMBIOS)
assert all(_capa.below == 'traces'
           for _capa in (*_fig.layout.map.layers, *_fig.layout.map2.layers))
assert ALTO_PANEL_TOP_PX > 0, 'sin alto de panel no se puede decidir si el rotulo cabe'

# `figura_de_mapas` y no `go.FigureWidget` a secas: con plotly 6.8.0, arrastrar o hacer
# zoom sobre un mapa MapLibre devuelve `map._derived` -- las esquinas que MapLibre acaba de
# calcular -- junto a `map.center` y `map.zoom`, y `plotly_relayout` lo rechaza con
# `Invalid property path 'map._derived' for layout`. El error salta en CADA arrastre y sale
# en la salida de la celda que muestra el widget, por encima del tablero, asi que se lee
# como si una celda anterior se hubiera roto.
fig = figura_de_mapas(_fig)
# Un `FigureWidget` NO es fluido por si solo: `width=None` mas `autosize` mas el CSS de la
# celda estiran el DIV, pero plotly sigue DIBUJANDO al ancho que midio al montar -- medido,
# 858 px dentro de un contenedor de 1.935. Su bundle trae un `ResizeObserver` que arregla
# exactamente esto, apagado detras de `config.responsive`. `_config` es un trait
# SINCRONIZADO: lo que se ponga aqui viaja al `newPlot` del navegador y lo enciende.
fig._config = {**(fig._config or {}), 'responsive': True}
assert fig._config.get('responsive') is True, (
    'sin `responsive` el FigureWidget dibuja al ancho de reserva y no al de la celda')
print(f'FigureWidget con {len(fig.data)} trazas en 7 paneles')


In [ ]:
# --- Fila 1: mapa historico con paridad 01.4 + seleccion por casilla o por clic ------
# Tres cosas que el mapa de 01.4 hace y este no hacia: se ENCUADRA sobre el circuito
# elegido (sin eso el circuito queda como un garabato diminuto en un mapa centrado en
# Manizales), dibuja transformadores e interruptores, y da hover por tramo. La cuarta es
# la seleccion: en 01.4 un vano se marca con su casilla O tocandolo en el mapa, y las dos
# vias son EL MISMO estado -- el clic alterna la casilla y deja que todo se rehaga desde
# ahi. Un registro paralelo es como la lista, el mapa y el ranking empiezan a contar
# cosas distintas.


def _seleccion_actual():
    return circuito_widget.value, ventana_widget.value, set(vano_widget.value)


def _plantilla_hover(campo, nombre_clase, ventana, *, marcado=False, extra=''):
    """El tooltip de una traza, como `hovertemplate` y no como texto por punto.

    Lo que varia DENTRO de una traza son solo el fid, el UITI y los eventos, y esos
    viajan crudos en `customdata`. La clase y la ventana son constantes de la traza --
    hay una traza por clase-- asi que van escritas en la plantilla y no se repiten en
    cada punto. Esa diferencia es la que permite densificar: medido sobre el peor
    circuito, repetir la etiqueta formateada cuesta 2,40 MB por capa y esto cuesta 0,66.

    `extra` agrega renglones que SI varian punto a punto y por eso citan `customdata`:
    es como el mapa simulado dice el grupo base de cada vano, que dentro de una traza --
    que es una clase SIMULADA -- cambia de vano a vano.
    """
    return (f'<b>Vano %{{customdata[0]}}</b><br>{ventana["etiqueta"]}: {ventana["periodo"]}'
            f'<br>{campo}: {nombre_clase}{extra}'
            '<br>UITI acumulado: %{customdata[1]}<br>Eventos: %{customdata[2]}'
            + ('<br>(marcado)' if marcado else '') + '<extra></extra>')


def _capas_de_la_seleccion(clases_por_fid, *, campo, nombres_clase,
                           extra_por_fid=None, plantilla_extra=''):
    """Las capas de UN mapa, con el customdata que alimentan el tooltip y el clic.

    `campo` nombra en el tooltip a que pertenece la clase -- "Criticidad original" en
    la fila 1, "Criticidad simulada" en la fila 2.

    `extra_por_fid` agrega columnas al `customdata` de cada punto, para lo que varia
    dentro de una traza y no cabe en la plantilla. `plantilla_extra` es el renglon del
    tooltip que las lee.

    `paso_densificado` interpola vertices cada ~25 m. El hover de una traza de lineas en
    Scattermap se resuelve contra los VERTICES, y los tramos de MVLINSEC traen
    exactamente dos: sin esto, el centro de un vano no muestra etiqueta, y como Plotly
    solo convierte un clic en evento donde hay hover, tampoco se puede marcar tocandolo
    ahi. Es la misma correccion que ya tenia el mapa de 01.
    """
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    ventana = VENTANAS[ventana_i]
    datos = DATOS_VENTANA[ventana_i]

    # Datos CRUDOS por vano; el formato lo pone la plantilla de cada traza.
    datos_por_fid = {fid: datos.get(fid, (0.0, 0)) for fid in geo['fids']}
    if extra_por_fid is not None:
        # La columna extra viaja para TODOS los fids y no solo para los que la tienen:
        # dentro de una traza `customdata` tiene que medir siempre lo mismo, o
        # `%{customdata[3]}` cae en el hueco del vano de al lado.
        datos_por_fid = {fid: (*crudos, *extra_por_fid.get(fid, ('sin dato',)))
                         for fid, crudos in datos_por_fid.items()}
    capas = capas_mapa_historico(
        geo, clases_por_fid, marcados=marcados, datos_por_fid=datos_por_fid,
        marca_extremos=MARCA_VANO, paso_densificado=PASO_VERTICE)
    # Sin celda en la ventana no hay clase, y eso NO es el grupo mas bajo: es la
    # ausencia del dato. Mismo criterio que el tooltip de 01.4.
    sin_dato = 'sin dato'
    capas['plantillas'] = {
        'clases': [_plantilla_hover(campo, nombres_clase[c], ventana,
                                    extra=plantilla_extra) for c in range(4)],
        'sin_dato': _plantilla_hover(campo, sin_dato, ventana, extra=plantilla_extra),
        'marcados_por_clase': [_plantilla_hover(campo, nombres_clase[c], ventana,
                                                marcado=True, extra=plantilla_extra)
                               for c in range(4)],
        'marcados_sin_dato': _plantilla_hover(campo, sin_dato, ventana, marcado=True,
                                              extra=plantilla_extra),
    }
    return capas


def _volcar_capa(traza, capa, plantilla=None):
    """Las tres columnas van juntas SIEMPRE: si `customdata` se desfasa de lat/lon,
    Plotly desalinea el resto de la traza y el clic devuelve el vano equivocado.

    `plantilla` es el `hovertemplate` de la traza. Sin ella la traza no muestra tooltip
    -- es lo que corresponde al halo blanco, que es decoracion y esta debajo de la linea
    de color, que si lo muestra."""
    traza.lat = capa['lat']
    traza.lon = capa['lon']
    traza.customdata = capa['customdata']
    if plantilla is None:
        traza.hoverinfo = 'skip'
    else:
        traza.hovertemplate = plantilla


def _tamanos_ventana_activa(ventana_i):
    """El arreglo de tamanos de marcador de las dos series, con la ventana vigente al
    triple. Es el mismo recurso de la serie del cuaderno 01: `marker.size` es un ARRAY,
    asi que agrandar un punto no obliga a partir la serie en una segunda traza, y mover
    el deslizador solo reescribe once numeros."""
    return (
        [SERIE_TAM_UITI * (FACTOR_PUNTO_ACTIVO if v['i'] == ventana_i else 1)
         for v in VENTANAS],
        [SERIE_TAM_EVENTOS * (FACTOR_PUNTO_ACTIVO if v['i'] == ventana_i else 1)
         for v in VENTANAS],
    )


def _redibujar_mapa_historico(*_ignorado):
    circuito, ventana_i, _marcados = _seleccion_actual()
    capas = _capas_de_la_seleccion(clases_para(circuito, ventana_i),
                                   campo='Criticidad original', nombres_clase=NOMBRES_GRUPOS)
    # El orden de las series sale de la GEOMETRIA y no del orden en que se fueron
    # marcando: asi marcar y desmarcar no baraja los colores bajo la mano.
    marcados_ordenados = [f for f in GEO_POR_CIRCUITO.get(circuito, {}).get('fids', [])
                          if f in _marcados]
    marcados_ordenados = list(dict.fromkeys(marcados_ordenados))[:MAX_VANOS_ANALISIS]
    # La serie describe SOLO los vanos elegidos: sin ninguno marcado queda vacia. Es el
    # mismo criterio que los violines de 01.4 -- una serie sobre el circuito entero y una
    # sobre tres vanos se dibujan igual y no miden lo mismo, asi que caer al circuito
    # cambiaria el sujeto del panel en silencio.
    series = series_temporal_vanos(TABLA, circuito=circuito, fids=marcados_ordenados,
                                   n_ventanas=len(VENTANAS))
    # El grupo de riesgo de cada punto, de UNA sola llamada a la geometria de 01.4 para
    # los hasta 55 puntos dibujados. El repintado corre en cada clic del mapa.
    clases_serie = clases_de_series(series)
    _pl = capas['plantillas']
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['clases'][_clase]], capas['clases'][_clase],
                         _pl['clases'][_clase])
            _volcar_capa(fig.data[IDX['marcados_clases'][_clase]],
                         capas['marcados_por_clase'][_clase],
                         _pl['marcados_por_clase'][_clase])
        _volcar_capa(fig.data[IDX['sin_dato']], capas['sin_dato'], _pl['sin_dato'])
        # El halo blanco va SIN tooltip: esta debajo de la linea de color, que ya lo
        # muestra, y dos etiquetas en el mismo punto solo se estorban.
        _volcar_capa(fig.data[IDX['marcados']], capas['marcados'])
        _volcar_capa(fig.data[IDX['marcados_sin_dato']], capas['marcados_sin_dato'],
                     _pl['marcados_sin_dato'])
        # La caja amarilla de lo seleccionado. Sale de la GEOMETRIA y no de las celdas de
        # la ventana: por eso el resaltado sigue puesto al mover el deslizador, incluso
        # sobre un vano que en esa ventana no tiene ni un evento. Se apaga solo al
        # desmarcar el vano -- por su casilla o volviendo a tocarlo en el mapa.
        fig.layout.map.layers[0].source = cajas_seleccion(
            GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []}),
            marcados=_marcados, lado_minimo=LADO_MINIMO_CAJA, margen=MARGEN_CAJA)
        # Fila 3 col 1-2: la serie de tiempo de cada vano elegido, UITI contra el eje
        # izquierdo y eventos contra el derecho. Una ventana sin celda va como `None` y
        # NO como cero: un cero se leeria como "no hubo UITI", y lo que paso es que no
        # hubo medicion. `connectgaps=False` corta la linea ahi.
        _tam_uiti, _tam_eventos = _tamanos_ventana_activa(ventana_i)
        for _cupo in range(MAX_VANOS_ANALISIS):
            _serie = series[_cupo] if _cupo < len(series) else None
            _t_uiti = fig.data[IDX['serie_uiti'][_cupo]]
            _t_eventos = fig.data[IDX['serie_eventos'][_cupo]]
            _sujeto = f'Vano {_serie["fid"]}' if _serie else ''
            _clases = clases_serie[_cupo] if _cupo < len(clases_serie) else []
            # El relleno del punto lleva el grupo de riesgo de ESE vano en ESA ventana;
            # el gris es la ventana sin celda, que no tiene grupo.
            _colores = [COLORES_GRUPOS[c] if c is not None else COLOR_SIN_GRUPO
                        for c in _clases]
            _etiquetas = ([
                f'<b>{_sujeto}</b><br>{VENTANAS[i]["etiqueta"]}: '
                f'{VENTANAS[i]["periodo"]}<br>UITI: {u}<br>Eventos: {e}'
                f'<br>Grupo: {NOMBRES_GRUPOS[c] if c is not None else "sin eventos"}'
                for i, u, e, c in zip(_serie['x'], _serie['uiti'], _serie['eventos'],
                                      _clases)
            ] if _serie else [])
            _t_uiti.x = _serie['x'] if _serie else []
            _t_uiti.y = _serie['uiti'] if _serie else []
            _t_uiti.hovertext = _etiquetas
            _t_uiti.marker.size = _tam_uiti if _serie else []
            _t_uiti.marker.color = _colores
            _t_uiti.name = _sujeto
            _t_eventos.x = _serie['x'] if _serie else []
            _t_eventos.y = _serie['eventos'] if _serie else []
            _t_eventos.hovertext = _etiquetas
            _t_eventos.marker.size = _tam_eventos if _serie else []
            _t_eventos.marker.color = _colores


def _alto_del_mapa_px():
    """El alto en pixeles del subplot de mapa, de su dominio por el alto de la figura.

    Sin esto el zoom salia del span en GRADOS, sin mirar el viewport, y un circuito alto
    quedaba recortado arriba y abajo. El ancho no se pasa: con `autosize` lo decide la
    celda, y encuadrar solo por el alto evita el recorte, que era el defecto."""
    _dom_y = fig.layout.map.domain.y
    return float(fig.layout.height) * float(_dom_y[1] - _dom_y[0])


def _vista_del_circuito(circuito):
    """El encuadre del circuito completo, o None si no tiene geometria. Es la vista de
    referencia de los dos mapas y la que el simulado recupera cuando no hay nada marcado
    sobre lo que acercarse."""
    return centro_y_zoom(GEO_POR_CIRCUITO.get(circuito, {}).get('bounds'),
                         alto_px=_alto_del_mapa_px())


def _aplicar_vista(nombre_mapa, vista):
    if vista is not None:
        getattr(fig.layout, nombre_mapa).center = vista['center']
        getattr(fig.layout, nombre_mapa).zoom = vista['zoom']


def _centrar_mapa(nombre_mapa):
    """Encuadra ESE mapa sobre los vanos marcados, o sobre el circuito si no hay ninguno.

    Existe porque las dos vistas se van de sitio por caminos legitimos: el usuario hace
    zoom para mirar un tramo, o el mapa simulado se acerca solo a los vanos que puntuo y
    deja de compartir geografia con el de la izquierda. Volver no deberia obligar a
    recargar la celda ni a cambiar de circuito y regresar.

    La vista se calcula EN EL CLIC y no se guarda al dibujar: entre un dibujo y el clic
    pueden haber cambiado los vanos marcados, y un encuadre precalculado llevaria a donde
    estaba la seleccion antes.
    """
    circuito, _ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    _aplicar_vista(nombre_mapa,
                   centro_y_zoom(bounds_de_fids(geo, marcados),
                                 alto_px=_alto_del_mapa_px())
                   or _vista_del_circuito(circuito))


def _pintar_circuito(*_ignorado):
    """Lo que depende del CIRCUITO y no de la ventana: equipos y encuadre. Se separa del
    repintado por ventana porque mover la ventana no tiene por que recentrar el mapa --
    en 01.4 el encuadre tambien se hace una sola vez por circuito (`ULTIMO_CENTRADO`)."""
    circuito = circuito_widget.value
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    vista = _vista_del_circuito(circuito)
    with fig.batch_update():
        # Solo la fila 1: los equipos de la fila 2 los pinta el mapa simulado, que antes
        # de la primera simulacion no muestra NADA.
        for _i_tr, _i_sw in ((IDX['trafos'], IDX['switches']),):
            fig.data[_i_tr].lat, fig.data[_i_tr].lon = tr['lat'], tr['lon']
            fig.data[_i_tr].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
            fig.data[_i_sw].lat, fig.data[_i_sw].lon = sw['lat'], sw['lon']
            fig.data[_i_sw].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        # Los dos arrancan sobre el circuito completo. Despues de simular, el de la
        # derecha se acerca a los vanos marcados (ver `_redibujar_mapa_predicho`); el de
        # la izquierda conserva SIEMPRE esta vista, que es la referencia contra la que se
        # mira el acercamiento.
        for _mapa in ('map', 'map2'):
            _aplicar_vista(_mapa, vista)


_DESC = {'description_width': 'initial'}  # sin esto ipywidgets trunca los rotulos
circuito_widget = widgets.Dropdown(options=CIRCUITOS, description='Circuito',
                                   style=_DESC)
# El rotulo lleva las fechas del intervalo y no solo "V1": una ventana sin sus fechas
# obliga a ir a buscar a que periodo corresponde cada vez que se mueve el deslizador.
def _opciones_de_ventana(circuito):
    """Las ventanas que ESE circuito puede mostrar, como pares (rotulo, indice).

    El rotulo lleva las fechas del intervalo y no solo "V1": una ventana sin sus fechas
    obliga a ir a buscar a que periodo corresponde cada vez que se mueve el deslizador.

    Nunca vacia: un `SelectionSlider` sin opciones lanza al construirse, y un circuito
    sin ninguna celda dejaria el panel sin arrancar. En ese caso el deslizador ofrece la
    primera ventana, y el mapa dira por su cuenta que no hay nada que pintar.
    """
    indices = VENTANAS_POR_CIRCUITO.get(circuito) or [0]
    return [(f'{VENTANAS[i]["etiqueta"]}: {VENTANAS[i]["periodo"]}', i) for i in indices]


# Arranca en la ULTIMA ventana del circuito, no en la primera: es el periodo mas reciente
# con eventos, y es la pregunta con la que se abre el tablero -- como esta esto AHORA. La
# primera ventana es historia, y quien la quiera la alcanza moviendo el deslizador.
# `_opciones_de_ventana` nunca devuelve vacio, asi que `[-1]` siempre existe.
_OPCIONES_INICIALES = _opciones_de_ventana(circuito_widget.value)
ventana_widget = widgets.SelectionSlider(
    options=_OPCIONES_INICIALES, value=_OPCIONES_INICIALES[-1][1],
    description='Ventana', continuous_update=False, style=_DESC,
    layout=widgets.Layout(width='560px'),
)
# Casillas, no SelectMultiple: es la unica forma de que un clic en el mapa alterne el
# MISMO control que el usuario ve, y de que marcar un vano no borre los ya marcados.
# TOPE de MAX_VANOS_ANALISIS: cada vano seleccionado recibe su propia COLUMNA de controles
# mas abajo, y una rejilla de 26 variables por 20 vanos no se lee ni se llena. Al llegar al
# tope las casillas sin marcar se deshabilitan solas, y el clic en el mapa respeta el mismo
# limite -- si no, el mapa seria una puerta trasera para marcar el sexto.
vano_widget = construir_selector_vanos(VANOS_POR_CIRCUITO.get(circuito_widget.value, []),
                                       maximo=MAX_VANOS_ANALISIS)


# Ya no hay "Marcar todos": con el tope en MAX_VANOS_ANALISIS marcaria los primeros cinco
# de la lista, que no es una eleccion que nadie quiera tomar. Queda "Desmarcar", que es el
# camino de vuelta al grano de circuito completo (seleccion vacia).
boton_desmarcar = widgets.Button(description='Desmarcar', button_style='')
boton_desmarcar.on_click(lambda _b: vano_widget.desmarcar_todos())


def _on_circuito_change(_change):
    circuito = circuito_widget.value
    vano_widget.poblar(VANOS_POR_CIRCUITO.get(circuito, []))
    # El deslizador se repuebla ANTES de repintar: si se repintara primero, el mapa se
    # dibujaria con la ventana del circuito anterior y se volveria a dibujar en cuanto
    # `options` moviera el valor. Se conserva la ventana vigente cuando el circuito nuevo
    # tambien la tiene -- moverse de circuito no deberia cambiar el mes que se esta
    # mirando -- y si no la tiene, cae en la primera que si.
    # La ventana vigente se lee ANTES de tocar `options`: asignar `options` reajusta
    # `value` a la primera opcion de inmediato, asi que leerlo despues siempre devuelve
    # esa primera y la ventana se perdia en cada cambio de circuito. Medido: pasar de un
    # circuito a otro que SI tiene la ventana 10 la dejaba igual en la 0.
    _vigente = ventana_widget.value
    _opciones = _opciones_de_ventana(circuito)
    _disponibles = [i for _rotulo, i in _opciones]
    ventana_widget.options = _opciones
    # Si el circuito nuevo no tiene la ventana vigente, cae en la ULTIMA que si tiene, por
    # el mismo motivo por el que el deslizador arranca ahi: lo reciente antes que lo viejo.
    ventana_widget.value = _vigente if _vigente in _disponibles else _disponibles[-1]
    _pintar_circuito()
    _redibujar_mapa_historico()


def _al_hacer_clic(traza, puntos, _estado):
    """Un clic sobre un tramo alterna su vano. El fid sale de `customdata` y no del
    indice del punto: los tramos viajan concatenados con un `None` de separador, asi que
    ese indice cambia con la ventana."""
    fid = fid_de_punto(traza.customdata, getattr(puntos, 'point_inds', ()) or ())
    if fid is not None:
        vano_widget.alternar(fid)


# SOLO el mapa base. La fila 2 es la SALIDA del modelo, no un control: marcar un vano
# desde ahi mezcla "lo que yo elegi" con "lo que el modelo predijo" sobre la misma
# superficie, que es justo la confusion que separa a las dos filas (D2).
# Nota sobre el alcance del clic: plotly solo convierte un clic en evento si en ese punto
# hay hover, y en un `scattermap` de lineas el hover se calcula contra los VERTICES del
# tramo (`scattermap/hover.js`: distancia por punto, radio minimo 3 px, tope
# `layout.hoverdistance`). Antes eso obligaba a tocar el tramo cerca de uno de sus dos
# extremos; ahora `paso_densificado` pone un vertice cada ~25 m, asi que el clic engancha
# en cualquier punto del vano. `hoverdistance` sigue en 30 px, por encima de los 20 por
# defecto, para que el blanco sea generoso sin llegar a marcar un vano lejano.
#
# Se cablean TAMBIEN las capas de vano marcado: quedan dibujadas encima de las de clase,
# asi que son las que recibe el cursor sobre un vano ya marcado. Sin ellas, marcar
# funcionaba y desmarcar tocando el mapa no.
for _i_traza in (IDX['clases'] + IDX['marcados_clases']
                 + [IDX['sin_dato'], IDX['marcados'], IDX['marcados_sin_dato']]):
    fig.data[_i_traza].on_click(_al_hacer_clic)
fig.layout.hoverdistance = 30

# Tier 0 del presupuesto de interactividad (design section A): elegir circuito, mover la
# ventana o marcar un vano no llama al modelo -- sin debounce ni epoch guard, que
# pertenecen al tier 1/2 (fila 2, ranking, boton "Simular"), fuera del alcance de este PR.
circuito_widget.observe(_on_circuito_change, names='value')
ventana_widget.observe(_redibujar_mapa_historico, names='value')
vano_widget.observe(_redibujar_mapa_historico, names='value')

_pintar_circuito()               # equipos y encuadre del circuito inicial
_redibujar_mapa_historico()      # primer dibujo, con la seleccion inicial

In [ ]:
# --- Fila 3, columnas 3-4: que baja el UITI de CADA vano ------------------------------
# El panel mostraba UN ranking, el de la seleccion entera. Con hasta cinco vanos bajo
# estudio eso contesta la pregunta equivocada: dice que variable mueve AL GRUPO, cuando la
# decision de mantenimiento necesita saber cual mueve a ESTE vano, el de la orden de
# trabajo que se esta costeando.
# Sigue sin ser SHAP (decision D5), y ahora por un motivo mas fuerte que antes: SHAP
# ATRIBUYE el UITI que ya hay a las variables que lo explican, y la pregunta del panel es
# la contraria -- que variable, y en que valor, lo BAJA. Una atribucion alta puede
# corresponder a una variable que no se puede mover en la direccion util, y su linea base
# es una distribucion de datos, no una intervencion.
# Tampoco es ya el barrido min-max, que tenia dos defectos para esa pregunta: su magnitud
# `max(|delta-|, |delta+|)` no llevaba SIGNO -- una variable que dispara el riesgo en los
# dos extremos encabezaba el ranking -- y solo miraba los dos EXTREMOS, cuando medido
# sobre este modelo 10 de los 15 controles tienen su mejor valor en el INTERIOR del rango
# para alguna bolsa.
# Lo que corre es una rejilla por control sobre el MISMO modelo y la MISMA unidad que el
# mapa simulado -- la bolsa (vano, ventana) del cuaderno 05 --: se prueba cada valor, se
# guarda el que MINIMIZA el u-hat de cada bolsa y se ordena por cuanto lo baja, en
# ordenes de magnitud. Cuesta `1 + puntos x knobs_numericos` pasadas para TODA la
# seleccion, no una tanda por vano: cada pasada ya devuelve un u-hat por bolsa (ver
# `relevancia_hacia_uiti_minimo`). Corre DENTRO del job del boton "Simular" y bajo la
# misma epoca, para que mapa, grafo y top describan siempre la MISMA seleccion.
TOP_VACIO = {}


def _calcular_top_por_vano(seleccion):
    """Que variables pueden llevar a cada vano a su UITI minimo, sobre las bolsas que
    ya resolvio el job de simulacion.

    Recorre `KNOBS_PANEL` y NUNCA `KNOBS` entero: el ranking se queda en los dos
    conjuntos que el panel ofrece -- intervencion y escenario -- por la misma razon por
    la que el panel no los ofrece. Con el catalogo completo entrarian las refutadas, y
    el tablero podria terminar diciendo que la variable mas relevante de un vano es
    `CNT_TRF`, los trafos afectados EN LA FALLA: se mide DESPUES del evento que el
    modelo intenta anticipar. Eso no seria un ranking flojo, seria la flecha del
    analisis al reves, sosteniendo una orden de trabajo que no arregla nada. Tampoco
    entran las de lectura unica: no se puede rankear por relevancia lo que no se deja
    mover.
    """
    return relevancia_hacia_uiti_minimo(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL, knobs=KNOBS_PANEL,
        top=TOP_VARIABLES_POR_VANO, puntos=PUNTOS_REJILLA_RELEVANCIA,
        grupos=GRUPO_POR_KNOB, label_encoders=label_encoders,
        max_values_imputed=max_values_imputed,
    )


def _calcular_plan(seleccion):
    """La COMBINACION de cambios que lleva a cada vano al grupo Bajo.

    El ranking contesta de a una variable, y eso alcanza en Medio y casi nunca mas
    arriba: medido sobre 59 bolsas de 40 circuitos, en Medio 20 de 33 llegan con una
    sola, en Medio-Alto 0 de 18 y en Alto 0 de 8. Para los dos grupos donde la pregunta
    de mantenimiento pesa mas, una sola variable NUNCA basta, asi que el plan no es un
    adorno del ranking sino su continuacion.
    """
    return plan_hacia_clase_minima(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL, knobs=KNOBS_PANEL,
        puntos=PUNTOS_REJILLA_RELEVANCIA, max_pasos=MAX_PASOS_PLAN,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )


def _diagnostico_del_circuito():
    """Los `TOP_VANOS_CIRCUITO` vanos de MAYOR UITI del circuito en la ventana activa, y
    que variables los bajarian al grupo Bajo.

    Contesta la pregunta con la que se abre una jornada -- por donde empiezo en este
    circuito -- que las demas vistas del tablero no contestan: el mapa exige mirar tramo
    a tramo y el panel exige haber elegido ya los vanos.

    El ranking se AGREGA sobre los diez y no se da vano por vano: la pregunta es que obra
    programar para el conjunto, y diez rankings sueltos son diez decisiones. Se promedia
    la caida en ordenes de magnitud, que es la escala de la geometria; en unidades de
    UITI, el vano mas caro se llevaria el promedio entero.

    Las dos mitades se reportan por SEPARADO y con tamanios distintos a proposito. Lo que
    se HACE -- intervencion -- es lo que se cotiza, y va mas largo; lo que se ANTICIPA --
    escenario -- sirve para saber bajo que condiciones esa obra rinde, y con tres basta.
    Mezclarlas en una sola lista dejaria al clima copandola, como ya se midio.
    """
    circuito, ventana_i, _marcados = _seleccion_actual()
    datos = DATOS_VENTANA[ventana_i]
    clases = clases_para(circuito, ventana_i)

    def _del_grupo(clase):
        # `datos` y `clases` estan indexados por TEXTO y `VANOS_POR_CIRCUITO` puede traer
        # los fid como numero: sin la coercion no coincide ninguno y el diagnostico sale
        # vacio sin decir por que.
        filas = [(str(f), *datos[str(f)]) for f in VANOS_POR_CIRCUITO.get(circuito, [])
                 if str(f) in datos and clases.get(str(f)) == clase]
        filas.sort(key=lambda t: -t[1])
        return filas

    # Alto primero y Medio-Alto para completar: la pregunta es por donde EMPEZAR, y un
    # vano en Medio no es por donde se empieza mientras queden vanos en Alto sin atender.
    # Dentro de cada grupo manda el UITI, que es lo que ordena la urgencia.
    por_grupo = {clase: _del_grupo(clase) for clase in GRUPOS_DIAGNOSTICO}
    peores = []
    for clase in GRUPOS_DIAGNOSTICO:
        peores.extend(por_grupo[clase][:TOP_VANOS_CIRCUITO - len(peores)])
    if not peores:
        return {'circuito': circuito, 'ventana': VENTANAS[ventana_i], 'vanos': [],
                'por_grupo': {c: 0 for c in GRUPOS_DIAGNOSTICO}, 'n_puntuados': 0,
                'clases': clases, 'ranking': {}, 'intervencion': [], 'escenario': []}

    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=[f for f, _u, _n in peores])
    if not seleccion['n_bolsas']:
        return None
    # `top` sin recorte y sin cuota: la cuota reparte DENTRO de un vano, y aqui el
    # reparto se hace despues, sobre el promedio de los diez.
    ranking = relevancia_hacia_uiti_minimo(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL, knobs=KNOBS_PANEL,
        top=len(KNOBS_PANEL), puntos=PUNTOS_REJILLA_RELEVANCIA,
        grupos=GRUPO_POR_KNOB, label_encoders=label_encoders,
        max_values_imputed=max_values_imputed,
    )
    acumulado = {}
    for datos_vano in ranking.values():
        for fila in datos_vano['filas']:
            entrada = acumulado.setdefault(
                fila['label'], {'grupo': fila['grupo'], 'knob_id': fila['knob_id'],
                                'caidas': [], 'alcanza': 0})
            entrada['caidas'].append(fila['caida_log'])
            entrada['alcanza'] += int(fila['alcanza'])
    for entrada in acumulado.values():
        entrada['media'] = float(np.mean(entrada['caidas']))
    def _mejores(grupo, cuantas):
        filas = [(lab, e) for lab, e in acumulado.items() if e['grupo'] == grupo]
        filas.sort(key=lambda t: -t[1]['media'])
        return filas[:cuantas]
    return {
        'circuito': circuito,
        'ventana': VENTANAS[ventana_i],
        'vanos': peores,
        # Cuantos aporto cada grupo: es lo que sostiene el aviso cuando no hay diez.
        'por_grupo': {clase: sum(1 for f, _u, _n in peores if clases.get(f) == clase)
                      for clase in GRUPOS_DIAGNOSTICO},
        'clases': clases,
        'n_puntuados': len(ranking),
        # El ranking COMPLETO por vano, no solo el promedio: los botones de aplicar
        # necesitan el valor sugerido para CADA vano, y el promedio no lo tiene.
        'ranking': ranking,
        'intervencion': _mejores('Intervencion', TOP_INTERVENCION_CIRCUITO),
        'escenario': _mejores('Escenario', TOP_ESCENARIO_CIRCUITO),
    }


def _texto_del_diagnostico(diag):
    """El diagnostico como tabla compacta. Va en HTML y no en la figura: son tres listas
    de largos distintos y meterlas en un panel de plotly obligaria a robarle sitio a los
    mapas para decir algo que se lee mejor como texto."""
    if diag is None:
        return ('<span style="font-size:12px;color:#5b4a48;">Presiona <b>Diagnostico del '
                'circuito</b> para ver los ' + str(TOP_VANOS_CIRCUITO) + ' vanos mas '
                'criticos de la ventana activa y que los bajaria.</span>')
    if not diag['vanos']:
        return ('<span style="font-size:12px;color:#b91c1c;">En '
                f'{diag["circuito"]}, {diag["ventana"]["etiqueta"]} no hay ningun vano en '
                'Alto ni en Medio-Alto. No hay diagnostico que dar: no es un fallo, es '
                'que ese circuito no tiene por donde empezar en esa ventana.</span>')
    _filas_vanos = ''.join(
        f'<tr><td><b>{fid}</b></td>'
        f'<td style="color:#5b4a48;">{NOMBRES_GRUPOS[diag["clases"].get(fid, 0)]}</td>'
        f'<td style="text-align:right;">{u:,.2f}</td>'
        f'<td style="text-align:right;">{n}</td></tr>'
        for fid, u, n in diag['vanos'])
    def _lista(titulo, filas, color):
        if not filas:
            return f'<div><b>{titulo}</b>: sin variables de este tipo.</div>'
        renglones = ''.join(
            f'<tr><td>{i + 1}.</td><td><b>{lab}</b></td>'
            f'<td style="text-align:right;">{e["media"]:.3f}</td>'
            f'<td style="color:#15803d;">'
            f'{"sola basta en " + str(e["alcanza"]) + " de " + str(diag["n_puntuados"]) if e["alcanza"] else ""}'
            f'</td></tr>'
            for i, (lab, e) in enumerate(filas))
        return (f'<div style="margin-right:26px;"><b style="color:{color};">{titulo}</b>'
                '<table style="font-size:11px;border-collapse:collapse;">'
                f'{renglones}</table></div>')
    _reparto = ' + '.join(
        f'{diag["por_grupo"][c]} en {NOMBRES_GRUPOS[c]}' for c in GRUPOS_DIAGNOSTICO
        if diag['por_grupo'].get(c))
    # El aviso NO es decoracion: sin el, una lista de cuatro vanos se lee como que el
    # circuito tiene cuatro criticos, cuando lo que pasa es que no llegan a diez.
    _aviso = ('' if len(diag['vanos']) >= TOP_VANOS_CIRCUITO else
              f'<br><span style="color:#b45309;">Se identificaron '
              f'<b>{len(diag["vanos"])}</b> vanos, no {TOP_VANOS_CIRCUITO}: '
              f'{diag["circuito"]} no tiene mas en Alto ni en Medio-Alto en esta '
              'ventana.</span>')
    return (
        '<div style="font-size:12px;color:#2b2b2b;">'
        f'<b>Diagnostico de {diag["circuito"]}</b> &mdash; {diag["ventana"]["etiqueta"]}: '
        f'{diag["ventana"]["periodo"]} &mdash; '
        f'{len(diag["vanos"])} vanos ({_reparto}), {diag["n_puntuados"]} puntuados'
        f'{_aviso}'
        '<div style="display:flex;flex-flow:row wrap;align-items:flex-start;'
        'margin-top:4px;">'
        '<div style="margin-right:26px;"><b>Vanos (grupo, UITI, eventos)</b>'
        f'<table style="font-size:11px;border-collapse:collapse;">{_filas_vanos}</table>'
        '</div>'
        + _lista(f'Intervencion &mdash; que HACER (caida media)', diag['intervencion'],
                 '#0072b2')
        + _lista(f'Escenario &mdash; bajo que CONDICIONES', diag['escenario'], '#b45309')
        + '</div>'
        '<span style="color:#5b4a48;">La caida es el promedio sobre los vanos listados, '
        'en ordenes de magnitud de UITI, y las dos columnas COMPARTEN esa escala: se '
        'pueden comparar entre si, y conviene hacerlo. Cuando el escenario saca uno o dos '
        'ordenes de magnitud a la intervencion, lo que dice el modelo es que en estos '
        'vanos manda el clima y la obra rinde poco; eso no invalida la obra, pero si la '
        'expectativa de cuanto va a bajar el riesgo. Es una variable a la vez: arriba de '
        'Medio casi nunca basta una sola, y la receta combinada esta en el plan de abajo '
        'para los vanos que marques.</span></div>')


def _texto_del_plan(plan):
    """El plan como una tabla compacta bajo el panel. Va en HTML y no en la figura: son
    hasta cinco listas de cuatro renglones con nombres largos y un valor cada uno, y eso
    en un panel de plotly obliga a elegir entre recortarlo o robarle sitio a los mapas."""
    if not plan:
        return ('<span style="font-size:12px;color:#5b4a48;">El plan hacia el grupo Bajo '
                'aparece al presionar <b>Simular</b>.</span>')
    filas = []
    for fid, datos in plan.items():
        if not datos['pasos']:
            estado = ('ya esta en el grupo Bajo' if datos['alcanza']
                      else 'ninguna variable lo mueve')
            filas.append(f'<tr><td><b>{fid}</b></td><td colspan="2"><i>{estado}</i></td></tr>')
            continue
        receta = ' + '.join(f'{p["label"]} &rarr; <b>{p["valor"]:,.4g}</b>'
                            if isinstance(p['valor'], (int, float))
                            else f'{p["label"]} &rarr; <b>{p["valor"]}</b>'
                            for p in datos['pasos'])
        veredicto = ('<span style="color:#15803d;">llega a Bajo</span>' if datos['alcanza']
                     else f'<span style="color:#b91c1c;">se queda en '
                          f'{NOMBRES_GRUPOS[datos["clase_final"]]}</span>')
        filas.append(f'<tr><td><b>{fid}</b></td><td>{receta}</td>'
                     f'<td>UITI {datos["u_base"]:,.2f} &rarr; {datos["u_final"]:,.2f} '
                     f'({veredicto})</td></tr>')
    return ('<div style="font-size:12px;color:#2b2b2b;"><b>Plan hacia el grupo Bajo</b> '
            '<span style="color:#5b4a48;">(descenso goloso, hasta '
            f'{MAX_PASOS_PLAN} cambios por vano; es un buen plan, no el minimo '
            'demostrable)</span>'
            '<table style="font-size:11px;border-collapse:collapse;">'
            + ''.join(filas) + '</table></div>')


def _pintar_top_por_vano(por_vano):
    """Repaint puro, cero pasadas del modelo.

    Un grupo de barras por vano. Cada TRAZA es una posicion del ranking -- la 1a, la 2a...
    -- y no una variable: las variables cambian de vano a vano, asi que una traza por
    variable necesitaria tantas como el catalogo entero y casi todas vacias.
    El nombre va DENTRO de la barra: con cinco grupos de diez barras no hay sitio para una
    leyenda de cincuenta entradas, y el rotulo pegado al dato no obliga a cruzarlo.

    Cual de los tres rotulos se escribe lo decide `rotulo_en_barra` con el largo de CADA
    barra en pixeles: el resumen si cabe, sus iniciales si no, y nada antes que un texto
    cortado que se monte sobre la barra vecina. El nombre completo esta siempre en la
    etiqueta del mouse, que es donde se resuelve la duda.
    """
    vanos = list(por_vano)
    # El rango del eje se decide ANTES de escribir las barras: el rotulo de cada una
    # depende de cuantos pixeles mide, y eso solo se sabe con el rango ya fijado. Con las
    # trazas vacias un eje lineal autoescala a [-1, 4] y muestra marcas negativas para una
    # caida que no puede serlo.
    _tope = max((f['caida_log'] for v in por_vano.values() for f in v['filas']),
                default=0.0)
    _rango = _tope * 1.15 if _tope > 0 else 1.0
    _px_por_unidad = ALTO_PANEL_TOP_PX / _rango
    with fig.batch_update():
        fig.update_yaxes(range=[0, _rango], row=3, col=3)
        for _posicion in range(TOP_VARIABLES_POR_VANO):
            _traza = fig.data[IDX['top_vano'][_posicion]]
            _x, _y, _texto, _hover, _colores = [], [], [], [], []
            for _fid in vanos:
                _datos = por_vano[_fid]
                _filas = _datos['filas']
                if _posicion >= len(_filas):
                    continue
                _fila = _filas[_posicion]
                _x.append(_fid)
                _y.append(_fila['caida_log'])
                _texto.append(rotulo_en_barra(_fila['label'],
                                              _fila['caida_log'] * _px_por_unidad))
                # VERDE si esa sola variable basta para caer en el grupo Bajo. Es el
                # mismo verde del recuadro del mapa simulado y significa lo mismo --
                # baja de grupo de criticidad -- asi que el color no pide aprender un
                # codigo nuevo. El resto conserva la rampa de posicion del ranking.
                _colores.append(COLOR_CAJA_MEJORA if _fila['alcanza']
                                else COLOR_POSICION_BARRA[_posicion])
                _meta = ('ya esta en el grupo Bajo' if _datos['ya_en_clase_minima']
                         else 'el grupo Bajo es inalcanzable con sus eventos'
                         if _datos['objetivo_u'] is None
                         else f'para Bajo: u &lt; {_datos["objetivo_u"]:,.3f}')
                _avance = ('' if _fila['avance'] is None
                           else f'<br>Cubre el {100 * _fila["avance"]:.0f}% del camino')
                # El valor de un control CATEGORICO es texto -- su categoria -- y no
                # un numero: formatearlo con `:,.4g` revienta el repintado entero.
                _valor = (f'{_fila["valor"]:,.4g}'
                          if isinstance(_fila['valor'], (int, float))
                          else str(_fila['valor']))
                _hover.append(
                    f'<b>Vano {_fid}</b><br>{_posicion + 1}o: {_fila["label"]}'
                    f'<br>Llevarla a <b>{_valor}</b>'
                    f'<br>UITI: {_datos["u_base"]:,.3f} -> {_fila["u_optimo"]:,.3f}'
                    f'<br>Caida: {_fila["caida_log"]:.2f} ordenes de magnitud'
                    f'{_avance}<br>{_meta}'
                    + ('<br><b>Sola alcanza el grupo Bajo</b>' if _fila['alcanza'] else '')
                )
            _traza.x, _traza.y = _x, _y
            _traza.text, _traza.hovertext = _texto, _hover
            _traza.marker.color = _colores


def _pintar_barras_uiti(tabla_simulada, ventana_i=None, circuito=None):
    """Fila 4: el UITI acumulado MEDIDO de cada vano contra el que predice la simulacion,
    y un ultimo grupo con el circuito completo.

    Reemplaza a los violines. Con diez vanos, dos violines resumian en una densidad lo
    que aqui se lee vano por vano, que es el grano en el que se decide una obra.

    Las dos barras son cantidades de NATURALEZA distinta -- la izquierda es lo que dice
    la base de datos, la derecha es lo que dice el modelo --, y eso no se puede esconder:
    medido sobre 599 bolsas reales, el modelo correlaciona 0,950 con el UITI observado
    (ordena bien) pero su nivel corre +34%. De ahi las dos cosas que acompanian a las
    barras: el error de la simulada es el desfase del modelo en la BASE de ese mismo vano
    -- lo unico local y medible que hay -- y el titulo publica la reduccion con su +-.
    Sin eso, el sesgo del modelo se leeria como ahorro.

    La alternativa descartada queda dicha porque parece mas rigurosa y no lo es:
    reamuestrear los eventos de cada bolsa y volver a predecir da desviacion 0,000
    (medido con 50 y 200 replicas). La prediccion no depende de que eventos cayeron en
    la bolsa, asi que esa barra de error habria sido adorno sobre la incertidumbre real,
    que es dos ordenes de magnitud mayor.
    """
    if tabla_simulada is None or len(tabla_simulada) == 0:
        barras = barras_uiti_por_vano(None, observados={}, total_circuito=0.0)
    else:
        _datos = DATOS_VENTANA[ventana_i]
        _observados = {str(f): _datos[str(f)][0] for f in tabla_simulada['FID_VANO']
                       if str(f) in _datos}
        # El total del circuito sale de TODOS sus vanos en la ventana activa, no solo de
        # los marcados: es lo que deja ver cuanto pesa la intervencion sobre el conjunto.
        _total = sum(_datos[str(f)][0] for f in VANOS_POR_CIRCUITO.get(circuito, [])
                     if str(f) in _datos)
        barras = barras_uiti_por_vano(tabla_simulada, observados=_observados,
                                      total_circuito=_total)
    with fig.batch_update():
        _obs = fig.data[IDX['barra_observada']]
        _sim = fig.data[IDX['barra_simulada']]
        _obs.x, _obs.y = barras['x'], barras['observado']
        _obs.hovertext = barras['hover']
        _sim.x, _sim.y = barras['x'], barras['simulado']
        _sim.hovertext = barras['hover']
        _sim.error_y.array = barras['error']
        fig.layout.annotations[IDX_ANOTACION_BARRAS].text = (
            '' if barras['x'] else 'Marca vanos y presiona <b>Simular</b>.')
        fig.layout.annotations[IDX_TITULO_BARRAS].text = _titulo_de_barras(barras)


def _titulo_de_barras(barras):
    """El titulo lleva la cifra que el tablero viene a producir: cuanto baja el UITI
    acumulado de los vanos intervenidos, y con cuanta incertidumbre.

    El `+-` no es un adorno estadistico: es el desfase acumulado del modelo, y en estos
    datos puede ser del orden de la propia reduccion. Publicar la reduccion sola la haria
    pasar por un resultado firme."""
    if barras['reduccion'] is None:
        return 'UITI acumulado: medido contra simulado'
    return (f'UITI acumulado: medido contra simulado &mdash; baja '
            f'<b>{barras["reduccion"]:,.1f}</b> &plusmn; {barras["desviacion"]:,.1f} '
            f'en los {len(barras["x"]) - 1} vanos')


def _pintar_costos(costos):
    """Fila 5: lo que cuesta ejecutar el plan, por vano y en total.

    Es la conclusion del tablero. Todo lo de arriba dice cuanto BAJA el riesgo; esto
    dice cuanto cuesta bajarlo, y una decision de mantenimiento no se toma con una sola
    de las dos mitades.

    El TOTAL va como una barra mas y en el MISMO eje, aunque sea siempre la mas alta:
    es la suma de las otras, y darle un eje propio dejaria de mostrar cuanto pesa cada
    vano dentro de ella. Se distingue por color, no por escala.

    Un vano marcado sin actividades da una barra en CERO, que es un dato y no un hueco:
    dice que se simulo su riesgo sin obra asociada. Quitarlo del eje lo haria parecer no
    estudiado.
    """
    if not costos or not costos['por_vano']:
        with fig.batch_update():
            _traza = fig.data[IDX['costos']]
            _traza.x, _traza.y = [], []
            _traza.text, _traza.hovertext = [], []
            _traza.marker.color = []
            fig.layout.annotations[IDX_ANOTACION_COSTOS].text = (
                'Marca actividades del contrato para cada vano y presiona '
                '<b>Simular</b>.')
        return

    _por_vano = costos['por_vano']
    _x = [*_por_vano, ETIQUETA_TOTAL]
    _y = [*(v['total'] for v in _por_vano.values()), costos['total']]
    _colores = [COLOR_BARRA_COSTO] * len(_por_vano) + [COLOR_BARRA_TOTAL]
    _hover = []
    for _fid, _datos in _por_vano.items():
        # El desglose viaja en el hover: el total contesta cuanto y el detalle contesta
        # por que, sin obligar a volver a abrir el panel para averiguarlo. Va ordenado de
        # mayor a menor, asi que la primera linea es la actividad que hay que negociar.
        _lineas = [f'<b>Vano {_fid}</b><br>Costo: {_datos["total"]:,.0f} COP']
        _lineas += [f'{_r["repeticiones"]} x {_r["item"][:60]}: {_r["subtotal"]:,.0f}'
                    for _r in _datos['renglones']] or ['Sin actividades marcadas']
        _hover.append('<br>'.join(_lineas))
    _hover.append(f'<b>{ETIQUETA_TOTAL}</b><br>{len(_por_vano)} vanos<br>'
                  f'{costos["total"]:,.0f} COP')
    with fig.batch_update():
        _traza = fig.data[IDX['costos']]
        _traza.x, _traza.y = _x, _y
        _traza.marker.color = _colores
        _traza.text = [f'{v:,.0f}' for v in _y]
        _traza.hovertext = _hover
        fig.layout.annotations[IDX_ANOTACION_COSTOS].text = ''


_pintar_top_por_vano(TOP_VACIO)   # vacio hasta el primer "Simular"
_pintar_barras_uiti(None)
_pintar_costos(None)


In [ ]:
# --- Fila 2: mapa "Criticidad Simulada" + boton "Simular" (design section A, decision D2)
# El boton es el UNICO disparador y hace TRES cosas de una sola vez, bajo la misma epoca:
# el mapa simulado, el grafo reconstruido de la seleccion y el barrido de importancia de
# la celda anterior. Ya no hay alternador base/simulado/delta: el mapa de la fila 2
# muestra SIEMPRE la clase simulada, que es lo que el boton promete.
#
# El mapa y el grafo salen del modelo MIL del cuaderno 05, que puntua BOLSAS: una bolsa es
# una celda (vano, ventana) y su clase sale de `asignar_clase(n_obs OBSERVADO, u-hat
# predicho)` sobre la geometria KMeans de 01.4 -- la MISMA con la que se pinta el mapa
# base, que es por lo que los dos mapas comparten paleta por construccion y no por
# convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.
# Debounce asincronico (design section A): `asyncio.ensure_future` + cancelacion en el
# propio event loop del kernel, NUNCA `threading.Timer` -- ipykernel enruta la salida de
# los widgets con el parent header thread-local, asi que una escritura desde un hilo en
# segundo plano cae en la celda equivocada. `_EPOCA` es el guard de epoca: cualquier evento
# que invalide un job en vuelo la avanza y la escritura tardia se descarta.
from chec_local_interpreter.vano_app_015 import (
    DEBOUNCE_SEGUNDOS,
    ESTADO_BASE,
    ESTADO_SIMULADO,
    aplicar_si_vigente,
    clases_por_fid_para_estado,
    siguiente_epoca,
)
from chec_local_interpreter.vano_widgets import widget_for_knob

# El estado vacio se PIDE a la funcion en vez de escribirlo a mano: escrito a mano se
# desincroniza en cuanto `trazas_grafo` agrega una columna, que es exactamente lo que
# paso al sumarle el indice de modalidad a cada nodo.
GRAFO_VACIO = trazas_grafo(np.zeros((1, 1)), [''])

_EPOCA = 0
_tarea_pendiente_simular = None
_ultimo_resultado_simulacion = None   # DataFrame de simulate_explicit_overrides, o None
_ultima_seleccion_simulada = None     # (circuito, ventana_i) al que corresponde ese resultado

STATUS = widgets.HTML(
    'Sin simular todavia -- elige variables (opcional) y presiona "Simular".'
)
PLAN = widgets.HTML(_texto_del_plan(None))

# El panel NO ofrece las variables refutadas. Mientras estuvieran en la lista, el
# tablero las presentaba como equivalentes a la poda o a la puesta a tierra, y tarde o
# temprano alguien mueve las coordenadas de un vano creyendo que eso es un escenario.
# Quitarlas del panel no las saca de la SIMULACION: un override solo se escribe si se
# fija, asi que entran al modelo con el valor OBSERVADO de cada vano, que es lo que
# corresponde. Lo unico que se pierde es poder moverlas.
KNOBS_PANEL = knobs_simulables(KNOBS)
KNOBS_BLOQUEADOS = knobs_bloqueados(KNOBS)
_knobs_por_id = {k.id: k for k in KNOBS_PANEL}
# Casillas y no `SelectMultiple`, por el mismo motivo que la lista de vanos: en un
# `SelectMultiple` un clic sin ctrl borra todo lo ya elegido, y aqui justamente se quiere
# simular VARIAS variables a la vez. Cada casilla es independiente y `value` sigue siendo
# la tupla de knob ids, asi que `_reconstruir_controles_knob` no se entera del cambio.
# CUATRO columnas: dos para lo que se puede hacer y dos para lo que se quiere
# anticipar. Una lista corrida de dieciocho casillas obliga a recordar el veredicto de
# cada variable para saber a cual de las dos preguntas pertenece -- "que obra hago" y
# "que pasa si" --; en columnas eso lo dice la posicion.
COLUMNAS_KNOBS = columnas_panel(KNOBS_PANEL)
knob_selector_widget = construir_selector_casillas(
    columnas=[(titulo, [(k.label, k.id) for k in knobs]) for titulo, knobs in COLUMNAS_KNOBS],
    titulo='', alto='210px', ancho_casilla='215px',
    layout=widgets.Layout(width='100%'),
)
# Se NOMBRAN en vez de dejarlas desaparecer: una lista que se acorta sin explicacion se
# lee como que faltan variables, no como una decision.
AVISO_BLOQUEADOS = widgets.HTML(
    '' if not KNOBS_BLOQUEADOS else
    '<span style="font-size:12px;color:#5b4a48;">No simulables: <b>'
    + ', '.join(k.label for k in KNOBS_BLOQUEADOS) + '</b>. '
    'Entran a la simulacion con el valor observado de cada vano, pero no se pueden '
    'mover: la tabla de arriba dice por que. </span>')
# --- Las actividades del contrato, la otra mitad de la decision -----------------------
# Mismo patron que la lista de variables, y por la misma razon: UNA lista compartida de
# casillas arriba, y cada vano marcado recibe abajo su propia fila por actividad elegida.
# Repetir las 125 casillas por vano seria una pantalla de 625 casillas para elegir tres.
# El rotulo arranca por el PRECIO. Es el dato con el que se elige entre dos podas, y los
# nombres del contrato llegan a 143 caracteres: puesto al final, el ancho fijo de la
# casilla lo recortaria justo a el.
item_selector_widget = construir_selector_casillas(
    [(f'$ {_item.costo:,.0f}  --  {_item.nombre}', _item.nombre)
     for _item in CATALOGO_COSTOS.items],
    titulo='', alto='150px', ancho_casilla=ANCHO_CASILLA_ITEM,
    layout=widgets.Layout(width='100%'),
)
# Se NOMBRAN, igual que las variables no simulables: doce actividades ausentes sin
# explicacion se leen como que el contrato no las incluye.
AVISO_SIN_COSTO = widgets.HTML(
    '' if not CATALOGO_COSTOS.sin_costo else
    '<span style="font-size:12px;color:#5b4a48;">'
    f'{len(CATALOGO_COSTOS.sin_costo)} actividades del contrato no tienen costo unitario '
    'en el libro y no se ofrecen: no se puede costear lo que no tiene precio. '
    f'<i>{", ".join(n[:40] for n in CATALOGO_COSTOS.sin_costo[:3])}...</i></span>')
controles_knob_box = widgets.VBox([])
# La rejilla se muestra de a `VANOS_POR_PAGINA` columnas. Los controles de los vanos que
# NO estan en pantalla siguen existiendo y conservando su valor: la simulacion los aplica
# igual, y paginar no puede ser una forma silenciosa de descartar lo que se fijo.
_COLUMNAS_VANO = []          # [(fid, VBox)] de TODOS los vanos, no solo los visibles
_PAGINA = 0
boton_pagina_anterior = widgets.Button(description='< Anteriores',
                                       layout=widgets.Layout(width='130px'))
boton_pagina_siguiente = widgets.Button(description='Siguientes >',
                                        layout=widgets.Layout(width='130px'))
PAGINA_LABEL = widgets.HTML('')
# {fid: {nombre_actividad: Dropdown de repeticiones}}. Vive aparte de `_controles_por_vano`
# porque son dos preguntas distintas -- que le muevo al vano y que obra le hago -- y el
# modelo solo consume la primera.
_costos_por_vano = {}
# {fid o GRANO_CIRCUITO: {knob_id: widget}}. Una COLUMNA por vano, con el vano escrito
# encima: sin ese encabezado cinco columnas de deslizadores identicos son indistinguibles.
_controles_por_vano = {}
GRANO_CIRCUITO = '(todo el circuito)'


def _seleccion_de_bolsas():
    """Las bolsas de la seleccion activa, o None si no hay ninguna. Es lo que hace falta
    para saber que vanos tienen columna y en que valor arranca cada control."""
    circuito, ventana_i, marcados = _seleccion_actual()
    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=marcados)
    return seleccion if seleccion['n_bolsas'] else None


def _valores_iniciales(seleccion, marcados):
    """En que valor abre cada control. La regla: el valor ACTUAL de esa variable para ESE
    vano en la ventana activa -- mediana si el vano trae varias instancias, moda si la
    variable es categorica (ver `valores_actuales_por_vano`).

    Un control que abriera en un valor por defecto pediria volver a teclear un dato que el
    modelo ya tiene, y peor: cualquier variable que se dejara quieta simularia al vano en
    un valor que nunca fue el suyo.

    Sin vanos marcados el grano es el circuito completo, y entonces hay UNA columna: todas
    las instancias se resumen juntas, que es exactamente lo que el override global escribe.
    """
    if seleccion is None:
        return {}
    X_sel = X_INST[seleccion['filas']]
    if marcados:
        return valores_actuales_por_vano(
            X_sel, FEATURES_MIL, instance_bag=seleccion['instance_bag'],
            fids=seleccion['fid'], knobs=KNOBS_PANEL, label_encoders=label_encoders,
        )
    return valores_actuales_por_vano(
        X_sel, FEATURES_MIL,
        instance_bag=np.zeros(len(X_sel), dtype=np.int64), fids=[GRANO_CIRCUITO],
        knobs=KNOBS_PANEL, label_encoders=label_encoders,
    )


def _control_con_valor(knob, valor):
    """El control del knob, abierto en `valor`. Un valor fuera de los limites del
    deslizador no se fuerza: se recorta, porque `FloatSlider` lanza si el valor cae fuera
    de [min, max] y tumbar el panel entero por un decimal no vale la pena."""
    control = widget_for_knob(knob)
    if valor is None:
        return control
    if knob.kind == 'numeric':
        lo, hi = knob.bounds
        control.value = float(min(max(float(valor), lo), hi))
    elif knob.kind == 'categorical' and valor in (knob.categories or ()):
        control.value = valor
    return control


def _fila_de_actividad(nombre, costo):
    """Una actividad dentro de la columna de un vano: cuanto vale y cuantas veces va.

    El costo unitario se muestra AL LADO del desplegable y no solo en el catalogo: al
    componer un plan de tres actividades, ir a buscar cada precio a la lista de arriba
    es exactamente el trabajo que el panel existe para evitar.
    """
    # Arranca en 1 y no en 0: si se marco la actividad es porque va. El CERO esta para
    # el vano donde NO va -- es lo que permite que una lista compartida no obligue a
    # darle la misma obra a los cinco vanos marcados.
    repeticiones = widgets.Dropdown(
        options=[(str(n), n) for n in range(0, MAX_REPETICIONES + 1)], value=1,
        description='', layout=widgets.Layout(width='58px'))
    etiqueta = widgets.HTML(
        f'<span style="font-size:11px;" title="{nombre}">{nombre[:46]}'
        f'{"..." if len(nombre) > 46 else ""}<br>'
        f'<b>$ {costo:,.0f}</b> c/u</span>')
    return repeticiones, widgets.HBox(
        [repeticiones, etiqueta],
        layout=widgets.Layout(align_items='center', margin='0 0 4px 0'))


def _bloque_de_costos(fid):
    """El bloque de actividades de un vano, o nada si no se marco ninguna.

    Solo aparece con vanos marcados: una intervencion se cotiza sobre vanos concretos,
    y "cuanto cuesta intervenir el circuito entero" no es una pregunta que este panel
    pueda contestar con esta lista de precios.
    """
    elegidas = list(item_selector_widget.value)
    if not elegidas or fid == GRANO_CIRCUITO:
        return []
    filas, controles = [], {}
    for nombre in elegidas:
        control, fila = _fila_de_actividad(nombre, COSTO_POR_ITEM[nombre])
        controles[nombre] = control
        filas.append(fila)
    _costos_por_vano[fid] = controles
    return [widgets.HTML(
        '<div style="font-size:11px;color:#5b4a48;border-top:1px solid #e4c4c0;'
        'margin-top:6px;padding-top:4px;">Actividades del contrato</div>'), *filas]


_PIE_REJILLA = widgets.HTML('')


def _mostrar_pagina():
    """La rebanada visible de la rejilla, con su navegacion.

    Con diez vanos y veintiseis controles la rejilla es un muro: cinco columnas es lo que
    cabe legible a lo ancho del panel, y por encima las columnas se estrechan hasta que el
    nombre de la variable y su deslizador dejan de caber en la misma linea.

    La navegacion solo aparece si HAY mas de una pagina. Dos botones deshabilitados sobre
    tres vanos son dos controles que no hacen nada y que hay que leer igual.
    """
    total = len(_COLUMNAS_VANO)
    if not total:
        controles_knob_box.children = [_PIE_REJILLA]
        return
    paginas = (total + VANOS_POR_PAGINA - 1) // VANOS_POR_PAGINA
    desde = _PAGINA * VANOS_POR_PAGINA
    visibles = _COLUMNAS_VANO[desde:desde + VANOS_POR_PAGINA]
    boton_pagina_anterior.disabled = _PAGINA <= 0
    boton_pagina_siguiente.disabled = _PAGINA >= paginas - 1
    PAGINA_LABEL.value = (
        f'<span style="font-size:12px;color:#5b4a48;">Vanos '
        f'<b>{desde + 1}-{min(desde + VANOS_POR_PAGINA, total)}</b> de {total} '
        f'(pagina {_PAGINA + 1} de {paginas}). Los controles de los vanos que no se ven '
        'conservan su valor y entran igual a la simulacion.</span>')
    navegacion = ([widgets.HBox([boton_pagina_anterior, boton_pagina_siguiente,
                                 PAGINA_LABEL],
                                layout=widgets.Layout(align_items='center'))]
                  if paginas > 1 else [])
    controles_knob_box.children = [
        *navegacion,
        widgets.Box([caja for _fid, caja in visibles],
                    layout=widgets.Layout(display='flex', flex_flow='row wrap',
                                          align_items='flex-start', width='100%')),
        _PIE_REJILLA,
    ]


def _mover_pagina(paso):
    global _PAGINA
    paginas = max(1, (len(_COLUMNAS_VANO) + VANOS_POR_PAGINA - 1) // VANOS_POR_PAGINA)
    _PAGINA = min(max(_PAGINA + paso, 0), paginas - 1)
    _mostrar_pagina()


boton_pagina_anterior.on_click(lambda _b: _mover_pagina(-1))
boton_pagina_siguiente.on_click(lambda _b: _mover_pagina(1))


def _reconstruir_controles_knob(_change=None):
    """La rejilla: filas = variables elegidas, columnas = vanos elegidos.

    Se rehace cuando cambia CUALQUIERA de las dos listas, y tambien al mover circuito o
    ventana: el valor inicial de cada control es el del vano EN ESA VENTANA, asi que una
    rejilla que sobreviviera al deslizador estaria mostrando los valores de otra.

    Debajo de las variables, cada columna lleva las ACTIVIDADES del contrato marcadas,
    con su costo unitario y cuantas veces se ejecutan. Son dos preguntas sobre el mismo
    vano -- que le muevo y que obra le hago -- y por eso comparten columna sin mezclarse:
    una linea las separa, y el modelo solo consume la de arriba.
    """
    global _controles_por_vano, _costos_por_vano, _COLUMNAS_VANO, _PAGINA
    _controles_por_vano = {}
    _costos_por_vano = {}
    knob_ids = list(knob_selector_widget.value)
    _circuito, _ventana_i, marcados = _seleccion_actual()
    seleccion = _seleccion_de_bolsas()
    valores = _valores_iniciales(seleccion, marcados)

    if not knob_ids and not item_selector_widget.value:
        _COLUMNAS_VANO.clear()
        controles_knob_box.children = [widgets.HTML(
            '<span style="font-size:12px;color:#5b4a48;">Elige arriba las variables a '
            'modificar y, si vas a costear, las actividades del contrato. Cada vano '
            'marcado recibe su propia columna.</span>')]
        return
    if seleccion is None:
        _COLUMNAS_VANO.clear()
        controles_knob_box.children = [widgets.HTML(
            '<span style="font-size:12px;color:#5b4a48;">Esta seleccion no tiene celdas '
            '(vano x ventana) en la ventana activa: no hay valores desde donde '
            'arrancar.</span>')]
        return

    # El ORDEN de las columnas sale de la geometria y no del orden en que se fueron
    # marcando: asi marcar y desmarcar no baraja las columnas bajo la mano.
    columnas_fid = ([f for f in seleccion['fid'] if f in valores] if marcados
                    else [GRANO_CIRCUITO])
    columnas = []
    for fid in columnas_fid:
        controles = {}
        for knob_id in knob_ids:
            knob = _knobs_por_id[knob_id]
            controles[knob_id] = _control_con_valor(knob, valores.get(fid, {}).get(knob_id))
        _controles_por_vano[fid] = controles
        encabezado = widgets.HTML(
            f'<div style="font-weight:600;border-bottom:2px solid rgb(203,24,29);'
            f'padding-bottom:2px;margin-bottom:4px;">{fid}</div>')
        columnas.append((fid, widgets.VBox(
            [encabezado, *controles.values(), *_bloque_de_costos(fid)],
            layout=widgets.Layout(margin='0 14px 0 0', align_items='flex-start'))))

    # Un vano marcado SIN celda en la ventana activa no tiene de donde sacar un valor
    # inicial, asi que no recibe columna. Se dice: medido, con 5 vanos marcados la rejilla
    # armaba 4 columnas y el quinto desaparecia sin que nada en pantalla lo explicara.
    sin_columna = [f for f in marcados if f not in _controles_por_vano]
    aviso_faltantes = ('' if not sin_columna else
                       f'<br>Sin columna: {", ".join(sorted(sin_columna))} -- '
                       f'{"ese vano no tiene" if len(sin_columna) == 1 else "esos vanos no tienen"} '
                       'eventos en la ventana activa, asi que no hay valor actual desde '
                       'donde arrancar. La simulacion tampoco los puntua.')
    pie = widgets.HTML(
        '<span style="font-size:12px;color:#5b4a48;">Cada control abre en el valor actual '
        'de esa variable para ese vano en la ventana activa (mediana de sus instancias; '
        f'moda si es categorica).{aviso_faltantes}</span>')
    # `flex_flow='row wrap'`: con cinco columnas y una variable de nombre largo la fila se
    # pasa del ancho del panel, y sin el wrap las ultimas columnas quedaban cortadas.
    _COLUMNAS_VANO = columnas
    _PAGINA = 0
    _PIE_REJILLA.value = pie.value
    _mostrar_pagina()


knob_selector_widget.observe(_reconstruir_controles_knob, names='value')
item_selector_widget.observe(_reconstruir_controles_knob, names='value')
# La rejilla depende de la ventana y del circuito por sus VALORES INICIALES, no solo por
# que vanos existen: mover el deslizador tiene que reabrir los controles en los valores de
# la ventana nueva.
vano_widget.observe(_reconstruir_controles_knob, names='value')
ventana_widget.observe(_reconstruir_controles_knob, names='value')
circuito_widget.observe(_reconstruir_controles_knob, names='value')

boton_simular = widgets.Button(description='Simular', button_style='primary')
# Disparador PROPIO y no parte de "Simular": contesta otra pregunta -- por donde empiezo
# en este circuito -- y no depende de lo que el usuario haya marcado ni de las variables
# que haya fijado. Colgarlo de "Simular" obligaria a recalcularlo en cada escenario que
# no lo cambia, y a esperarlo aunque no se quiera.
boton_diagnostico = widgets.Button(description='Diagnostico',
                                   button_style='', tooltip='Los vanos de mayor UITI en '
                                   'la ventana activa y que los bajaria')
DIAGNOSTICO = widgets.HTML(_texto_del_diagnostico(None))
# El ultimo diagnostico se guarda porque los botones de aplicar necesitan el valor
# sugerido para CADA vano, y el texto en pantalla solo lleva el promedio.
_ULTIMO_DIAGNOSTICO = None
# Que mitades del diagnostico se aplicaron al diagnostico VIGENTE. Es lo que permite que
# el segundo boton sume al primero sin que un solo clic marque las dos.
GRUPOS_SUGERIDOS = ('intervencion', 'escenario')
NOMBRES_SUGERIDOS = {'intervencion': 'Intervencion', 'escenario': 'Escenario'}
_GRUPOS_APLICADOS = []
boton_aplicar_intervencion = widgets.Button(
    description='Aplicar intervencion sugerida', layout=widgets.Layout(width='260px'),
    tooltip='Marca los vanos del diagnostico y abre sus controles en el valor sugerido')
boton_aplicar_escenario = widgets.Button(
    description='Aplicar escenario sugerido', layout=widgets.Layout(width='260px'),
    tooltip='Marca los vanos del diagnostico y abre sus controles en el valor sugerido')
AVISO_APLICAR = widgets.HTML('')


_CAPA_VACIA = {'lat': [], 'lon': [], 'hovertext': [], 'customdata': []}


def _redibujar_mapa_predicho(*_ignorado):
    """Repaint puro, CERO llamadas al modelo.

    Antes de la primera simulacion de la seleccion activa el mapa no se dibuja: ni
    tramos, ni equipos, ni leyenda -- solo el aviso de que hay que presionar "Simular".
    Un mapa completo pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma
    forma que un resultado, y esa es justamente la confusion que la fila 2 no puede
    permitirse (D2).

    Con resultado, cada vano va del color del grupo que el simulador le predijo -- la
    MISMA paleta del mapa base, porque es la misma geometria -- y NEGRO todo lo demas
    (vano sin evento en la ventana, o no seleccionado), igual que la estructura del
    circuito en 01.4. Ademas de eso, este mapa hace dos cosas que el base no:

    - encierra a cada vano simulado en un recuadro cuyo COLOR es el desenlace -- bajo,
      se quedo igual o subio de grupo --, en tres capas de `layout.map2.layers`;
    - se ACERCA a los vanos marcados, en vez de quedarse en el encuadre del circuito.
    """
    circuito, ventana_i, _marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    hay_resultado = (
        _ultimo_resultado_simulacion is not None
        and _ultima_seleccion_simulada == (circuito, ventana_i)
    )
    if not hay_resultado:
        with fig.batch_update():
            for _i in (IDX['pred_clases'] + [IDX['pred_sin_dato'],
                                             IDX['pred_trafos'], IDX['pred_switches']]):
                fig.data[_i].lat, fig.data[_i].lon = [], []
                fig.data[_i].hovertext = []
                fig.data[_i].showlegend = False
            # Las tres cajas se apagan juntas: un recuadro de desenlace sobre un mapa que
            # no muestra ningun resultado afirmaria un cambio que nadie calculo.
            for _capa in fig.layout.map2.layers:
                _capa.source = {'type': 'FeatureCollection', 'features': []}
            _aplicar_vista('map2', _vista_del_circuito(circuito))
            fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = (
                'El mapa simulado aparece al presionar <b>Simular</b>.'
            )
        return

    clases_por_fid = clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_SIMULADO)
    # El grupo BASE viaja como una columna mas del `customdata` y no en la plantilla:
    # dentro de una traza -- que es UNA clase simulada -- el grupo base cambia de vano a
    # vano. Sin los dos numeros en la misma etiqueta, saber si el vano mejoro obliga a
    # cruzar al mapa de al lado y acordarse del color.
    clases_base = clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_BASE)
    capas = _capas_de_la_seleccion(
        clases_por_fid, campo='Criticidad simulada', nombres_clase=NOMBRES_GRUPOS,
        extra_por_fid={_fid: (NOMBRES_GRUPOS[_c],) for _fid, _c in clases_base.items()},
        plantilla_extra='<br>Criticidad base: %{customdata[3]}')
    _pl = capas['plantillas']
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['pred_clases'][_clase]], capas['clases'][_clase],
                         _pl['clases'][_clase])
            # NO vuelve a la leyenda: el mapa simulado usa la MISMA geometria KMeans
            # que el base, asi que sus cuatro clases ya estan nombradas alli. Repetirlas
            # agregaba un renglon a la leyenda horizontal que caia sobre los titulos de
            # la fila 3 -- medido, 22 px de solape.
            fig.data[IDX['pred_clases'][_clase]].showlegend = False
        # Negro: sin evento en la ventana, o fuera de la seleccion simulada. Un vano que
        # el simulador no puntuo no tiene clase, y la ausencia no es la clase mas baja.
        # Este mapa NO lleva halo de marcado: lo coloreado ES la seleccion, asi que un
        # halo encima no distinguiria nada que el color no diga ya.
        _volcar_capa(fig.data[IDX['pred_sin_dato']], capas['sin_dato'], _pl['sin_dato'])
        fig.data[IDX['pred_sin_dato']].name = 'Sin evento / no simulado'
        fig.data[IDX['pred_sin_dato']].line.color = COLOR_SIN_EVENTO
        fig.data[IDX['pred_sin_dato']].showlegend = False
        fig.data[IDX['pred_trafos']].lat, fig.data[IDX['pred_trafos']].lon = tr['lat'], tr['lon']
        fig.data[IDX['pred_trafos']].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
        fig.data[IDX['pred_switches']].lat, fig.data[IDX['pred_switches']].lon = sw['lat'], sw['lon']
        fig.data[IDX['pred_switches']].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        # El recuadro de este mapa dice QUE LE PASO al vano y no cual elegi -- eso ya lo
        # dice el de la izquierda, sobre el mismo vano. Verde si bajo de grupo, amarillo
        # si se quedo igual, rojo si subio; el amarillo es el mismo del mapa base porque
        # "no cambio" es justo el estado en que los dos mapas dicen lo mismo. Un vano
        # marcado sin celda en la ventana no recibe caja: la simulacion no lo puntuo, asi
        # que no tiene desenlace que pintar.
        # Marcados que la simulacion PUNTUO. No es lo mismo que los marcados: marcar un
        # vano mas despues de simular no lo mete en el resultado, y ni la caja ni el
        # encuadre pueden seguirlo hasta que se vuelva a presionar "Simular".
        _simulados = set(_ultimo_resultado_simulacion['FID_VANO'].astype(str))
        _marcados_simulados = [f for f in _marcados if f in _simulados]
        _cajas = cajas_por_cambio_de_grupo(
            geo, _ultimo_resultado_simulacion, marcados=_marcados_simulados,
            lado_minimo=LADO_MINIMO_CAJA, margen=MARGEN_CAJA)
        for _cambio, _i_capa in IDX_CAPA_CAMBIO.items():
            fig.layout.map2.layers[_i_capa].source = _cajas[_cambio]
        # Y se ACERCA a los vanos marcados. Los dos mapas dejan de compartir vista a
        # proposito: una vez simulado, la pregunta es que le paso a ESOS vanos, y buscarlos
        # otra vez dentro del circuito entero es trabajo que el tablero puede ahorrar. El
        # de la izquierda conserva el encuadre del circuito, que queda como la referencia.
        # Sin vanos marcados -- grano de circuito completo -- no hay sobre que acercarse y
        # se vuelve a ese mismo encuadre, en vez de quedarse en el de la seleccion anterior.
        _aplicar_vista('map2', centro_y_zoom(bounds_de_fids(geo, _marcados_simulados),
                                             alto_px=_alto_del_mapa_px())
                       or _vista_del_circuito(circuito))
        fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = ''


def _limpiar_resultado_simulacion(_change=None):
    """Circuito o ventana cambiaron: el ultimo resultado ya NO corresponde a la
    seleccion activa -- se descarta (fila 2 vuelve a "Aun no simulado" y el panel de
    importancia se vacia) en vez de mostrar la corrida de OTRA seleccion, que violaria
    la regla anti-confusion (D2)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada, _EPOCA
    _ultimo_resultado_simulacion = None
    _ultima_seleccion_simulada = None
    _EPOCA = siguiente_epoca(_EPOCA)  # invalida cualquier job en vuelo
    _redibujar_mapa_predicho()
    _pintar_grafo(None)
    _pintar_top_por_vano(TOP_VACIO)
    _pintar_barras_uiti(None)
    _pintar_costos(None)
    PLAN.value = _texto_del_plan(None)


def _pintar_rotulos_del_grafo(nodos):
    """Los nombres de los nodos, girados para seguir el radio de cada uno.

    Con los rotulos horizontales, los nombres de nodos vecinos se montaban unos sobre
    otros alrededor del anillo -- lo peor arriba y abajo, donde el circulo es mas plano y
    dos nodos consecutivos casi comparten altura. A lo largo del radio se abren en abanico
    con los propios nodos, asi que la separacion entre rotulos crece con la distancia al
    centro en vez de depender de donde caiga cada uno.

    Van como anotaciones y no como texto de la traza porque un `Scatter` no puede girar su
    texto: comprobado contra plotly 6.8.0, solo `Bar` y las anotaciones llevan `textangle`.

    El rotulo se planta un poco AFUERA del nodo (`RADIO_ROTULO_GRAFO`) para que el giro no
    lo haga cruzar por encima de su propio marcador.

    La reserva de anotaciones es fija: las que sobran quedan invisibles en vez de
    borrarse. Quitar anotaciones correria los indices de los avisos del grafo, de los
    costos y del mapa simulado, que se guardan por posicion.
    """
    for _k, _i_anotacion in enumerate(IDX_ANOTACIONES_NODOS):
        _anotacion = fig.layout.annotations[_i_anotacion]
        if _k >= len(nodos['texto']):
            _anotacion.visible = False
            continue
        _x, _y = nodos['x'][_k], nodos['y'][_k]
        _giro, _anclaje = rotacion_radial(_x, _y)
        _anotacion.x = _x * RADIO_ROTULO_GRAFO
        _anotacion.y = _y * RADIO_ROTULO_GRAFO
        _anotacion.text = nodos['texto'][_k]
        _anotacion.textangle = _giro
        _anotacion.xanchor = _anclaje
        _anotacion.visible = True


def _pintar_grafo(grafo):
    """Repaint puro del panel del grafo. Un grafo ANULADO no se dibuja a medias: se
    vacian las trazas y se dice por que. `estadistico_colapso` anula cuando las
    compuertas no varian entre vanos -- y su veredicto incluye `effective_rank <= 1`,
    que con menos de 3 vanos se cumple por construccion (la matriz centrada de 1 o 2
    filas tiene rango 1). Dibujar igual seria presentar el grafo experto FIJO como si
    lo hubiera estimado esta seleccion."""
    if grafo is None:
        trazas, mensaje = GRAFO_VACIO, 'Presiona "Simular" para ver que movio el grafo.'
    elif grafo['voided']:
        trazas = GRAFO_VACIO
        mensaje = (f'Grafo no estimable: las compuertas no varian entre los '
                   f'{grafo["n_vanos"]} vanos de la seleccion.<br>'
                   '<sup>Hacen falta al menos 3 vanos con comportamiento distinto.</sup>')
    elif not float(np.abs(grafo['matriz']).max()):
        # Todo en cero es un RESULTADO, no un panel vacio: la simulacion no movio una
        # sola relacion. Decirlo evita que se lea como que el grafo fallo.
        trazas = GRAFO_VACIO
        mensaje = ('La simulacion no movio ninguna relacion del grafo.<br>'
                   '<sup>Las variables aplicadas no cambian las compuertas de estos '
                   'vanos.</sup>')
    else:
        trazas, mensaje = trazas_grafo(grafo['matriz'], FEATURES_MIL), ''

    with fig.batch_update():
        fig.data[IDX['grafo_aristas']].x = trazas['aristas']['x']
        fig.data[IDX['grafo_aristas']].y = trazas['aristas']['y']
        _pesos = trazas['pesos']
        fig.data[IDX['grafo_pesos']].x = _pesos['x']
        fig.data[IDX['grafo_pesos']].y = _pesos['y']
        fig.data[IDX['grafo_pesos']].hovertext = _pesos['hovertext']
        # El tamano codifica el peso relativo DE ESTA seleccion: los pesos absolutos
        # cambian dos ordenes de magnitud entre ventanas y un tamano fijo por valor
        # dejaria el panel vacio o saturado segun cual se mire.
        _maximo = max(_pesos['peso'], default=0.0) or 1.0
        fig.data[IDX['grafo_pesos']].marker.size = [4 + 10 * (p / _maximo) for p in _pesos['peso']]
        fig.data[IDX['grafo_pesos']].marker.color = list(_pesos['peso'])
        # Un nodo por variable, con el color de su modo. El NOMBRE ya no va en la traza:
        # va como anotacion girada, mas abajo, porque un `Scatter` no puede girar texto.
        _nodos = trazas['nodos']
        for _i_traza, _modalidad in zip(IDX['grafo_nodos'], MODALIDADES_MIL):
            _cuales = [k for k, col in enumerate(_nodos['indice'])
                       if col in COLUMNAS_MODALIDAD[_modalidad]]
            _traza_nodo = fig.data[_i_traza]
            _traza_nodo.x = [_nodos['x'][k] for k in _cuales]
            _traza_nodo.y = [_nodos['y'][k] for k in _cuales]
            _traza_nodo.hovertext = [f'<b>{_nodos["texto"][k]}</b><br>Modo: {_modalidad}'
                                     for k in _cuales]
        _pintar_rotulos_del_grafo(_nodos)
        fig.layout.annotations[IDX_ANOTACION_GRAFO].text = mensaje


def _simular(epoca_job):
    """Computo pesado -- bloqueante dentro de la corutina (design section A: un job ya
    iniciado no se puede interrumpir). Mapa simulado, grafo e importancia, en ese orden y
    en el mismo job. Guarda y repinta SOLO si `epoca_job` sigue vigente al terminar
    (epoch guard)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
    circuito, ventana_i, marcados = _seleccion_actual()
    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=marcados)
    if seleccion['n_bolsas'] == 0:
        aplicar_si_vigente(
            lambda: setattr(STATUS, 'value',
                            'Sin bolsas (vano x ventana) para esta seleccion.'),
            epoca_job=epoca_job, epoca_actual=lambda: _EPOCA,
        )
        return

    # Cada columna de la rejilla es un vano, y cada vano lleva SUS valores a sus propias
    # instancias. Con la escritura global anterior el ultimo vano pisaba a todos los demas
    # y la simulacion contestaba por un escenario que nadie habia pedido.
    # Sin vanos marcados el grano es el circuito completo y hay una sola columna: ahi el
    # override vuelve a ser global, que es lo que corresponde a una pregunta sobre todo el
    # circuito.
    # `KNOBS` completo y no `KNOBS_PANEL`: el diccionario solo se usa para resolver
    # que features toca cada knob, y solo llegan aqui los que el panel ofrecio.
    por_vano = {
        fid: expand_knob_overrides(
            {knob_id: control.value for knob_id, control in controles.items()}, KNOBS)
        for fid, controles in _controles_por_vano.items()
    }
    global_ = por_vano.pop(GRANO_CIRCUITO, None)

    t0 = time.perf_counter()
    resultado, metadata = simular_bolsas(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL,
        overrides=global_ if global_ is not None else None,
        overrides_por_vano=por_vano or None,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )
    # El grafo del panel es |base - simulado|: cuanto MOVIO la simulacion cada relacion.
    # Antes se estimaba solo sobre las features observadas, y ese grafo es casi todo el
    # peso fijo del experto -- las compuertas solo lo reescalan --, asi que el antes y el
    # despues se ven iguales y el efecto de la intervencion no se aprecia.
    # Las features simuladas salen de la metadata y no se rearman aqui: repetir la
    # expansion de overrides es la forma segura de que el grafo acabe describiendo un
    # escenario distinto del que puntuo el mapa.
    _instancias = seleccion['instance_bag']
    gates_base = gates_de_bolsas(MIL, X_INST[seleccion['filas']], _instancias,
                                 seleccion['n_bolsas'])
    gates_simuladas = gates_de_bolsas(MIL, metadata['X_simulado'], _instancias,
                                      seleccion['n_bolsas'])
    grafo = grafo_diferencia(gates_base, gates_simuladas, MIL.model.edge_index,
                             n_features=len(FEATURES_MIL))
    # El top por vano reusa la MISMA seleccion de bolsas que acaba de puntuar el mapa: no
    # se vuelve a resolver, que era lo que hacia el cache por (circuito, ventana, marcados).
    top_por_vano = _calcular_top_por_vano(seleccion)
    # El plan comparte las bolsas ya resueltas y corre en el MISMO job: un plan que se
    # calculara aparte podria describir una seleccion distinta de la que muestra el mapa.
    plan = _calcular_plan(seleccion)
    # El costo sale de la rejilla, no del modelo: es aritmetica sobre la lista de precios
    # del contrato. Se calcula AQUI y no al marcar una casilla porque el tablero tiene un
    # solo disparador -- "Simular" -- y un costo que cambiara solo, mientras el mapa de
    # al lado sigue mostrando la corrida anterior, describiria dos planes a la vez.
    # Solo los vanos que la simulacion PUNTUO: costear un vano que el modelo no vio
    # pondria un precio al lado de un riesgo que nadie estimo.
    _puntuados = set(resultado['FID_VANO'].astype(str))
    costos = costos_de_intervencion(
        {fid: {nombre: int(control.value) for nombre, control in actividades.items()}
         for fid, actividades in _costos_por_vano.items() if fid in _puntuados},
        CATALOGO_COSTOS,
    )
    duracion = time.perf_counter() - t0

    def _escribir():
        global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
        _ultimo_resultado_simulacion = resultado
        _ultima_seleccion_simulada = (circuito, ventana_i)
        grano = f'{len(marcados)} vanos marcados' if marcados else 'todo el circuito'
        # DOS cuentas y no una. `variables_aplicadas` son las columnas del modelo que se
        # tocaron, y un solo control del panel abre varias: tres variables de escenario
        # llegan al modelo como veinticinco columnas. Publicar solo la segunda decia
        # "25 variables aplicadas" debajo de un panel con tres casillas marcadas, que se
        # lee como que la simulacion metio variables que nadie eligio.
        del_panel = len({k for c in _controles_por_vano.values() for k in c})
        columnas = len(metadata['variables_aplicadas'])
        aplicadas = (f'{del_panel} variables del panel' if del_panel == columnas else
                     f'{del_panel} variables del panel ({columnas} columnas del modelo)')
        cambian = int((resultado['delta_riesgo_ordinal'] != 0).sum())
        avisos = f' | {len(metadata["avisos"])} avisos' if metadata['avisos'] else ''
        costo = (f' | intervencion: {costos["total"]:,.0f} COP'
                 if costos['total'] else '')
        STATUS.value = (
            f'{duracion:.2f} s | MIL sobre {metadata["n_vanos"]} bolsas '
            f'({metadata["n_instancias"]:,} eventos) | '
            f'{aplicadas} | '
            f'{cambian} vanos cambian de clase | grafo sobre {grano}'
            f' | top {TOP_VARIABLES_POR_VANO} de {len(top_por_vano)} vanos'
            f'{costo}{avisos}'
        )
        _redibujar_mapa_predicho()
        _pintar_grafo(grafo)
        _pintar_top_por_vano(top_por_vano)
        _pintar_barras_uiti(resultado, ventana_i, circuito)
        _pintar_costos(costos)
        PLAN.value = _texto_del_plan(plan)

    aplicar_si_vigente(_escribir, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_simulacion(*_ignorado):
    global _EPOCA, _tarea_pendiente_simular
    if _tarea_pendiente_simular is not None and not _tarea_pendiente_simular.done():
        _tarea_pendiente_simular.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA
    STATUS.value = 'Simulando...'

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _simular(epoca_job)

    _tarea_pendiente_simular = asyncio.ensure_future(_tarea())


def _al_pedir_diagnostico(*_ignorado):
    """Calcula el diagnostico y deja MARCADOS los vanos que identifico.

    Marcarlos es parte de la respuesta, no un paso aparte: el diagnostico nombra diez
    vanos y sin marcarlos hay que buscarlos a mano en la lista de casillas y otra vez en
    el mapa, que es justo el trabajo que el boton venia a ahorrar. Al marcarlos, el mapa
    base los encierra en su recuadro y la rejilla les abre columna.

    Las VARIABLES no se tocan aqui. Que vanos mirar y que moverles son dos decisiones, y
    los dos botones de aplicar son los que responden la segunda.
    """
    global _ULTIMO_DIAGNOSTICO, _GRUPOS_APLICADOS
    DIAGNOSTICO.value = ('<span style="font-size:12px;color:#5b4a48;">'
                         'Calculando el diagnostico del circuito...</span>')
    _ULTIMO_DIAGNOSTICO = _diagnostico_del_circuito()
    DIAGNOSTICO.value = _texto_del_diagnostico(_ULTIMO_DIAGNOSTICO)
    # Un diagnostico nuevo empieza sin nada aplicado: los grupos que se hubieran aplicado
    # al anterior describian otros vanos.
    _GRUPOS_APLICADOS = []
    AVISO_APLICAR.value = ''
    if _ULTIMO_DIAGNOSTICO and _ULTIMO_DIAGNOSTICO['vanos']:
        vano_widget.value = tuple(
            f for f, _u, _n in _ULTIMO_DIAGNOSTICO['vanos'][:MAX_VANOS_ANALISIS])


def _aplicar_sugerencia(clave, nombre):
    """Marca los vanos del diagnostico y abre sus controles en el valor SUGERIDO.

    Es el puente entre el diagnostico y el simulador: sin el, leer "lleva NR_T a 116"
    para diez vanos obliga a marcarlos uno por uno y teclear cuarenta valores, y en ese
    trayecto se pierde justamente lo que el diagnostico acababa de calcular.

    El valor es el de CADA vano y no el promedio: el promedio ordena la lista, pero lo
    que baja a un vano concreto es su propio optimo, y aplicar el promedio simularia un
    escenario que no es el de ninguno.

    La seleccion de variables queda EXACTAMENTE en los grupos aplicados, y no se suma a
    lo que hubiera marcado antes. Es lo que hace que el resultado sea legible: si al
    aplicar intervencion quedaran ademas variables de escenario de una vuelta anterior,
    la simulacion mezclaria obra y clima y no se sabria cual de los dos movio el UITI.
    Para ver los dos efectos juntos se presionan los DOS botones -- el segundo conserva
    lo del primero --, que es una decision del usuario y no un residuo.
    """
    global _GRUPOS_APLICADOS
    if _ULTIMO_DIAGNOSTICO is None or not _ULTIMO_DIAGNOSTICO['vanos']:
        AVISO_APLICAR.value = ('<span style="font-size:12px;color:#b91c1c;">Primero '
                               'presiona <b>Diagnostico</b>.</span>')
        return
    diag = _ULTIMO_DIAGNOSTICO
    sugeridas = diag[clave]
    if not sugeridas:
        AVISO_APLICAR.value = (f'<span style="font-size:12px;color:#b91c1c;">El '
                               f'diagnostico no trae variables de {nombre}.</span>')
        return
    fids = [f for f, _u, _n in diag['vanos']][:MAX_VANOS_ANALISIS]
    if clave not in _GRUPOS_APLICADOS:
        _GRUPOS_APLICADOS.append(clave)
    # El ORDEN es el de los botones y no el de los clics, para que la rejilla no se
    # baraje segun por cual se empezo.
    ids_sugeridos = [e['knob_id'] for g in GRUPOS_SUGERIDOS if g in _GRUPOS_APLICADOS
                     for _lab, e in diag[g]]

    vano_widget.value = tuple(fids)
    knob_selector_widget.value = tuple(dict.fromkeys(ids_sugeridos))
    _reconstruir_controles_knob()

    # Y ahora el valor sugerido de cada vano, encima del valor actual con que abrio cada
    # control. Un valor fuera de los limites del deslizador se recorta: `FloatSlider`
    # lanza si cae fuera de [min, max], y tumbar el panel por un decimal no vale la pena.
    aplicados = 0
    for fid in fids:
        filas = {f['knob_id']: f['valor'] for f in diag['ranking'].get(fid, {}).get('filas', [])}
        for knob_id in ids_sugeridos:
            control = _controles_por_vano.get(fid, {}).get(knob_id)
            if control is None or knob_id not in filas:
                continue
            knob = _knobs_por_id[knob_id]
            valor = filas[knob_id]
            if knob.kind == 'numeric':
                lo, hi = knob.bounds
                control.value = float(min(max(float(valor), lo), hi))
            elif knob.kind == 'categorical' and valor in (knob.categories or ()):
                control.value = valor
            aplicados += 1
    _activos = ' + '.join(NOMBRES_SUGERIDOS[g] for g in GRUPOS_SUGERIDOS
                          if g in _GRUPOS_APLICADOS)
    AVISO_APLICAR.value = (
        f'<span style="font-size:12px;color:#15803d;">{len(fids)} vanos marcados y '
        f'{aplicados} controles abiertos en su valor sugerido. La simulacion va a usar '
        f'<b>solo variables de {_activos}</b>'
        + ('.' if len(_GRUPOS_APLICADOS) == len(GRUPOS_SUGERIDOS) else
           f'; presiona tambien el otro boton si quieres las dos mitades.')
        + ' Presiona <b>Simular</b> para ver el efecto.</span>')


boton_aplicar_intervencion.on_click(
    lambda _b: _aplicar_sugerencia('intervencion', 'Intervencion'))
boton_aplicar_escenario.on_click(lambda _b: _aplicar_sugerencia('escenario', 'Escenario'))


boton_diagnostico.on_click(_al_pedir_diagnostico)
# El diagnostico describe UN circuito en UNA ventana: al cambiar cualquiera de los dos
# deja de corresponder, y dejarlo en pantalla seria describir otra seleccion.
def _olvidar_diagnostico(_cambio=None):
    global _ULTIMO_DIAGNOSTICO, _GRUPOS_APLICADOS
    _ULTIMO_DIAGNOSTICO = None
    _GRUPOS_APLICADOS = []
    DIAGNOSTICO.value = _texto_del_diagnostico(None)
    AVISO_APLICAR.value = ''


circuito_widget.observe(_olvidar_diagnostico, names='value')
ventana_widget.observe(_olvidar_diagnostico, names='value')

boton_simular.on_click(_programar_simulacion)
circuito_widget.observe(_limpiar_resultado_simulacion, names='value')
ventana_widget.observe(_limpiar_resultado_simulacion, names='value')
vano_widget.observe(_redibujar_mapa_predicho, names='value')  # solo redibuja el halo marcado

_redibujar_mapa_predicho()  # primer dibujo: sin simulacion todavia -> "Aun no simulado"
_pintar_grafo(None)
_pintar_costos(None)
PLAN.value = _texto_del_plan(None)
_reconstruir_controles_knob()   # la rejilla arranca con su aviso, no vacia


## El tablero

Todo lo de arriba es preparacion; **lo que se usa es esto**. Al ejecutar la
celda siguiente aparece el tablero completo y ya no hace falta volver a subir.

**Por donde empezar**

1. Elige **circuito** y **ventana**. El deslizador arranca en la ventana mas
   reciente con eventos de ese circuito y solo recorre las que ese circuito
   tiene.
2. **Diagnostico** responde *por donde empiezo*: marca los vanos
   mas criticos de la ventana activa y lista que los bajaria de grupo.
3. **Aplicar intervencion** o **Aplicar escenario** abre los controles de esos
   vanos en el valor sugerido. Cada boton trae **solo su mitad**; presiona los
   dos si quieres ver las dos juntas.
4. **Simular** puntua la seleccion y llena el mapa de la derecha, las series,
   los violines y el costo.

Tambien se puede marcar vanos con las casillas o haciendo clic sobre el mapa
base, y mover a mano cualquier variable sin pasar por el diagnostico.

> El codigo de las celdas esta plegado. Para leerlo, despliega la celda desde
> el margen izquierdo; lo que hace cada pieza esta explicado en las celdas de
> texto de arriba y de abajo.


In [ ]:
# --- El panel, ARRIBA y del ancho de la figura (paridad 01.4) -----------------------
# Una sola columna, en el orden en que se usa: circuito -> ventana -> vanos -> variables
# del simulador -> el control de cada variable elegida -> "Simular" -> estado. Cada paso
# depende del anterior, asi que apilarlos evita el zigzag de un flex-wrap donde el boton
# podia quedar antes de los deslizadores que lo alimentan.
#
# El estilo va por CSS y no por `Layout` porque ipywidgets 8 no expone `background`,
# `box-sizing` ni `gap` como traits -- solo `border`, `padding`, `margin` y el flexbox
# basico. `add_class` es la via soportada para lo demas.
ESTILO = widgets.HTML('''
<style>
  .panel-v15 {
    box-sizing: border-box;
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b; font-size: 13px;
  }
  /* Cada grupo, un renglon completo: el panel es una columna, no una grilla. */
  .panel-v15 .grupo-v15 { margin: 0 0 10px 0; width: 100%; }
  .panel-v15 .titulo-v15 { font-weight: 600; margin-bottom: 2px; }
  /* La figura no lleva ancho fijo: sin esto se quedaria en el ancho intrinseco que
     plotly.js calcula al montar, en vez de ocupar la celda. */
  .app-v15, .app-v15 > .widget-vbox, .app-v15 .js-plotly-plot,
  .app-v15 .plot-container, .app-v15 .svg-container { width: 100% !important; }
  /* La lista compacta de 01.4: letra 12px, muchas casillas por renglon y scroll propio en
     vez de estirar el panel cuando el circuito tiene cientos de vanos.
     El ancho de cada casilla NO se toca aqui: viaja como estilo inline desde su `Layout`
     y le ganaria a esta hoja igual. */
  .lista-vanos, .lista-variables, .lista-items { font-size: 12px; }
  .lista-vanos .widget-checkbox label,
  .lista-variables .widget-checkbox label { white-space: nowrap; font-weight: 400; }
  /* Las actividades del contrato llegan a 143 caracteres y el ancho de la casilla es
     fijo: sin `ellipsis` el nombre se corta a la mitad de una palabra y dos actividades
     de la misma familia se vuelven indistinguibles. El `title` de la casilla lo lleva
     completo. */
  .lista-items .widget-checkbox label {
    white-space: nowrap; font-weight: 400;
    overflow: hidden; text-overflow: ellipsis;
  }
</style>''')


def _grupo(*hijos):
    """Un bloque del panel: su rotulo y sus controles juntos, como los `div` de 01.4."""
    caja = widgets.VBox(list(hijos), layout=widgets.Layout(align_items='flex-start',
                                                           width='100%'))
    caja.add_class('grupo-v15')
    return caja


def _titulo(texto):
    return widgets.HTML(f'<span class="titulo-v15">{texto}</span>')


vano_widget.caja.add_class('lista-vanos')
knob_selector_widget.caja.add_class('lista-variables')
item_selector_widget.caja.add_class('lista-items')

# El panel se alinea con el AREA DE DIBUJO de la figura, no con su borde. Los dos
# ocupan el mismo ancho, pero la figura reserva margen para los rotulos de los ejes de
# la primera columna, asi que sus paneles empiezan mas adentro que los controles y las
# dos cosas se leian corridas. El relleno se DERIVA del margen -- menos el ancho de los
# bordes del propio panel -- para que cambiar uno mueva al otro y no se desincronicen.
_BORDE_IZQ, _BORDE = 4, 1     # los mismos que declara `layout` mas abajo
_RELLENO_IZQ = max(int(fig.layout.margin.l) - _BORDE_IZQ, 0)
_RELLENO_DER = max(int(fig.layout.margin.r) - _BORDE, 0)

PANEL = widgets.VBox(
    [
        _grupo(_titulo('Circuito'), circuito_widget),
        _grupo(_titulo('Ventana'), ventana_widget),
        _grupo(vano_widget,
               widgets.HBox([boton_desmarcar]),
               widgets.HTML(f'<span style="font-size:12px;color:#5b4a48;">Hasta '
                            f'{MAX_VANOS_ANALISIS} vanos a la vez. Tambien puedes marcar y '
                            'desmarcar un vano haciendo clic sobre el en el mapa; sin '
                            'ninguno marcado el simulador toma el circuito completo.'
                            '</span>')),
        _grupo(_titulo('Variables del simulador'), knob_selector_widget,
               AVISO_BLOQUEADOS),
        # Las actividades van DESPUES de las variables y antes de la rejilla, en el
        # orden en que se usa el panel: primero que le muevo al vano, despues que obra
        # le hago, y la rejilla de abajo reune las dos por vano.
        _grupo(_titulo('Actividades del contrato (costo de intervencion)'),
               item_selector_widget,
               widgets.HTML('<span style="font-size:12px;color:#5b4a48;">Lo que marques '
                            'aqui aparece como una fila bajo CADA vano marcado, con su '
                            'costo unitario y cuantas veces se ejecuta. El costo no '
                            'alimenta al modelo: el simulador estima el riesgo y esta '
                            'lista lo que vale el plan; unirlos es tu decision.</span>'),
               AVISO_SIN_COSTO),
        _grupo(controles_knob_box),
        _grupo(widgets.HBox([boton_diagnostico]), DIAGNOSTICO,
               widgets.HBox([boton_aplicar_intervencion, boton_aplicar_escenario]),
               AVISO_APLICAR),
        _grupo(boton_simular),
        _grupo(STATUS),
        _grupo(PLAN),
    ],
    layout=widgets.Layout(
        width='100%', align_items='flex-start',
        padding=f'12px {_RELLENO_DER}px 12px {_RELLENO_IZQ}px', margin='0 0 6px 0',
        border=f'{_BORDE}px solid #e4c4c0',
        border_left=f'{_BORDE_IZQ}px solid rgb(203,24,29)',
    ),
)
PANEL.add_class('panel-v15')

# Un boton de encuadre por mapa, en una fila que los deja cada uno sobre el suyo: los dos
# mapas ocupan mitades iguales del ancho, asi que dos cajas al 50% ponen cada boton donde
# empieza su mapa. Son widgets y no botones de plotly (`updatemenus`) porque un
# `updatemenu` lleva argumentos FIJOS, calculados al dibujar: entre el dibujo y el clic
# pueden haber cambiado los vanos marcados, y el boton llevaria a donde estaba la
# seleccion antes. Aqui la vista se calcula en el clic.
def _boton_encuadre(nombre_mapa, etiqueta):
    boton = widgets.Button(description=etiqueta, layout=widgets.Layout(width='260px'),
                           tooltip='Centra sobre los vanos marcados, o sobre el circuito '
                                   'si no hay ninguno')
    boton.on_click(lambda _b: _centrar_mapa(nombre_mapa))
    return widgets.Box([boton], layout=widgets.Layout(width='50%'))


ENCUADRES = widgets.HBox(
    [_boton_encuadre('map', 'Centrar mapa base'),
     _boton_encuadre('map2', 'Centrar mapa simulado')],
    layout=widgets.Layout(width='100%',
                          padding=f'0 {_RELLENO_DER}px 0 {_RELLENO_IZQ}px'))

APP = widgets.VBox([ESTILO, PANEL, ENCUADRES, fig], layout=widgets.Layout(width='100%'))
APP.add_class('app-v15')
# `display` explicito y una sola vez. `add_class` devuelve el propio widget, asi que
# dejarlo como ultima expresion de la celda hacia que Jupyter lo auto-mostrara ADEMAS del
# display de la celda siguiente: el tablero aparecia dos veces.
display(APP)


## Como leerlo

**Los dos mapas.** A la izquierda, la criticidad historica; a la derecha, la simulada.
Van lado a lado y no apilados: la unica comparacion que justifica que haya dos mapas es la
del mismo vano antes y despues de simular, y apilados obligaba a mover la vista de arriba
a abajo para hacerla. Nunca comparten leyenda ni titulo, porque mezclarlos invita a leer
una prediccion como un hecho observado.

Arrancan con el mismo encuadre, pero **al simular el mapa de la derecha se acerca a los
vanos marcados**. Se paga a sabiendas que los dos dejen de mirar la misma geografia: una
vez que el modelo corrio, la pregunta ya no es donde queda el circuito sino que le paso a
ESOS vanos, y buscarlos otra vez dentro del garabato completo es trabajo que el tablero
puede ahorrar. El de la izquierda conserva SIEMPRE el encuadre del circuito, que queda
como la referencia contra la cual se lee el acercamiento; y desmarcar todo devuelve al de
la derecha a esa misma vista.

**Una sola leyenda, horizontal y debajo de los mapas.** Vertical y a la derecha se
llevaba 196 px medidos de ancho -- una columna entera de la fila 3 -- para decir siete
nombres. Y es UNA sola para los dos mapas: usan la misma geometria KMeans de 01.4, asi
que las cuatro clases significan exactamente lo mismo en los dos y repetirlas era decir
dos veces la misma escala.

**El rango de un control sale de valores reales.** `ALTURA` usa 99 como codigo de "sin
dato" -- 327 registros, y el siguiente valor real es 25; un poste de 99 m no existe en
una red de distribucion. Se excluye del rango y del valor inicial, asi que el deslizador
va de 4 a 25 y no de 4 a 99, donde 74 de sus 95 puntos de recorrido caian en un tramo
que ningun vano puede ocupar. Se declara variable por variable en
`vano_controls.VALORES_NO_VALIDOS`: una regla automatica del tipo "los nueves son
relleno" tumbaria el 9 de `LONG_CRUCETA` y el 10 de `VAL_CRIT_APOYO`, que son reales.

**La tabla de variables** dice, de cada control, su rango real y si simularlo significa
algo. No todas las variables del modelo son palancas: unas se mueven con una cuadrilla (la
poda, la puesta a tierra, el conductor), otras son escenarios que nadie controla pero que
son justamente el what-if (el clima, las descargas, el crecimiento de la demanda), otras
describen lo que el vano ES y no algo que se le pueda hacer, y una -- los trafos afectados
EN LA FALLA -- se mide despues del evento que el modelo intenta anticipar, asi que
simularla es circular. El veredicto y su motivo salen del diccionario del propio proyecto
y viven en `simulador_variables.JUICIO_SIMULACION`, con sus pruebas.

La tabla trae la **unidad de medida cuando aplica**: un rango sin unidad no se puede
juzgar -- 25 puede ser una altura razonable o un disparate segun si son metros o pies.
Quedan sin unidad las categoricas, las binarias, los indices como `NR_T` y
`VAL_CRIT_APOYO` -- que son puntajes y no magnitudes -- y `DDT`, cuya descripcion
implica una unidad por area pero no dice cual; antes que estampar una equivocada, la
celda queda vacia.

**Las variables del simulador van en cuatro columnas**: dos para lo que se puede hacer
-- intervencion -- y dos para lo que se quiere anticipar -- escenario. Una lista corrida
de dieciocho casillas obliga a recordar el veredicto de cada una para saber a cual de
las dos preguntas pertenece; en columnas eso lo dice la posicion.

**El panel se alinea con el AREA DE DIBUJO de la figura, no con su borde.** Los dos
ocupan el mismo ancho, pero la figura reserva margen a la izquierda para los rotulos de
los ejes de la primera columna, asi que sus paneles empiezan mas adentro que los
controles. El relleno del panel se deriva de ese margen -- no se escribe a mano -- para
que cambiar uno mueva al otro.

**Las variables refutadas y las de lectura unica ya no aparecen en el panel.** Mientras estuvieran ahi, el
tablero las ofrecia como equivalentes a la poda o a la puesta a tierra. Las "Limitado"
salen por el mismo motivo: hay UNA lectura bajo la cual se interpretan -- adelantar una
fecha equivale a reponer el activo -- y un deslizador no puede transmitir esa condicion,
quien lo mueve ve el numero y no el motivo. Quitarlas NO las
saca de la simulacion: un override solo se escribe si se fija, asi que entran al modelo
con el valor OBSERVADO de cada vano, que es exactamente lo que corresponde -- lo unico
que se pierde es poder moverlas. El panel las nombra debajo de la lista de variables, o
acortarse sin explicacion se leeria como que faltan.

**Negro = sin evento**, en los dos mapas. Un vano sin eventos en la ventana no tiene clase,
y la ausencia no es el grupo mas bajo. En el mapa simulado el negro cubre ademas lo que
quedo fuera de la seleccion.

**Se analizan hasta 5 vanos a la vez.** Cada vano marcado recibe su propia COLUMNA de
controles abajo del panel, rotulada con el vano que gobierna, y cada control abre en el
valor ACTUAL de esa variable para ESE vano en la ventana activa -- la mediana de sus
instancias, o la moda si la variable es categorica. Eso es lo que permite preguntar "que
pasa si podo SOLO este": los demas vanos quedan exactamente como estaban, no en un valor
por defecto. Al llegar a los cinco las casillas restantes se deshabilitan solas, y el clic
en el mapa respeta el mismo tope. Sin ningun vano marcado el grano vuelve a ser el circuito
completo y hay una sola columna, cuyos valores se aplican a todo. Un vano marcado SIN
eventos en la ventana activa no recibe columna -- no hay valor actual desde donde arrancar
-- y el pie de la rejilla lo nombra en vez de dejarlo desaparecer en silencio.

**"Diagnostico"** contesta la pregunta con la que se abre una jornada -- por
donde empiezo aqui -- que ninguna otra vista del tablero contesta: el mapa exige mirar
tramo a tramo y el panel exige haber elegido ya los vanos. Lista los **10 vanos de mayor
UITI** del circuito en la ventana activa y, sobre esos diez, las **5 variables de
intervencion** y las **3 de escenario** que mas los bajarian.

Tiene disparador propio y no viaja con "Simular": contesta otra pregunta, no depende de lo
que este marcado ni de las variables fijadas, y colgarlo del boton obligaria a
recalcularlo en cada escenario que no lo cambia. Se borra al cambiar de circuito o de
ventana, porque describe UNO de cada uno.

El ranking se **agrega sobre los diez** y no se da vano por vano: la pregunta es que obra
programar para el conjunto, y diez rankings sueltos son diez decisiones. Las dos mitades
van separadas y de tamanios distintos a proposito -- lo que se HACE es lo que se cotiza, y
lo que se ANTICIPA sirve para saber bajo que condiciones esa obra rinde. Mezclarlas
dejaria al clima copando la lista, como ya se midio en el panel.

**Las dos columnas comparten escala, y conviene compararlas.** Medido sobre los diez
peores vanos de un circuito real: la intervencion promedia entre 0,016 y 0,045 ordenes de
magnitud, y el escenario entre 1,39 y 1,75 -- unas cuarenta veces mas. Lo que dice el
modelo ahi es que en esos vanos **manda el clima y la obra rinde poco**. No invalida la
obra, pero si la expectativa de cuanto va a bajar el riesgo, y es mejor saberlo antes de
costearla que despues.

**Los dos botones de encuadre**, uno sobre cada mapa, devuelven la vista a los vanos
marcados -- o al circuito si no hay ninguno. Existen porque las dos vistas se van de sitio
por caminos legitimos: haciendo zoom para mirar un tramo, o porque el mapa simulado se
acerca solo a los vanos que puntuo. La vista se calcula **en el clic** y no al dibujar:
entre un dibujo y el clic pueden haber cambiado los vanos marcados, y un encuadre
precalculado llevaria a donde estaba la seleccion antes.

**El deslizador de ventana solo recorre las ventanas que ese circuito tiene.** No son las
once para todos: un circuito tranquilo puede no registrar una sola celda en media ventana
del anio, y antes el deslizador lo llevaba igual hasta ahi -- a un mapa sin un solo tramo de
color, que se lee como que el tablero se rompio y no como que no hubo eventos. Al cambiar
de circuito se conserva la ventana vigente si el circuito nuevo tambien la tiene: moverse de
circuito no deberia cambiar el mes que se esta mirando. Si no la tiene, cae en la primera
que si.

**Marcar un vano** se hace con su casilla o tocandolo en el mapa de la izquierda: las dos
vias son el mismo estado, porque el clic alterna la casilla. Solo el mapa base acepta clic;
el de la derecha es la salida del modelo, no un control, y marcar desde ahi mezclaria "lo
que elegi" con "lo que el modelo predijo" sobre la misma superficie. Un vano marcado se dibuja con el
color de SU clase sobre un halo blanco -- no con un color plano de "seleccionado", que
congelaria lo que se ve cuando la ventana cambia la clase por debajo.

**La caja amarilla** encierra cada vano marcado: es su rectangulo envolvente, translucido,
y esta para responder *cual estoy estudiando* en un circuito de cientos de tramos, donde
un trazo un poco mas ancho ya no basta. Va DEBAJO de las lineas del mapa, asi que no tapa
el color de clase del vano ni se come el clic. Sale de la geometria y no de los datos de
la ventana: **sigue puesta al mover el deslizador de ventana**, incluso sobre un vano que
en esa ventana no registro un solo evento. Se apaga de una sola forma -- desmarcando el
vano, con su casilla o volviendo a tocarlo en el mapa.

**En el mapa simulado el mismo recuadro cambia de pregunta**: alli el vano ya esta
identificado a la izquierda, asi que el color dice QUE LE PASO. Verde claro si bajo de
grupo de criticidad, amarillo si se quedo en el mismo, rojo si subio. El amarillo es
exactamente el del mapa base porque "no cambio" es justo el estado en que los dos mapas
dicen lo mismo. Son tres capas y no una porque una capa del mapa pinta con un solo color.
Un vano marcado que la simulacion no puntuo -- sin celda en la ventana activa, o marcado
DESPUES de presionar "Simular" -- no recibe recuadro: no tiene grupo base ni simulado, y
pintarlo de amarillo afirmaria que no cambio, que es justo lo que nadie midio.

**La etiqueta del mapa simulado trae los DOS grupos**, el base y el simulado, sobre el
mismo vano. Sin eso, saber si un vano mejoro obliga a cruzar al mapa de al lado y
acordarse del color. El grupo base viaja por punto y no en la plantilla de la traza porque
dentro de una traza -- que es una clase SIMULADA -- el grupo base cambia de vano a vano.

**"Simular" es el unico disparador** y produce las tres salidas en el mismo trabajo,
aplicando solo las variables elegidas en el panel. Cada variable aparece como un control
-- deslizador si es numerica, lista si es categorica -- y una familia climatica
(precipitacion, temperatura, rafaga y viento) mueve sus 12 rezagos horarios de una vez.
El mapa simulado **no existe hasta que se presiona**: antes solo muestra el aviso, porque
un mapa pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma forma que un
resultado.

**Cambiar circuito o ventana descarta la ultima simulacion.** La fila 2, el grafo y la
importancia se vacian: mostrar la corrida de otra seleccion es la misma confusion que
todo lo anterior evita.

**El "Grafo de relevancia"** (fila 4, columnas 1-2) es el grafo experto tal como lo usa la
seleccion: `media_vanos(compuerta) x peso_fijo` por arista, en disposicion circular. Cada
nodo lleva su variable y el color de su modo -- `climaticos` o `estructurales` --, paleta
deliberadamente ajena a la de los grupos KMeans. Se **anula** cuando las compuertas no
varian entre los vanos, lo que incluye por construccion cualquier seleccion de menos de 3
vanos: dibujarlo igual seria presentar el grafo experto fijo como si lo hubiera estimado
esta seleccion.

**"UITI acumulado y eventos por ventana"** (fila 3, columnas 1-2) es la serie de tiempo de
los vanos elegidos, y **solo aparece al elegirlos**: sin ninguno marcado queda vacia a
proposito, por el mismo motivo que los violines de 01.4 -- una serie sobre el circuito
entero y una sobre tres vanos se dibujan igual y no miden lo mismo, asi que caer al
circuito cambiaria el sujeto del panel en silencio.

Conviven DOS codigos de color sobre el mismo punto, separados por canal para que los dos
se lean a la vez: la **linea y el anillo** llevan el color de identidad del vano -- dicen
de QUE vano es la serie -- y el **relleno del punto** lleva el color del grupo de riesgo
en que cayo ese vano en ESA ventana, con la paleta del mapa. Por eso un mismo vano cambia
de relleno a lo largo de su serie: medido, uno pasa de Bajo a Medio-Alto y vuelve a Bajo
en cuatro ventanas. Un relleno gris es una ventana sin celda, que no tiene grupo --
distinto de caer en el mas bajo.

El eje x lleva la **fecha en que empieza cada ventana**, no su etiqueta `V1`, `V2`: un
rotulo `V7` obliga a ir a buscar a que periodo corresponde cada vez que se mira el panel.
Cada celda (vano, ventana) es UNA fila ya agregada, asi que cada punto tiene exactamente
una clase: a este grano no hay un conjunto de etiquetas del que tomar la moda.
Doble eje porque UITI y eventos viven en escalas muy distintas. Una ventana sin celda va
como hueco y no en cero -- no es "no hubo UITI", es "no hubo medicion" -- y la linea se
corta ahi. El punto de la ventana vigente se dibuja al triple y viaja con el deslizador.

**"Top 10: que baja el UITI de cada vano"** (fila 3, columnas 3-4) es un grupo de barras
por vano. Responde la pregunta que sostiene una orden de trabajo: **que muevo, y a que
valor, para que ESTE vano baje**. No mide sensibilidad, mide **caida alcanzable**.

El barrido anterior ordenaba por `max(|delta-|, |delta+|)`, y eso tenia dos defectos para
esta pregunta. Primero, la magnitud **no llevaba signo**: una variable que dispara el
riesgo en los dos extremos encabezaba el ranking, y la cabeza de la lista se llenaba de
palancas que no hay que tocar. Segundo, solo miraba los **dos extremos**, cuando --medido
sobre este modelo-- 10 de los 15 controles numericos tienen su mejor valor en el INTERIOR
del rango para alguna bolsa (`DDT` para todas). El modelo es marcadamente no monotono, asi
que los extremos son, simplemente, los dos puntos equivocados. El efecto se ve: sobre un
vano real los dos rankings no comparten **ni una** de sus cinco primeras variables.

Lo que corre ahora recorre cada control en una rejilla de nueve valores, guarda el que
**minimiza** el u-hat de la bolsa, y ordena por cuanto lo baja. La altura de la barra son
**ordenes de magnitud de UITI**, el mismo eje que usa la geometria KMeans; en unidades, el
ranking de un vano caro seria incomparable con el de uno barato. Cada barra trae en el
hover **el valor que lo consigue**, asi que la lista se lee como una instruccion --
"lleva ALTURA a 25 m" -- y no como un puntaje.

Y dice si eso **cambia de grupo**. Con `n_obs` fijo hay un UITI por debajo del cual la
bolsa cae en el grupo Bajo, y esa meta va en el hover. Una barra **verde** es una variable
que, ella sola, lleva al vano hasta ahi -- el mismo verde del recuadro del mapa simulado, y
significa lo mismo. Cuando **ninguna** lo logra, el panel no lo disimula: medido, a un vano
de Medio-Alto con UITI 271 la mejor variable sola lo deja en 57, y hacen falta varias. Y un
vano que **ya esta** en Bajo no pinta nada de verde: no hay adonde bajar, y darlo por
alcanzado presentaria diez variables inertes como palancas decisivas.

Tampoco es SHAP, y ahora por un motivo mas fuerte que antes: SHAP **atribuye** el UITI que
ya hay a las variables que lo explican, y aqui la pregunta es la contraria -- cual, y en que
valor, lo **baja**. Una atribucion alta puede corresponder a una variable que no se puede
mover en la direccion util, y su linea base es una distribucion de datos, no una
intervencion.

El ranking recorre **todos** los controles que el panel ofrece, tambien los
**categoricos** -- su rejilla son sus categorias. El barrido anterior los saltaba, y con
ellos se caian de la lista el conductor, el calibre del neutro y el tipo de proteccion:
tres obras que CHEC efectivamente ejecuta. Y el top **reserva sitio para los dos grupos**,
intervencion y escenario: sin la reserva, las cuatro familias climaticas copan la lista y
no queda ni una palanca que una cuadrilla pueda ejecutar.

**El plan combinado**, debajo del panel, es la continuacion necesaria del ranking. Medido
sobre 59 bolsas de 40 circuitos: en **Medio**, 20 de 33 vanos alcanzan el grupo Bajo con
UNA sola variable; en **Medio-Alto, 0 de 18**; en **Alto, 0 de 8**. Para los dos grupos
donde la pregunta de mantenimiento pesa mas, una variable sola NO basta nunca, asi que el
tablero encadena hasta cuatro cambios por vano -- descenso goloso, cada control como mucho
una vez -- y dice si con eso llega o en que grupo se queda. Es un buen plan, **no el
minimo demostrable**: con 18 controles y 9 valores, dos cambios simultaneos ya son 13 mil
combinaciones y cuatro son 26 millones, fuera del presupuesto de un boton.

Cuesta `1 + 9 x controles` pasadas el ranking, y hasta `4 x 9 x controles` mas el plan,
todo para TODA la seleccion y no una tanda por vano. Medido de punta a punta -- mapa,
grafo, ranking, plan y costos sobre cuatro vanos: **1,3 s**. Pasar de cinco a diez
variables mostradas no cuesta ninguna pasada mas: ya estan todas puntuadas y el top solo
decide cuantas se dibujan.

El rotulo dentro de la barra se elige **barra por barra segun lo que mida**: el nombre
resumido si cabe, sus iniciales si no, y nada antes que un texto cortado que se monte
sobre la barra vecina. Plotly no sabe hacer esa cascada -- o escribe el texto entero o lo
esconde --, asi que la decide el cuaderno con el largo de cada barra en pixeles. **El
nombre completo esta siempre en la etiqueta del mouse**, que es donde se resuelve la duda.
El color de la barra codifica la POSICION en el ranking y nada mas: es una rampa de un
solo color, deliberadamente ajena a la paleta de los grupos, para que una barra no se lea
como un grupo de criticidad.

**"Costo de la intervencion"** (fila 5, a todo lo ancho) es la conclusion del tablero: una
barra por vano con lo que vale su plan, y la barra **TOTAL** con la orden de trabajo
completa. El total comparte eje con los vanos aunque sea siempre el mas alto -- es su
suma, y darle un eje propio dejaria de mostrar cuanto pesa cada vano dentro de el --, asi
que se distingue por color y no por escala. El **desglose por actividad va en el hover**,
ordenado de mayor a menor: la primera linea es la que hay que negociar.

Las actividades salen del contrato de CHEC. Se marcan **una sola vez arriba** y aparecen
como una fila bajo **cada** vano marcado, con su costo unitario y un desplegable de **0 a
5** para decir cuantas veces se ejecutan sobre ESE vano. Una sola lista compartida y no
una por vano: repetir el catalogo cinco veces serian 625 casillas para elegir tres.

El **cero** es lo que hace que esa lista compartida no imponga la misma obra a todos. La
casilla de arriba elige que actividades entran al PLAN; el cero de cada fila dice en cuales
de los vanos marcados esa actividad no se ejecuta. Asi se expresa "podar solo este": se
marca la poda una vez y se pone en cero donde no va. Una actividad en cero no aparece en el
desglose del hover -- no cuesta nada y listarla llenaria el detalle de renglones vacios --,
pero un vano con TODO en cero sigue teniendo su barra: decir "a este no le hago nada" es
una respuesta, distinta de un vano que nunca se marco.

Tres detalles del libro de precios que el cuaderno resuelve y conviene saber. Es una
exportacion de **tabla dinamica**, asi que su ultima fila es el pie `Total general`:
ofrecida como actividad se veria igual que las demas y agregaria 254 mil pesos de puro
artefacto. **Doce actividades no traen costo unitario** y no se ofrecen -- no se puede
costear lo que no tiene precio --, pero el panel las nombra en vez de dejarlas
desaparecer. Y como el archivo **no trae codigo de item**, el nombre es la clave: uno
llegaba ilegible (`CONDUCCIÃ“N`, UTF-8 leido como cp1252) y se repara al cargarlo, porque
una clave que nadie puede leer es una actividad que nadie va a marcar.

El costo se calcula **al presionar "Simular"**, con todo lo demas, y solo sobre los vanos
que el modelo puntuo. Recalcularlo al marcar una casilla dejaria el costo de un plan al
lado del mapa de otro; costear un vano que la simulacion no vio pondria un precio junto a
un riesgo que nadie estimo.

**"UITI de la seleccion: base contra simulado"** (fila 4, columnas 3-4) mide la misma
cantidad dos veces sobre los mismos vanos. Es la lectura que cierra el tablero: si la
simulacion no mueve la distribucion, no la movio, y eso se ve sin comparar dos mapas tramo
a tramo. Con pocos vanos un violin es casi una linea, por eso van tambien los puntos: con
tres o cuatro datos lo honesto es mostrarlos, no dibujar una densidad que no existe.


## La matematica de lo que hace "Simular"

Todo lo de abajo describe UNA pulsacion del boton sobre la seleccion activa
$(c, w, M)$: circuito, ventana y conjunto de vanos marcados.

### 1. Las bolsas de la seleccion

La unidad de prediccion no es el evento: es la **bolsa**, la celda
$(\text{circuito}, \text{vano}, \text{ventana})$. Es la misma unidad en la que 04 define
la criticidad, y por eso el mapa simulado se puede comparar con el historico.

$$\mathcal{B}(c,w,M)=\{\,b=(c,v,w)\;:\;v\in M\,\},\qquad M=\varnothing\;\Rightarrow\;M:=V(c,w)$$

donde $V(c,w)$ son todos los vanos del circuito con al menos un evento en esa ventana:
sin vanos marcados el grano es el circuito completo, no un panel vacio.

Cada bolsa $b$ agrupa sus instancias $I_b$ -- las filas de evento de ese vano dentro de
esa ventana -- y trae dos cosas que **no se predicen nunca**:

$$n_b=|I_b|\quad(\text{eventos OBSERVADOS}),\qquad x_i\in\mathbb{R}^{p},\;p=80$$

Las $p=80$ columnas son 22 estructurales + 48 rezagos de clima + `COD_CAUSA` y sus 9
indicadores. Conviene no confundir esa cuenta con la **particion por modalidades** que usa
el modelo, que no es la misma: `climaticos` son las 50 columnas de los 48 rezagos **mas
`DDT` y `NR_T`** (descargas y nivel de tormenta son clima, aunque viajen como columnas
estaticas), y `estructurales` son las 30 restantes -- las otras 20 estructurales mas
`COD_CAUSA` y sus 9 indicadores. El almacenamiento es CSR (`offsets`, `counts`), no una matriz rellenada:
el 52,7% de las bolsas son de un solo evento y el maximo es 46, asi que rellenar
desperdiciaria mas de 40x en la mitad de los datos. Al seleccionar, el indice de bolsa se
**renumera** desde 0, porque el modelo toma `n_bags = max(instance_bag)+1` y los ids
originales reservarian una bolsa vacia por cada celda no seleccionada.

**Los controles del simulador** actuan sobre las instancias, no sobre las bolsas. Un
control $\kappa$ gobierna un conjunto de columnas $F(\kappa)$ -- una sola para una
variable estructural, las 12 de una familia climatica -- y aplicarlo es

$$x_{i,j}\;\leftarrow\;\phi_j(\text{valor}),\qquad \forall\, i\in\textstyle\bigcup_b I_b,\;\forall\, j\in F(\kappa)$$

con $\phi_j$ la coercion a espacio de modelo (categoria por su codificador, fecha a
epoch, NaN a su centinela). No hay escalador despues: la matriz de instancias del MIL es
espacio crudo. **$n_b$ jamas se toca**: es un eje del espacio que define la clase, y
moverlo desplazaria al vano por una dimension que el modelo no predice.

### 2. La prediccion de clase de cada vano

El modelo hace **dos pasadas** sobre el mismo codificador. La primera existe solo para
producir las compuertas del grafo.

**(a) Codificacion y atencion.** Cada instancia se codifica por modalidad
(30 estructurales, 50 climaticas, segun la particion de arriba) y se concatena en
$z_i^{(1)}$. La bolsa se resume con
atencion tipo Ilse, normalizada dentro de la bolsa:

$$e_i=\mathbf{w}^{\top}\tanh(V z_i^{(1)}),\qquad
a_i=\frac{\exp(e_i)}{\sum_{i'\in I_b}\exp(e_{i'})},\qquad
z_b^{(1)}=\sum_{i\in I_b}a_i\,z_i^{(1)}$$

Esto es **invariante a la cardinalidad por construccion**: duplicar cada instancia de una
bolsa no cambia ningun $e_i$, el denominador se duplica, cada copia recibe $a_i/2$ y la
suma queda igual.

**(b) Compuertas del grafo experto.** Un decodificador lee el resumen de la bolsa y
produce una compuerta por arista:

$$g_b=2\,\sigma(W_g\,z_b^{(1)})\;\in\;(0,2)^{E},\qquad E=64$$

Se inicializa en cero, de modo que $g_b=\mathbf{1}$ al arrancar y el grafo aprendido
**parte exactamente del grafo experto fijo**.

**(c) Propagacion.** El grafo experto es una adyacencia fija $W\in\mathbb{R}^{80\times 80}$
con soporte en $E$ aristas. Cada instancia recibe, en la columna destino de cada arista:

$$x'_{i,j}\;=\;x_{i,j}\;+\;\alpha\!\!\sum_{e\,:\,\mathrm{dst}(e)=j}\!\! g_{b(i),e}\;w_e\;x_{i,\mathrm{src}(e)},
\qquad \alpha=0{,}2$$

Una columna que no es destino de ninguna arista queda intacta, exactamente.

**(d) Segunda pasada y fusion FiLM.** El MISMO codificador procesa $x'$, se vuelve a
agrupar con la MISMA atencion y las dos modalidades se fusionan modulando la
estructural con la climatica (`film_modulated_modality = estructurales`, del artefacto):

$$\hat z_b=z_b^{\text{est}}\odot\bigl(1+\gamma(z_b^{\text{clim}})\bigr)+\beta(z_b^{\text{clim}}),
\qquad p_b=h(\hat z_b),\qquad \hat u_b=\mathrm{expm1}(p_b)$$

La concatenacion es aditiva entre modalidades y no puede representar un producto entre una
variable estructural y una climatica; FiLM hace que el clima **reescale** lo estructural,
que es tambien la afirmacion de dominio: una rafaga pesa mas sobre un apoyo alto, viejo y
degradado. El modelo aprende en $\log(1+u)$ y se devuelve a UITI con `expm1`.

**(e) Clase.** Con el UITI predicho y los eventos observados, la clase sale de la
geometria KMeans de 04 -- **la misma que pinta el mapa base**, verificada al cargar el
modelo. En el espacio canonico `2` el eje de eventos es lineal y el de UITI logaritmico:

$$\zeta_b=\left(\frac{n_b-\mu_0}{s_0},\;\frac{\log_{10}\max(\hat u_b,\varepsilon)-\mu_1}{s_1}\right),
\qquad \hat k_b=\arg\min_{k\in\{0,1,2,3\}}\lVert \zeta_b-c_k\rVert^2$$

El mapa pinta $\hat k_b$ con la paleta de los cuatro grupos; lo que no tiene bolsa en la
ventana, o quedo fuera de la seleccion, va en negro. El simulador corre esto **dos veces**
-- sin y con los controles aplicados -- y $\Delta_b=\hat k_b^{\text{sim}}-\hat k_b^{\text{base}}$
es cuantos vanos cambian de clase.

### 3. Que variables bajan el UITI de cada vano

**No es SHAP**, y no por costumbre: SHAP *atribuye* el $\hat u$ que ya hay a las variables
que lo explican, y la pregunta del panel es la contraria -- cual, y en que valor, lo
**baja**. Tampoco es ya un barrido min-max; por que dejo de serlo esta al final.

**La meta.** Con $n_b$ fijo -- nunca se simula --, la clase solo depende de $\hat u$, asi
que existe un umbral por debajo del cual la bolsa cae en el grupo mas bajo:

$$u^{\star}(n_b)=\max\{\,u>0 \;:\; \arg\min_k\lVert\zeta(n_b,u)-c_k\rVert = 0\,\}$$

Se resuelve por rejilla y no por biseccion: nada garantiza que al subir $u$ con $n_b$ fijo
se recorran los grupos en orden, y una biseccion asume esa monotonia. Medido sobre la
geometria real, $u^{\star}$ se desploma cuando se acumulan eventos -- **4,41** con un
evento, **0,0029** con cuarenta y seis --, asi que un vano con muchos eventos necesita un
UITI casi nulo para bajar de grupo. Es una propiedad del espacio, no del panel.

**El barrido.** Para cada control **numerico** $\kappa$ se recorre una rejilla de $G=9$
valores sobre su rango observado, moviendo sus columnas $F(\kappa)$ a la vez, y se guarda
el valor que MINIMIZA el UITI de cada bolsa:

$$v^{\star}_{b,\kappa}=\arg\min_{v\in\mathcal{G}_\kappa}\hat u_b\!\left(X^{\kappa\to v}\right),
\qquad
\hat u^{\star}_{b,\kappa}=\min_{v\in\mathcal{G}_\kappa}\hat u_b\!\left(X^{\kappa\to v}\right)$$

y se ordena por la **caida en ordenes de magnitud**, que es el eje que usa la geometria:

$$c_{b,\kappa}=\log_{10}\hat u_b(X)-\log_{10}\hat u^{\star}_{b,\kappa},
\qquad
\text{avance}_{b,\kappa}=\frac{c_{b,\kappa}}{\log_{10}\hat u_b(X)-\log_{10}u^{\star}(n_b)}$$

En unidades de UITI el ranking de un vano caro seria incomparable con el de uno barato; en
ordenes de magnitud, dos barras de la misma altura significan lo mismo en cualquier grupo.
El avance dice que fraccion del camino al grupo Bajo cubre esa sola variable, y
$\hat u^{\star}_{b,\kappa}\le u^{\star}(n_b)$ es la condicion que pinta la barra de verde.

**Por que dejo de ser min-max.** Dos defectos, los dos medidos:

- $s=\max(|\Delta^-|,|\Delta^+|)$ **no lleva signo**. Una variable que dispara el riesgo
  en los dos extremos encabezaba el ranking. Sobre un vano real, los dos rankings no
  comparten **ni una** de sus cinco primeras variables.
- Solo miraba los **dos extremos**. En este modelo, **10 de los 15** controles numericos
  tienen su optimo en el INTERIOR del rango para alguna bolsa (`DDT` para todas): la
  funcion no es monotona y los extremos son los dos puntos equivocados.

Los controles **categoricos y constantes se omiten**: sin limites numericos no hay rejilla,
e inventarles un rango seria puntuar un escenario que nadie pidio. Los que el panel no
ofrece -- refutados y de lectura unica -- tampoco entran: no se puede rankear por
relevancia lo que no se deja mover.

### 4. El grafo inferido

Las compuertas de la parte (b) son lo unico del grafo que depende de la seleccion. Se
juntan en una matriz $G\in\mathbb{R}^{|\mathcal{B}|\times E}$, con $G_{b,e}=g_{b,e}$.

Antes de reconstruir nada se mide si esas compuertas **varian** entre vanos. Con
$\tilde G$ la matriz centrada por columnas y $\sigma_1,\dots$ sus valores singulares:

$$\mathrm{var}=\frac{1}{E}\sum_e \mathrm{Var}_b(G_{b,e}),\qquad
\mathrm{rank}_{\text{ef}}=\frac{\bigl(\sum_r\sigma_r^2\bigr)^2}{\sum_r\sigma_r^4},\qquad
\text{colapso}\iff \max_e \mathrm{std}_b(G_{b,e})<10^{-6}\;\lor\;\mathrm{rank}_{\text{ef}}\le 1$$

El rango efectivo es el cociente de participacion: vale $\approx 1$ cuando toda la
variacion vive en una sola direccion, es decir, cuando todos los vanos estan compuertados
igual. Un colapso **anula** el grafo y el panel lo dice, en vez de dibujar el grafo
experto fijo como si lo hubiera estimado esta seleccion. De ahi sale un limite duro:
con $|\mathcal{B}|<3$ la matriz centrada tiene rango 1 por construccion, asi que **menos
de 3 vanos nunca producen grafo**.

Si no hay colapso, el peso reconstruido de cada arista es el peso experto fijo tal como lo
usa esta familia de vanos:

$$\bar g_e=\frac{1}{|\mathcal{B}|}\sum_{b}G_{b,e},\qquad
A_{\mathrm{src}(e),\,\mathrm{dst}(e)}=\bar g_e\cdot w_e,\qquad A_{ij}=0 \text{ fuera del soporte}$$

El panel lo dibuja en disposicion circular sobre las variables que participan de al menos
una arista, con $A_{ij}$ en un marcador sobre el punto medio de cada arista.

### Presupuesto de una pulsacion

| Paso | Pasadas de bolsas |
|---|---|
| Mapa simulado (base + simulado) | 2 |
| Compuertas para el grafo | 1 |
| Relevancia (base compartida + rejilla de $G$ por control) | $1+GK$ |

$K$ son los controles **numericos que el panel ofrece** y $G=9$ los puntos de la rejilla.
De ahi sale la propiedad que sostiene el panel: subir el top de cinco a diez variables **no
cuesta una sola pasada mas**. El barrido ya calculo $\hat u^{\star}_{b,\kappa}$ para todos
los controles y todas las bolsas; el top solo decide cuantos de esos numeros se dibujan.

Medido sobre el modelo real: **0,20 s** con la rejilla de nueve, contra 0,05 s del barrido
min-max de dos puntos. Cuatro veces mas pasadas y sigue siendo instantaneo, que es lo que
permite pagar la rejilla en vez de conformarse con los extremos.

## De donde sale el UITI de los violines

Los dos violines de la fila 4 miden **la misma cantidad dos veces sobre los mismos vanos**.
Vale la pena decir con precision cual, porque el nombre "UITI acumulado" tambien es el de
una columna del historico y **no es esa** la que se dibuja aqui.

### La unidad es la bolsa, no el evento

Cada punto de un violin es **una bolsa**: la celda $(\text{vano}, \text{ventana})$ de la
seleccion activa. Con la ventana fija y hasta cinco vanos marcados, un violin tiene como
maximo cinco puntos -- por eso van dibujados uno por uno (`points='all'`) y no solo como
densidad: con tres o cuatro datos, lo honesto es mostrarlos.

### Los dos numeros

Las bolsas de la seleccion aportan su matriz de instancias $X$ -- una fila por evento --
y su mapa instancia $\to$ bolsa. De ahi salen los dos vectores, con **una pasada del
modelo cada uno**:

$$\hat u^{\text{base}}_b=f_\theta\bigl(X,\;\text{bolsa}\bigr)_b,
\qquad
\hat u^{\text{sim}}_b=f_\theta\bigl(X',\;\text{bolsa}\bigr)_b$$

donde $f_\theta$ es el modelo MIL del cuaderno 05 -- el mismo que pinta los dos mapas -- y
$X'$ es **exactamente $X$** con las columnas de los controles fijados sobreescritas, cada
vano en sus propias filas. Todo lo que no se toco en el panel entra a $X'$ con su valor
**observado**: un control que no se fija no escribe nada.

### Lo que hay que tener claro al leerlos

- **El violin "Base" tambien es una prediccion**, no el historico. Es el modelo puntuando
  los mismos vanos con sus valores observados. Se compara prediccion contra prediccion a
  proposito: asi lo unico que separa a los dos violines es lo que se movio en el panel.
  Contra el UITI medido se estaria midiendo otra cosa -- el efecto de la simulacion mas el
  error del modelo, mezclados y sin forma de separarlos.
- **La escala es $\hat u$, el UITI acumulado predicho de la bolsa**, la misma cantidad que
  entra al eje vertical de la geometria KMeans -- por eso mover un violin y ver cambiar de
  grupo un vano en el mapa son la misma cosa vista dos veces.
- **$n_b$, los eventos observados, jamas se simula**. Es el otro eje del espacio que define
  la clase. De ahi que toda la diferencia entre los dos violines venga de $\hat u$ y de
  nada mas.
- **Si los dos violines se superponen, la simulacion no movio nada.** Es la lectura que
  cierra el tablero y se hace de un vistazo, sin comparar dos mapas tramo a tramo.
- Los violines describen a los **vanos marcados**. Sin ninguno marcado el grano es el
  circuito completo, y entonces cada punto es un vano del circuito con eventos en la
  ventana: mas puntos y otra pregunta.

### Lo que cuesta

**Dos pasadas de bolsas en total**, nunca una por vano: los valores de cada vano se
escriben en la MISMA matriz y se puntuan juntos. Es la misma corrida que produce el mapa
simulado -- los violines no vuelven a llamar al modelo, leen la tabla que ya devolvio.
